Continuation of Chapters 1–2. **Cell 1** holds all configuration and builders; each following cell exports **one** MP4 as **`ch3_00_…` through `ch3_58_…`** (script; see `CH3_EXPORT_FILES`).


In [108]:
import gc
import io
import os
import shutil
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import ticker as mticker
from matplotlib.gridspec import GridSpecFromSubplotSpec
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image, ImageDraw
np.random.seed(7)
# =============================================================================
# CHAPTER 3 — CONFIG (single cell; edit here only)
# =============================================================================
OUTPUT_DIR = Path("renders")
OUTPUT_DIR.mkdir(exist_ok=True)
EXPORT_FIGSIZE = (15.0, 9.5)
EXPORT_DPI = 200
SAVE_PAD_INCHES = 0.12
FONT_SIZE = 11 * 1.25
AXIS_LABEL_SIZE = 12 * 1.25
LEGEND_SIZE = 20
TITLE_SIZE = 12 * 1.25
NOTE_SIZE = 10 * 1.25
ANNOT_SIZE = 9 * 1.25
# Fast preview: ``export CH3_DRAFT_EXPORT=1`` (fewer frames, lower DPI, lighter σ contours).
_CH3_DRAFT = os.environ.get("CH3_DRAFT_EXPORT", "").strip().lower() in {"1", "true", "yes", "y"}
CH3_ANIM_DPI = min(int(EXPORT_DPI), 110)
if _CH3_DRAFT:
    CH3_ANIM_DPI = min(CH3_ANIM_DPI, 78)
CH3_MP4_MS_PER_FRAME = 35
CH3_W_ST_REF, CH3_W_EL_REF, CH3_B_REF = 1.5, -0.5, 0.0
# All loss MP4s: each active knob sweeps **reference ± CH3_LOSS_SYM_DELTA** (one number; same for w₁, w₂, b).
CH3_LOSS_SYM_DELTA = 1.0
GIF_SMOOTH_FACTOR = 1 if _CH3_DRAFT else 4
def _smooth_n(n):
    return max(2, int(round(float(n) * GIF_SMOOTH_FACTOR)))
CH3_SWEEP_NSEG = max(28, _smooth_n(22))
CH3_N_GROW = max(6, _smooth_n(4))
CH3_N_SHRINK = max(6, _smooth_n(4))
CH3_DUO_WIDTH_RATIOS = (1.22, 1.0)
CH3_DUO_WSPACE = 0.16
CH3_LEFT_HEIGHT_RATIOS = (1.0, 0.34)
CH3_LEFT_HSPACE = 0.12
CH3_KNOB_WSPACE = 0.08
# After layout, shrink the knob row horizontally to the 2D data panel (``_ch3_align_knob_axes_under_data``).
CH3_KNOB_DATA_PAD_FRAC = 0.055
# Shift the whole knob strip left (fraction of 2D data axis width).
CH3_KNOB_STRIP_HSHIFT_FRAC = 0.075
CH3_KNOB_AXES_ZORDER = 10000.0
CH3_KNOB_MAX_DEG = 52.0
CH3_KNOB_DEG_PER_UNIT = 360.0
CH3_KNOB_PIL_MAX_DEG = 20.0 * 360.0
CH3_KNOB_ACTIVE_SCALE = 1.25
def _ch3_align_knob_axes_under_data(fig, ax_data, axes_k):
    """Match the three knob axes to the 2D data panel width (nested gridspec has no left/right here)."""
    axes_k = tuple(axes_k)
    if len(axes_k) != 3:
        return
    fig.canvas.draw()
    d = ax_data.get_position()
    pad = float(CH3_KNOB_DATA_PAD_FRAC) * float(d.width)
    hshift = float(CH3_KNOB_STRIP_HSHIFT_FRAC) * float(d.width)
    x0 = float(d.x0) + pad - hshift
    x1 = float(d.x0) + float(d.width) - pad - hshift
    span = max(x1 - x0, 1e-6)
    y0 = min(float(ax.get_position().y0) for ax in axes_k)
    y1 = max(float(ax.get_position().y1) for ax in axes_k)
    h = max(y1 - y0, 1e-6)
    n = 3
    ws = float(CH3_KNOB_WSPACE)
    w_each = span / (n + (n - 1) * ws)
    gap = ws * w_each
    for i, ax in enumerate(axes_k):
        xi = x0 + i * (w_each + gap)
        ax.set_position([xi, y0, w_each, h])
def ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k):
    """Map knob x/width to ch3_00 end, scaled to the current data panel (post-align y-band)."""
    axes_k = tuple(axes_k)
    if len(axes_k) != 3:
        return
    data_canon, knob_canon = ch3_duo_left_layout_rects()
    dx0, _, dw, _ = data_canon
    dw = max(float(dw), 1e-9)
    fig.canvas.draw()
    pr = ax_data.get_position()
    x0d, _, wd, _ = float(pr.x0), float(pr.y0), float(pr.width), float(pr.height)
    wd = max(wd, 1e-9)
    y0 = min(float(ax.get_position().y0) for ax in axes_k)
    y1 = max(float(ax.get_position().y1) for ax in axes_k)
    h = max(y1 - y0, 1e-6)
    for i, ax in enumerate(axes_k):
        x0k, _, wk, _ = knob_canon[i]
        rel_x0 = (float(x0k) - float(dx0)) / dw
        rel_w = float(wk) / dw
        ax.set_position([x0d + rel_x0 * wd, float(y0), rel_w * wd, float(h)])
def _ch3_raise_knob_axes(fig, axes_k):
    """Composite the three knob axes above sibling 2D / 3D / loss panels when bounding boxes overlap."""
    zo = float(CH3_KNOB_AXES_ZORDER)
    axes_k = tuple(axes_k)
    if len(axes_k) != 3:
        return
    for ax in axes_k:
        ax.set_zorder(zo)
        ax.patch.set_facecolor("none")
        ax.patch.set_edgecolor("none")
        ax.patch.set_linewidth(0.0)
        for im in ax.images:
            im.set_zorder(zo + 1.0)
_CH3_SIGMA_CONTOUR_LEVELS = 26 if _CH3_DRAFT else 45
_CH3_SIGMA_CONTOUR_ALPHA = 0.32
plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelpad": 8.0 * 1.25,
        "axes.titlepad": 10.0 * 1.25,
        "legend.fontsize": LEGEND_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.96,
        "legend.borderaxespad": 0.55,
        "legend.labelspacing": 0.35,
        "legend.handlelength": 1.35,
        "legend.handletextpad": 0.65,
        "savefig.pad_inches": SAVE_PAD_INCHES,
    }
)
PASS_COLOR = "#2ca02c"
FAIL_COLOR = "#d62728"
NEUTRAL_COLOR = "#4f4f4f"
CHECK_ICON_PATH = OUTPUT_DIR / "check.png"
CROSS_ICON_PATH = OUTPUT_DIR / "cross.png"
def _ensure_outcome_icons(size=120, line_width=15):
    _sc = size / 96.0
    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line(
        [(int(round(20 * _sc)), int(round(55 * _sc))), (int(round(42 * _sc)), int(round(76 * _sc))), (int(round(78 * _sc)), int(round(24 * _sc)))],
        fill=(44, 160, 44, 255),
        width=line_width,
    )
    img.save(CHECK_ICON_PATH)
    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line([(int(round(24 * _sc)), int(round(24 * _sc))), (int(round(72 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    draw.line([(int(round(72 * _sc)), int(round(24 * _sc))), (int(round(24 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    img.save(CROSS_ICON_PATH)
_ensure_outcome_icons()
CHECK_ICON = np.asarray(Image.open(CHECK_ICON_PATH).convert("RGBA"))
CROSS_ICON = np.asarray(Image.open(CROSS_ICON_PATH).convert("RGBA"))
def add_outcome_icon(ax, x, y_value, passed, zoom=0.2, alpha=1.0, rotation_deg=0):
    image_array = CHECK_ICON if passed else CROSS_ICON
    if rotation_deg:
        arr = np.asarray(image_array, dtype=np.uint8)
        if int(round(float(rotation_deg) / 180.0)) % 2:
            arr = np.flip(arr, axis=1)
            arr = np.rot90(arr, k=2, axes=(0, 1))
        image_array = arr
    icon = OffsetImage(image_array, zoom=zoom)
    icon.set_alpha(alpha)
    ab = AnnotationBbox(icon, (x, y_value), frameon=False)
    ab.set_clip_on(False)
    ax.add_artist(ab)
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))
def fig_to_image(fig, dpi=None, tight_layout=False, transparent=False):
    if dpi is None:
        dpi = EXPORT_DPI
    buf = io.BytesIO()
    kw = {"format": "png", "dpi": dpi, "pad_inches": SAVE_PAD_INCHES}
    kw["bbox_inches"] = "tight" if tight_layout else None
    kw["transparent"] = bool(transparent)
    fig.savefig(buf, **kw)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGBA" if transparent else "RGB")
def save_mp4(images, filename, duration=None):
    """Write an MP4 from PIL frames without materializing all ``ndarray`` frames at once.
    Accepts a ``list``/``tuple`` of images or a **generator** of images (streaming). The old
    implementation built a full list of RGB arrays plus ``mimsave``, which could OOM on long
    pair-3D ribbon clips.
    """
    import imageio.v2 as imageio
    if duration is None:
        duration = CH3_MP4_MS_PER_FRAME
    fps = 1000.0 / max(float(duration), 1e-3)
    out_path = OUTPUT_DIR / filename
    def _rgb(im):
        return im.convert("RGB")
    def _norm_arr(im_rgb, w, h, bg=(255, 255, 255)):
        if im_rgb.size == (w, h):
            return np.asarray(im_rgb)
        canvas = Image.new("RGB", (w, h), bg)
        canvas.paste(im_rgb, ((w - im_rgb.width) // 2, (h - im_rgb.height) // 2))
        return np.asarray(canvas)
    if isinstance(images, (list, tuple)):
        if not images:
            raise ValueError("images list is empty")
        w = 0
        h = 0
        for im in images:
            r = _rgb(im)
            w = max(w, int(r.width))
            h = max(h, int(r.height))
        with imageio.get_writer(
            str(out_path),
            fps=fps,
            codec="libx264",
            ffmpeg_params=["-crf", "20"],
        ) as writer:
            for im in images:
                arr = _norm_arr(_rgb(im), w, h)
                writer.append_data(arr)
                del arr
        return
    it = iter(images)
    try:
        first = _rgb(next(it))
    except StopIteration:
        raise ValueError("images iterator is empty")
    w, h = int(first.width), int(first.height)
    with imageio.get_writer(
        str(out_path),
        fps=fps,
        codec="libx264",
        ffmpeg_params=["-crf", "20"],
    ) as writer:
        writer.append_data(_norm_arr(first, w, h))
        del first
        for im in it:
            arr = _norm_arr(_rgb(im), w, h)
            writer.append_data(arr)
            del arr
separable_points = [
    (2, 3, 0),
    (4, 5, 0), (5, 6, 0),
    (1, 3, 0), (2, 4, 0), (4, 6, 0),
    (1, 4, 0), (3, 6, 0), (1, 6, 0),
    (3, 2, 1),
    (5, 4, 1), (6, 5, 1),
    (4, 2, 1), (6, 4, 1), (3, 1, 1),
    (4, 1, 1), (5, 2, 1), (6, 3, 1),
    (6, 2, 1),
    (6, 1, 1),
]
noisy_symmetric_points = [
    (2, 1, 0),
    (1, 2, 1),
    (3, 4, 1),
    (4, 3, 0),
    (3, 5, 1),
    (5, 3, 0),
]
realistic_points = separable_points + noisy_symmetric_points
def unpack_points(point_list):
    arr = np.array(point_list, dtype=float)
    s = arr[:, 0]
    e = arr[:, 1]
    labels = arr[:, 2].astype(int)
    diff = s - e
    return s, e, labels, diff
study_sep, exam_sep, y_sep, z_sep = unpack_points(separable_points)
study_real, exam_real, y_real, z_real = unpack_points(realistic_points)
xlim = (0, 7)
ylim = (0, 7)
midpoint_shift = (z_sep[y_sep == 0].max() + z_sep[y_sep == 1].min()) / 2.0
_h_anim = 0.07
_ks_st = np.arange(float(xlim[0]), float(xlim[1]) + 1e-9, float(_h_anim) / 2.0)
_ks_el = np.arange(float(ylim[0]), float(ylim[1]) + 1e-9, float(_h_anim) / 2.0)
ST_KNOB, EL_KNOB = np.meshgrid(_ks_st, _ks_el)
cvals = [0, 0.5, 1]
colors = ["red", "white", "green"]
_norm = plt.Normalize(min(cvals), max(cvals))
_tuples = list(zip(map(_norm, cvals), colors))
CMAP_GD = mpl.colors.LinearSegmentedColormap.from_list("gd_sigmoid_plane", _tuples, 256)
CH3_LEGEND_TEX = shutil.which("latex") is not None and os.environ.get("CH3_DISABLE_TEX", "") == ""
def finalize_style_legend_tex(ax):
    if not CH3_LEGEND_TEX:
        return
    leg = ax.get_legend()
    if leg is None:
        return
    for t in leg.get_texts():
        t.set_usetex(True)
def logits_plane(w_st, w_el, b, st, el):
    return b + w_st * st + w_el * el
def boundary_line_xy(w_st, w_el, b, xa, xb, ya, yb):
    if abs(w_el) < 1e-9:
        if abs(w_st) < 1e-9:
            return None
        x0 = -b / w_st
        return np.array([x0, x0]), np.array([ya, yb])
    xs = np.linspace(xa, xb, 400)
    ys = -(b + w_st * xs) / w_el
    m = (ys >= ya) & (ys <= yb)
    if not np.any(m):
        return xs, ys
    return xs[m], ys[m]
def draw_dataset(ax, study_vals, exam_vals, labels, mask=None, alpha=0.95):
    if mask is None:
        mask = np.ones(len(study_vals), dtype=bool)
    for i in np.where(mask)[0]:
        add_outcome_icon(ax, study_vals[i], exam_vals[i], passed=bool(labels[i]), alpha=alpha)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
def mistake_mask(study, exam, y, w_st, w_el, b):
    logit = logits_plane(w_st, w_el, b, study, exam)
    pred = (logit >= 0.0).astype(int)
    return pred != y
def highlight_mistakes(ax, study, exam, y, w_st, w_el, b, radius=0.32, lw=2.8):
    wrong = mistake_mask(study, exam, y, w_st, w_el, b)
    for i in np.where(wrong)[0]:
        circ = plt.Circle(
            (study[i], exam[i]),
            radius,
            facecolor="none",
            edgecolor="red",
            linewidth=lw,
            zorder=12,
        )
        ax.add_patch(circ)

def highlight_mistakes_at(ax, points, *, radius=0.32, lw=2.8, edgecolor="red"):
    """Red rings at explicit (study, exam) locations."""
    for sx, sy in points:
        ax.add_patch(
            plt.Circle(
                (float(sx), float(sy)),
                radius,
                facecolor="none",
                edgecolor=edgecolor,
                linewidth=lw,
                zorder=14,
            )
        )

def _trim_num_coeff(x):
    x = float(x)
    xr = round(x, 2)
    if abs(xr - round(xr)) < 1e-12:
        return str(int(round(xr)))
    return f"{xr:.2f}".rstrip("0").rstrip(".")
def legend_linear_equation_values_bold_param(ws, we, b, emphasize=None):
    ws, we, b = float(ws), float(we), float(b)
    def piece_st(wv, bold):
        axv = abs(float(wv))
        t = _trim_num_coeff(axv)
        core = rf"\mathbf{{{t}}}" if bold else t
        if wv < 0:
            return rf"-{core}\,\mathrm{{ST}}"
        return rf"{core}\,\mathrm{{ST}}"
    def piece_el(wv, bold):
        axv = abs(float(wv))
        t = _trim_num_coeff(axv)
        core = rf"\mathbf{{{t}}}" if bold else t
        sign = "+" if wv >= 0 else "-"
        return rf"{sign}{core}\,\mathrm{{EL}}"
    def piece_bias(wv, bold):
        axv = abs(float(wv))
        t = _trim_num_coeff(axv)
        core = rf"\mathbf{{{t}}}" if bold else t
        sign = "+" if wv >= 0 else "-"
        return rf"{sign}{core}"
    if emphasize is None:
        em_st = em_el = em_b = False
    else:
        em_st = emphasize == "st"
        em_el = emphasize == "el"
        em_b = emphasize == "b"
    return "$" + piece_st(ws, em_st) + piece_el(we, em_el) + piece_bias(b, em_b) + r" = 0$"
def ch3_sigma_contourf(ax, stg, elg, Z, *, alpha=None, zorder=1):
    a = float(_CH3_SIGMA_CONTOUR_ALPHA if alpha is None else alpha)
    levs = np.linspace(0.0, 1.0, int(_CH3_SIGMA_CONTOUR_LEVELS))
    ax.contourf(
        stg,
        elg,
        Z,
        levels=levs,
        cmap=CMAP_GD,
        vmin=0,
        vmax=1,
        antialiased=not _CH3_DRAFT,
        alpha=a,
        zorder=zorder,
    )
def _ch3_sigma_z_cache_key(ws, we, bb):
    return (round(float(ws), 10), round(float(we), 10), round(float(bb), 10))
def ch3_draw_left_panel(
    ax,
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    legend_label,
    *,
    show_colormap,
    highlight_mistakes_flag=True,
    sigma_Z=None,
    colormap_alpha=None,
    mistake_points=None,
):
    stg, elg = ST_KNOB, EL_KNOB
    if show_colormap:
        if sigma_Z is None:
            Z = sigmoid(logits_plane(w_st, w_el, b, stg, elg))
        else:
            Z = sigma_Z
        ca = _CH3_SIGMA_CONTOUR_ALPHA if colormap_alpha is None else float(colormap_alpha)
        ch3_sigma_contourf(ax, stg, elg, Z, alpha=ca, zorder=1)
    bxy = boundary_line_xy(w_st, w_el, b, float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    if bxy is not None:
        bx, by = bxy
        ax.plot(
            bx,
            by,
            c="grey",
            linestyle="--",
            linewidth=1.8,
            label=legend_label,
            zorder=3,
        )
    draw_dataset(ax, study, exam, y)
    if highlight_mistakes_flag:
        highlight_mistakes(ax, study, exam, y, w_st, w_el, b)
    if mistake_points:
        highlight_mistakes_at(ax, mistake_points)
    ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
    finalize_style_legend_tex(ax)
def ch3_draw_dataset_panel(
    ax,
    study,
    exam,
    y,
    *,
    w_st=None,
    w_el=None,
    b=None,
    show_colormap=False,
    highlight_mistakes_flag=False,
):
    """2D roster only (no knobs): optional σ fill, optional decision line."""
    stg, elg = ST_KNOB, EL_KNOB
    if show_colormap and w_st is not None:
        Z = sigmoid(logits_plane(float(w_st), float(w_el), float(b), stg, elg))
        ch3_sigma_contourf(ax, stg, elg, Z, zorder=1)
    leg = None
    if w_st is not None:
        leg = legend_linear_equation_values_bold_param(float(w_st), float(w_el), float(b), "all")
        bxy = boundary_line_xy(
            float(w_st), float(w_el), float(b), float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1])
        )
        if bxy is not None:
            bx, by = bxy
            ax.plot(bx, by, c="grey", linestyle="--", linewidth=1.8, label=leg, zorder=3)
    draw_dataset(ax, study, exam, y)
    if highlight_mistakes_flag and w_st is not None:
        highlight_mistakes(ax, study, exam, y, float(w_st), float(w_el), float(b))
    if leg is not None:
        ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
    finalize_style_legend_tex(ax)
def ch3_frame_dataset_still(study, exam, y, *, w_st=None, w_el=None, b=None, show_colormap=False):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    ch3_draw_dataset_panel(
        ax, study, exam, y, w_st=w_st, w_el=w_el, b=b, show_colormap=show_colormap, highlight_mistakes_flag=False
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 110))
def ch3_build_frames_dataset_still(study, exam, y, *, w_st=None, w_el=None, b=None, show_colormap=False, n_hold=None):
    n_hold = int(CH3_SCRIPT_N_STILL if n_hold is None else n_hold)
    im = ch3_frame_dataset_still(study, exam, y, w_st=w_st, w_el=w_el, b=b, show_colormap=show_colormap)
    return [im.copy() for _ in range(max(1, n_hold))]
def loss_mistake_count(w_st, w_el, b, study, exam, y):
    z = logits_plane(w_st, w_el, b, study, exam)
    pred = (z >= 0.0).astype(int)
    return float(np.sum(pred != y))
def loss_sum_sigma(w_st, w_el, b, study, exam, y):
    z = logits_plane(w_st, w_el, b, study, exam)
    return float(np.sum(sigmoid(z)))
def loss_sum_p_y_given_x(w_st, w_el, b, study, exam, y):
    """Sum of p(y_i | x_i) for the Bernoulli logistic model (true-class probability per point)."""
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    yy = y.astype(float)
    return float(np.sum(yy * p + (1.0 - yy) * (1.0 - p)))
def loss_neg_log_likelihood(w_st, w_el, b, study, exam, y):
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    eps = 1e-12
    yy = y.astype(float)
    nll = -(yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps))
    return float(np.sum(nll))
def loss_likelihood(w_st, w_el, b, study, exam, y):
    """Bernoulli likelihood (product of p(y_i|x_i)); computed as exp(sum log p) for numerical stability."""
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    eps = 1e-12
    yy = y.astype(float)
    logp = yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps)
    logL = float(np.sum(logp))
    return float(np.exp(np.clip(logL, -700.0, 700.0)))
CH3_LOSS_SPECS = {
    "mistakes": {
        "fn": loss_mistake_count,
        "ylabel": "Mistake count",
        "title": "Mistake count",
        "colormap": False,
    },
    "sumprob": {
        "fn": loss_sum_p_y_given_x,
        "ylabel": r"$\sum_i p(y_i \mid x_i)$",
        "title": r"$\sum_i p(y_i \mid x_i)$",
        "colormap": True,
    },
    "negloglik": {
        "fn": loss_neg_log_likelihood,
        "ylabel": r"$-\sum_i \log p(y_i \mid x_i)$",
        "title": "Negative log-likelihood",
        "colormap": True,
    },
    "likelihood": {
        "fn": loss_likelihood,
        "ylabel": r"Likelihood $\mathcal{L}$",
        "title": "Likelihood",
        "colormap": True,
    },
}
def ch3_triplet(which, val, base=None):
    if base is None:
        base = (CH3_W_ST_REF, CH3_W_EL_REF, CH3_B_REF)
    w1, w2, bb = base
    if which == "st":
        return float(val), float(w2), float(bb)
    if which == "el":
        return float(w1), float(val), float(bb)
    if which == "b":
        return float(w1), float(w2), float(val)
    raise ValueError(which)
def ch3_active_value(which, w_st, w_el, b):
    if which == "st":
        return float(w_st)
    if which == "el":
        return float(w_el)
    if which == "b":
        return float(b)
    raise ValueError(which)
def ch3_quad_sweep(center, delta, nseg):
    nseg = int(max(2, nseg))
    up = np.linspace(center, center + delta, nseg, endpoint=True)
    dn = np.linspace(center + delta, center, nseg, endpoint=True)
    up2 = np.linspace(center, center - delta, nseg, endpoint=True)
    dn2 = np.linspace(center - delta, center, nseg, endpoint=True)
    return np.concatenate([up, dn[1:], up2[1:], dn2[1:]])
def ch3_knob_rot_deg_absolute(param):
    """Integer coefficient → upright (0°); half-integers → 180°; else −frac·360° (PIL CW)."""
    v = float(param)
    if abs(v - round(v)) < 1e-5:
        return 0.0
    if abs(2.0 * v - round(2.0 * v)) < 1e-5:
        return 180.0
    frac = v - round(v)
    return float(-frac * CH3_KNOB_DEG_PER_UNIT)
def ch3_knob_rots_for_weights(ws, we, bb):
    """PIL rotations: integers upright; 0.5 and 1.5 (half-integers) at ±180°."""
    md = float(CH3_KNOB_PIL_MAX_DEG)
    def one(v):
        return float(np.clip(ch3_knob_rot_deg_absolute(v), -md, md))
    return (one(ws), one(we), one(bb))
def ch3_knob_turn_deg_relative(val, ref, *, int_eps=1e-5):
    """PIL degrees: 360° per unit vs ``ref``; integers snap upright (CW when value increases)."""
    v = float(val)
    r = float(ref)
    if abs(v - round(v)) < int_eps or abs(v - r) < int_eps:
        return 0.0
    md = float(CH3_KNOB_PIL_MAX_DEG)
    return float(np.clip(-(v - r) * float(CH3_KNOB_DEG_PER_UNIT), -md, md))
def ch3_knob_strip_rotation_deg(val, lo, hi, *, ref=None):
    """Strip sweep: 0° at ``ref`` (default ``lo``); limited to ±CH3_KNOB_MAX_DEG over [lo, hi]."""
    lo, hi = float(lo), float(hi)
    if ref is None:
        ref = lo
    span = hi - lo
    amp = max(abs(span), 1e-6)
    delta = float(val) - float(ref)
    deg = -float(CH3_KNOB_MAX_DEG) * (delta / amp)
    lim = float(CH3_KNOB_MAX_DEG) * 1.05
    return float(np.clip(deg, -lim, lim))
def ch3_knob_rotation_from_center(val, center=None, half_span=None):
    """K1 w₁ dial rotation from absolute value (``center`` unused)."""
    _ = center, half_span
    return ch3_knob_rots_for_weights(float(val), 0.0, 0.0)[0]
def ch3_knob_smoothstep(u):
    u = float(np.clip(u, 0.0, 1.0))
    return u * u * (3.0 - 2.0 * u)
CH3_K1_ACTIVE_SCALE_FULL = float(CH3_KNOB_ACTIVE_SCALE)
def ch3_k1_knob_rots_at(w1, w2, b):
    return list(ch3_knob_rots_for_weights(float(w1), float(w2), float(b)))
def ch3_k1_active_scale_grow_sequence():
    """Active knob scales 1.0 → CH3_KNOB_ACTIVE_SCALE at clip start (rotation pinned separately)."""
    if CH3_N_GROW <= 1:
        return [float(CH3_KNOB_ACTIVE_SCALE)]
    return [
        1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(float(j) / float(CH3_N_GROW - 1))
        for j in range(CH3_N_GROW)
    ]
def ch3_k1_active_scale_shrink_sequence():
    """Active knob scales CH3_KNOB_ACTIVE_SCALE → 1.0 at clip end."""
    if CH3_N_SHRINK <= 1:
        return [1.0]
    return [
        CH3_KNOB_ACTIVE_SCALE - (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(float(j) / float(CH3_N_SHRINK - 1))
        for j in range(CH3_N_SHRINK)
    ]
_CH3_KNOB_NORM_R = 448
_CH3_KNOB_NORM_CACHE = {}
def _ch3_knob_normalized_square(knob_rgba):
    kid = id(knob_rgba)
    hit = _CH3_KNOB_NORM_CACHE.get(kid)
    if hit is not None:
        return hit
    R = int(_CH3_KNOB_NORM_R)
    w0, h0 = knob_rgba.size
    scale = float(R) / float(max(int(w0), int(h0), 1))
    nw = max(1, int(round(float(w0) * scale)))
    nh = max(1, int(round(float(h0) * scale)))
    im1 = knob_rgba.resize((nw, nh), resample=Image.Resampling.LANCZOS)
    out = Image.new("RGBA", (R, R), (0, 0, 0, 0))
    out.paste(im1, ((R - nw) // 2, (R - nh) // 2), im1)
    _CH3_KNOB_NORM_CACHE[kid] = out
    return out
def _ch3_probe_knob_canvas_side(knob_rgba, *, deg_lo=-180.0, deg_hi=180.0, n=37, pad=8):
    """Max rotated bbox on the normalized square (must cover ±180° for half-integer coeffs)."""
    base = _ch3_knob_normalized_square(knob_rgba)
    s = 0
    for deg in np.linspace(float(deg_lo), float(deg_hi), int(n)):
        im = base.rotate(float(deg), resample=Image.Resampling.BICUBIC, expand=True)
        s = max(s, im.width, im.height)
    return int(s + pad)
def _ch3_knob_pil_rotated_square(knob_rgba, deg, canvas_side):
    """Rotate on a fixed square raster, then resize to ``canvas_side`` — no per-angle fit zoom (ch2)."""
    side = int(canvas_side)
    if abs(float(deg)) <= 1e-9:
        base = _ch3_knob_normalized_square(knob_rgba)
        return base.resize((side, side), resample=Image.Resampling.LANCZOS)
    base = _ch3_knob_normalized_square(knob_rgba)
    im = base.rotate(float(deg), resample=Image.Resampling.BICUBIC, expand=False)
    return im.resize((side, side), resample=Image.Resampling.LANCZOS)
_CH3_KNOB_SLOT_RECTS = None
def ch3_duo_canonical_knob_rects():
    """Knob slot rects at ch3_00 bridge end (duo left column, after align)."""
    _, knobs = ch3_duo_left_layout_rects()
    return knobs
def ch3_knob_slot_rects():
    """Alias: canonical duo knob slots (matches ``ch3_00`` end)."""
    return ch3_duo_canonical_knob_rects()
def ch3_ensure_knob_pngs():
    """Prefer ``renders/knob_{1,2,3}_cropped.png`` from Chapter 2; else write minimal dial placeholders."""
    for k in (1, 2, 3):
        p = OUTPUT_DIR / f"knob_{k}_cropped.png"
        if p.is_file():
            continue
        im = Image.new("RGBA", (180, 180), (248, 248, 248, 0))
        dr = ImageDraw.Draw(im)
        dr.ellipse((10, 10, 170, 170), outline=(100, 100, 100, 255), width=5)
        dr.text((74, 68), str(k), fill=(50, 50, 50, 255))
        im.save(p)
_CH3_KNOB_ASSET_PACK = None
def ch3_knob_asset_pack():
    global _CH3_KNOB_ASSET_PACK
    if _CH3_KNOB_ASSET_PACK is not None:
        return _CH3_KNOB_ASSET_PACK
    ch3_ensure_knob_pngs()
    rgbs = tuple(Image.open(OUTPUT_DIR / f"knob_{k}_cropped.png").convert("RGBA") for k in (1, 2, 3))
    sides = tuple(_ch3_probe_knob_canvas_side(im) for im in rgbs)
    side_uni = int(max(sides))
    sides = (side_uni, side_uni, side_uni)
    _CH3_KNOB_ASSET_PACK = (rgbs, sides)
    return _CH3_KNOB_ASSET_PACK
def ch3_knob_scales_emphasize(emphasize, active_scale=1.0):
    """Per-slot scale: active knob → ``active_scale`` (≤ CH3_KNOB_ACTIVE_SCALE); others 1.0."""
    slot = {"st": 0, "el": 1, "b": 2}[str(emphasize).lower()]
    sc = float(np.clip(float(active_scale), 1.0, float(CH3_KNOB_ACTIVE_SCALE)))
    scales = [1.0, 1.0, 1.0]
    scales[slot] = sc
    return scales
def _ch3_figfrac_xywh(pr):
    """Figure-fraction (x0, y0, width, height) from a Bbox or 4-tuple."""
    if hasattr(pr, "x0"):
        return (
            float(pr.x0),
            float(pr.y0),
            float(pr.width),
            float(pr.height),
        )
    x0, y0, w, h = pr
    return float(x0), float(y0), float(w), float(h)
def ch3_place_pil_knob_row(fig, axes_k, knob_rgbs, canvas_sides, rot_degs, slot_scales, *, ax_data=None):
    """Draw knobs; only ``slot_scales`` changes dial box size (ch2 grow/sweep/shrink)."""
    axes_k = tuple(axes_k)
    scales = [float(np.clip(float(slot_scales[i]), 1.0, float(CH3_KNOB_ACTIVE_SCALE))) for i in range(3)]
    fig.canvas.draw()
    if ax_data is not None:
        x0d, _, wd, _ = _ch3_figfrac_xywh(ax_data.get_position())
        data_canon, knob_canon = ch3_duo_left_layout_rects()
        dx0, _, dw, _ = data_canon
        dw = max(float(dw), 1e-9)
        wd = max(float(wd), 1e-9)
        y0 = min(float(ax.get_position().y0) for ax in axes_k)
        y1 = max(float(ax.get_position().y1) for ax in axes_k)
        h = max(y1 - y0, 1e-6)
        slot_rects = []
        for i, _ax in enumerate(axes_k):
            x0k, _, wk, _ = knob_canon[i]
            rel_x0 = (float(x0k) - float(dx0)) / dw
            rel_w = float(wk) / dw
            slot_rects.append((x0d + rel_x0 * wd, y0, rel_w * wd, h))
    else:
        slot_rects = [_ch3_figfrac_xywh(axk.get_position()) for axk in axes_k]
    active_i = int(np.argmax(scales))
    draw_order = [i for i in range(3) if i != active_i] + [active_i]
    for i in draw_order:
        axk = axes_k[i]
        x0, y0, w, h = slot_rects[i]
        cx = x0 + 0.5 * w
        cy = y0 + 0.5 * h
        sc = scales[i]
        sw = sc * w
        sh = sc * h
        axk.set_position((cx - 0.5 * sw, cy - 0.5 * sh, sw, sh))
        rd = float(rot_degs[i])
        arr = np.asarray(_ch3_knob_pil_rotated_square(knob_rgbs[i], rd, canvas_sides[i]))
        axk.clear()
        axk.imshow(arr, interpolation="nearest")
        axk.axis("off")
    fig.canvas.draw()
def ch3_figure_duo():
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    ax_data = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    ax_loss = fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, ax_loss, axes_k
def ch3_draw_knob_row(
    fig,
    axes_k,
    ws,
    we,
    bb,
    which,
    knob_rgbs,
    canvas_sides,
    *,
    rot_strip_deg=0.0,
    strip_scale=1.0,
    knob_rots=None,
    knob_scales=None,
    ax_data=None,
):
    if knob_rots is not None and knob_scales is not None:
        rots = [float(knob_rots[i]) for i in range(3)]
        scales = [float(np.clip(float(knob_scales[i]), 1.0, float(CH3_KNOB_ACTIVE_SCALE))) for i in range(3)]
    else:
        slot = {"st": 0, "el": 1, "b": 2}[which]
        rots = list(ch3_knob_rots_for_weights(ws, we, bb))
        if abs(float(rot_strip_deg)) > 1e-12:
            rots[slot] = float(rot_strip_deg)
        scales = [1.0, 1.0, 1.0]
        scales[slot] = float(np.clip(float(strip_scale), 1.0, float(CH3_KNOB_ACTIVE_SCALE)))
    ch3_place_pil_knob_row(
        fig, axes_k, knob_rgbs, canvas_sides, tuple(rots), tuple(scales), ax_data=ax_data,
    )
    _ch3_raise_knob_axes(fig, axes_k)
def _ch3_ytick_labels_integers_only(ax_loss):
    """Keep y tick marks; show numbers only at integer mistake counts."""
    ymin, ymax = ax_loss.get_ylim()
    if ymax <= ymin:
        return
    ticks = mticker.MaxNLocator(nbins=8).tick_values(ymin, ymax)
    ticks = ticks[(ticks >= ymin - 1e-9) & (ticks <= ymax + 1e-9)]
    if ticks.size == 0:
        return
    ax_loss.set_yticks(ticks)

    def _fmt(val, _pos):
        r = round(float(val))
        if abs(float(val) - r) < 1e-9:
            return str(int(r))
        return ""

    ax_loss.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt))
def _ch3_style_loss_axis(ax_loss, *, show_axis_labels=False, show_tick_labels=False):
    if show_axis_labels:
        ax_loss.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    elif show_tick_labels:
        ax_loss.set_xlabel("")
        ax_loss.set_ylabel("")
        ax_loss.tick_params(
            axis="both",
            which="major",
            labelsize=ANNOT_SIZE,
            labelbottom=True,
            labelleft=True,
            bottom=True,
            left=True,
        )
        _ch3_ytick_labels_integers_only(ax_loss)
    else:
        ax_loss.set_xlabel("")
        ax_loss.set_ylabel("")
        ax_loss.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)
def ch3_clear_loss_panel(ax_loss, x_lim, y_lim):
    """Reserve triptych slot with no curve (invisible auxiliary panels)."""
    ax_loss.clear()
    ax_loss.set_xlim(x_lim[0], x_lim[1])
    ax_loss.set_ylim(y_lim[0], y_lim[1])
    ax_loss.axis("off")
    for sp in ax_loss.spines.values():
        sp.set_visible(False)
def ch3_draw_loss_panel(
    ax_loss,
    xs_prefix,
    ys_prefix,
    spec,
    x_lim,
    y_lim,
    *,
    title_override=None,
    show_axis_labels=False,
    show_tick_labels=False,
    curve_alpha=1.0,
    marker_xy=None,
):
    ax_loss.clear()
    if title_override is None:
        ax_loss.set_title(spec["title"], fontsize=TITLE_SIZE, pad=8)
    elif title_override:
        ax_loss.set_title(title_override, fontsize=TITLE_SIZE, pad=8)
    if show_axis_labels:
        ax_loss.set_ylabel(spec["ylabel"], fontsize=AXIS_LABEL_SIZE)
    ca = float(np.clip(curve_alpha, 0.0, 1.0))
    if len(xs_prefix) >= 2:
        ax_loss.plot(
            xs_prefix,
            ys_prefix,
            color=FAIL_COLOR,
            linewidth=2.8,
            zorder=2,
            solid_capstyle="round",
            alpha=ca,
        )
    tip_x, tip_y = None, None
    if marker_xy is not None:
        tip_x, tip_y = float(marker_xy[0]), float(marker_xy[1])
    elif len(xs_prefix) >= 1:
        tip_x, tip_y = float(xs_prefix[-1]), float(ys_prefix[-1])
    if tip_x is not None:
        ax_loss.scatter(
            [tip_x],
            [tip_y],
            color=FAIL_COLOR,
            s=320,
            zorder=3,
            edgecolors="white",
            linewidths=1.2,
            alpha=ca,
        )
    if len(xs_prefix) >= 2:
        ax_loss.scatter(
            xs_prefix[:-1],
            ys_prefix[:-1],
            color=FAIL_COLOR,
            s=36,
            zorder=2,
            alpha=0.5 * ca,
            edgecolors="white",
            linewidths=0.45,
        )
    ax_loss.set_xlim(x_lim[0], x_lim[1])
    ax_loss.set_ylim(y_lim[0], y_lim[1])
    ax_loss.grid(alpha=0.25 * ca)
    _ch3_style_loss_axis(
        ax_loss, show_axis_labels=show_axis_labels, show_tick_labels=show_tick_labels,
    )
def ch3_frame(
    w_st,
    w_el,
    b,
    which,
    study,
    exam,
    y,
    spec,
    xs_hist,
    ys_hist,
    knob_rgbs,
    canvas_sides,
    *,
    show_colormap,
    rot_strip_deg,
    strip_scale,
    x_lim_loss,
    y_lim_loss,
):
    fig, ax_data, ax_loss, axes_k = ch3_figure_duo()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, which)
    ch3_draw_left_panel(
        ax_data,
        w_st,
        w_el,
        b,
        study,
        exam,
        y,
        leg,
        show_colormap=show_colormap,
        highlight_mistakes_flag=True,
    )
    ch3_draw_knob_row(
        fig, axes_k, w_st, w_el, b, which, knob_rgbs, canvas_sides,
        rot_strip_deg=rot_strip_deg, strip_scale=strip_scale, ax_data=ax_data,
    )
    ch3_draw_loss_panel(ax_loss, xs_hist, ys_hist, spec, x_lim_loss, y_lim_loss, show_axis_labels=False)
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_build_frames(loss_key, which, with_noise):
    spec = CH3_LOSS_SPECS[loss_key]
    loss_fn = spec["fn"]
    show_cmap = bool(spec["colormap"])
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    center = ch3_active_value(which, CH3_W_ST_REF, CH3_W_EL_REF, CH3_B_REF)
    seq = ch3_quad_sweep(center, CH3_LOSS_SYM_DELTA, CH3_SWEEP_NSEG)
    triples = [ch3_triplet(which, float(v)) for v in seq]
    losses = np.array([loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples], dtype=float)
    pad_y = 0.06 * max(1e-6, float(np.nanmax(losses) - np.nanmin(losses)))
    y_lo = float(np.nanmin(losses) - pad_y)
    y_hi = float(np.nanmax(losses) + pad_y)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or abs(y_hi - y_lo) < 1e-9:
        y_lo, y_hi = y_lo - 1.0, y_hi + 1.0
    span_x = float(np.max(seq) - np.min(seq))
    pad_x = max(0.06 * span_x, 0.08)
    x_lo = float(np.min(seq) - pad_x)
    x_hi = float(np.max(seq) + pad_x)
    lo_rot = float(np.min(seq))
    hi_rot = float(np.max(seq))
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    frames = []
    def emit(ws, we, bb, n_hist, rot_strip_deg, strip_scale):
        xs = seq[:n_hist].tolist()
        ys = losses[:n_hist].tolist()
        frames.append(
            ch3_frame(
                ws,
                we,
                bb,
                which,
                study,
                exam,
                y,
                spec,
                xs,
                ys,
                knob_rgbs,
                canvas_sides,
                show_colormap=show_cmap,
                rot_strip_deg=rot_strip_deg,
                strip_scale=strip_scale,
                x_lim_loss=(x_lo, x_hi),
                y_lim_loss=(y_lo, y_hi),
            )
        )
    ws0, we0, bb0 = triples[0]
    if CH3_N_GROW <= 1:
        ug = [1.0]
    else:
        ug = [float(j) / float(CH3_N_GROW - 1) for j in range(CH3_N_GROW)]
    for u in ug:
        sc = 1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
        emit(ws0, we0, bb0, 1, 0.0, sc)
    for i in range(len(seq)):
        ws, we, bb = triples[i]
        val = ch3_active_value(which, ws, we, bb)
        rot = ch3_knob_rots_for_weights(*ch3_script_weights(which, val, w_st, w_el, b))[{"st": 0, "el": 1, "b": 2}[which]]
        emit(ws, we, bb, i + 1, rot, CH3_KNOB_ACTIVE_SCALE)
    ws1, we1, bb1 = triples[-1]
    val_end = ch3_active_value(which, ws1, we1, bb1)
    rot_end = ch3_knob_rots_for_weights(*triples[-1])[{"st": 0, "el": 1, "b": 2}[which]]
    if CH3_N_SHRINK <= 1:
        us = [1.0]
    else:
        us = [float(j) / float(CH3_N_SHRINK - 1) for j in range(CH3_N_SHRINK)]
    for u in us:
        sc = CH3_KNOB_ACTIVE_SCALE - (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
        emit(ws1, we1, bb1, len(seq), rot_end, sc)
    return frames
def ch3_export_mp4(loss_key, which, with_noise):
    tag = "noise" if with_noise else "clean"
    fn = ch3_export_filename(f"{loss_key}_{which}_{tag}", fallback=f"ch3_{loss_key}_{which}_{tag}.mp4")
    frames = ch3_build_frames(loss_key, which, with_noise)
    save_mp4(frames, fn)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
# --- Triptych: three 1D loss slices ($w_1$, $w_2$, $b$) + dataset / knobs; then arrows for descent/ascent ---
CH3_TRIP_FIGSIZE = (18.5, 11.0)
CH3_TRIP_WIDTH_RATIOS = (1.02, 1.38)
CH3_TRIP_WSPACE = 0.10
CH3_TRIP_LOSS_PANEL_HSPACE = 0.20
CH3_TRIP_HOLD_END = max(14, _smooth_n(10))
CH3_TRIP_MS = CH3_MP4_MS_PER_FRAME
CH3_TRIP_ARROW_FS = 26
_CH3_TRIP_PARAM_LABEL = {"st": r"$w_1$", "el": r"$w_2$", "b": r"$b$"}
def ch3_figure_duo_triptych():
    fig = plt.figure(figsize=CH3_TRIP_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    ax_data = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    g_r = GridSpecFromSubplotSpec(
        3, 1, subplot_spec=gs[0, 1], hspace=CH3_TRIP_LOSS_PANEL_HSPACE
    )
    ax_st = fig.add_subplot(g_r[0, 0])
    ax_el = fig.add_subplot(g_r[1, 0])
    ax_b = fig.add_subplot(g_r[2, 0])
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, (ax_st, ax_el, ax_b), axes_k
def _ch3_triptych_pack_knob_rots(ws, we, bb, ref_ws=None, ref_we=None, ref_bb=None):
    _ = ref_ws, ref_we, ref_bb
    return list(ch3_knob_rots_for_weights(float(ws), float(we), float(bb)))
def _ch3_triptych_arrow_better_increase(loss_fn, study, exam, y, wr, we, br, which, *, maximize):
    """If True, increasing the parameter improves the objective (lower loss unless ``maximize``)."""
    wr, we, br = float(wr), float(we), float(br)
    v0 = ch3_active_value(which, wr, we, br)
    h = 1e-3 * max(1.0, abs(v0)) + 1e-5
    if which == "st":
        Lp = float(loss_fn(wr + h, we, br, study, exam, y))
        Lm = float(loss_fn(wr - h, we, br, study, exam, y))
    elif which == "el":
        Lp = float(loss_fn(wr, we + h, br, study, exam, y))
        Lm = float(loss_fn(wr, we - h, br, study, exam, y))
    else:
        Lp = float(loss_fn(wr, we, br + h, study, exam, y))
        Lm = float(loss_fn(wr, we, br - h, study, exam, y))
    if not (np.isfinite(Lp) and np.isfinite(Lm)):
        return None
    if abs(Lp - Lm) <= 1e-9 * max(1.0, abs(Lp), abs(Lm)):
        return None
    if maximize:
        return Lp > Lm
    return Lp < Lm
def _ch3_triptych_draw_left_arrows(axes):
    ax = axes[0]
    ax.annotate(
        r"$\leftarrow$",
        xy=(0.5, 0.9),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=CH3_TRIP_ARROW_FS,
        color="0.15",
        zorder=10,
    )


def _ch3_triptych_draw_arrows(axes, ups):
    for ax, up in zip(axes, ups):
        if up is None:
            continue
        sym = r"$\uparrow$" if up else r"$\downarrow$"
        ax.annotate(
            sym,
            xy=(0.5, 0.9),
            xycoords="axes fraction",
            ha="center",
            va="center",
            fontsize=CH3_TRIP_ARROW_FS,
            color="0.15",
            zorder=10,
        )
def ch3_triptych_frame(
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    spec,
    *,
    show_colormap,
    xs_st,
    ys_st,
    xs_el,
    ys_el,
    xs_b,
    ys_b,
    x_lim_st,
    x_lim_el,
    x_lim_b,
    y_lo,
    y_hi,
    knob_rots,
    knob_scales,
    arrows,
    sigma_Z=None,
    panel_visible=(True, True, True),
    emphasize_knob="st",
):
    """Triptych: three right slots; hide panels with no sweep data yet."""
    fig, ax_data, axes_loss, axes_k = ch3_figure_duo_triptych()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, emphasize_knob)
    ch3_draw_left_panel(
        ax_data,
        w_st,
        w_el,
        b,
        study,
        exam,
        y,
        leg,
        show_colormap=show_colormap,
        highlight_mistakes_flag=True,
        sigma_Z=sigma_Z,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig,
        axes_k,
        w_st,
        w_el,
        b,
        emphasize_knob,
        knob_rgbs,
        canvas_sides,
        rot_strip_deg=0.0,
        strip_scale=1.0,
        knob_rots=knob_rots,
        knob_scales=knob_scales,
        ax_data=ax_data,
    )
    ax_st_l, ax_el_l, ax_b_l = axes_loss
    y_lim = (y_lo, y_hi)
    panels = (
        (ax_st_l, xs_st, ys_st, x_lim_st, panel_visible[0]),
        (ax_el_l, xs_el, ys_el, x_lim_el, panel_visible[1]),
        (ax_b_l, xs_b, ys_b, x_lim_b, panel_visible[2]),
    )
    for ax_p, xs, ys, xlim_p, vis in panels:
        if vis and len(xs) > 0:
            ch3_draw_loss_panel(
                ax_p, xs, ys, spec, xlim_p, y_lim, title_override="", show_axis_labels=False
            )
        else:
            ch3_clear_loss_panel(ax_p, xlim_p, y_lim)
    if arrows is not None:
        _ch3_triptych_draw_arrows((ax_st_l, ax_el_l, ax_b_l), arrows)
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_triptych_script_weights():
    """Reference weights for ch3_15+ triptych exports ($w_1$ at 1/4 of script range)."""
    return float(CH3_SCRIPT_W1_REF), float(CH3_SCRIPT_W2), float(CH3_SCRIPT_B)
def ch3_build_frames_triptych(loss_key, with_noise):
    spec = CH3_LOSS_SPECS[loss_key]
    loss_fn = spec["fn"]
    show_cmap = bool(spec["colormap"])
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    maximize = loss_key == "likelihood"
    wr, we, br = ch3_triptych_script_weights()
    seqs = {}
    triples = {}
    losses = {}
    for which in ("st", "el", "b"):
        c = ch3_active_value(which, wr, we, br)
        seqs[which] = ch3_quad_sweep(c, CH3_LOSS_SYM_DELTA, CH3_SWEEP_NSEG)
        triples[which] = [ch3_triplet(which, float(v)) for v in seqs[which]]
        losses[which] = np.array(
            [loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples[which]],
            dtype=float,
        )
    allL = np.concatenate([losses["st"], losses["el"], losses["b"]])
    pad_y = 0.06 * max(1e-6, float(np.nanmax(allL) - np.nanmin(allL)))
    y_lo = float(np.nanmin(allL) - pad_y)
    y_hi = float(np.nanmax(allL) + pad_y)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or abs(y_hi - y_lo) < 1e-9:
        y_lo, y_hi = y_lo - 1.0, y_hi + 1.0
    def _xlim_for(seq):
        span = float(np.max(seq) - np.min(seq))
        pad_x = max(0.06 * span, 0.08)
        return float(np.min(seq) - pad_x), float(np.max(seq) + pad_x)
    x_lim_st = _xlim_for(seqs["st"])
    x_lim_el = _xlim_for(seqs["el"])
    x_lim_b = _xlim_for(seqs["b"])
    stg, elg = ST_KNOB, EL_KNOB
    sigma_z_cache = {}
    if show_cmap:
        for which in ("st", "el", "b"):
            for ws, we, bb in triples[which]:
                k = _ch3_sigma_z_cache_key(ws, we, bb)
                sigma_z_cache[k] = sigmoid(logits_plane(ws, we, bb, stg, elg))
        k_ref = _ch3_sigma_z_cache_key(wr, we, br)
        sigma_z_cache[k_ref] = sigmoid(logits_plane(wr, we, br, stg, elg))
    frames = []
    def emit(ws, we, bb, n_st, n_el, n_b, knob_scales, arrows, *, phase_which=None):
        rots = _ch3_triptych_pack_knob_rots(ws, we, bb, wr, we, br)
        sz = None
        if show_cmap:
            sz = sigma_z_cache[_ch3_sigma_z_cache_key(ws, we, bb)]
        if phase_which is None:
            vis = (n_st > 0, n_el > 0, n_b > 0)
            emph = "all"
        else:
            vis = (
                phase_which == "st" and n_st > 0,
                phase_which == "el" and n_el > 0,
                phase_which == "b" and n_b > 0,
            )
            emph = phase_which
        sc = [float(s) for s in knob_scales]
        if any(s > 1.0 + 1e-6 for s in sc):
            idx = max(range(3), key=lambda i: sc[i])
            emph = ("st", "el", "b")[idx]
        frames.append(
            ch3_triptych_frame(
                ws,
                we,
                bb,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                xs_st=seqs["st"][:n_st].tolist(),
                ys_st=losses["st"][:n_st].tolist(),
                xs_el=seqs["el"][:n_el].tolist(),
                ys_el=losses["el"][:n_el].tolist(),
                xs_b=seqs["b"][:n_b].tolist(),
                ys_b=losses["b"][:n_b].tolist(),
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lo=y_lo,
                y_hi=y_hi,
                knob_rots=rots,
                knob_scales=knob_scales,
                arrows=arrows,
                sigma_Z=sz,
                panel_visible=vis,
                emphasize_knob=emph,
            )
        )
    if CH3_N_GROW <= 1:
        ug = [1.0]
    else:
        ug = [float(j) / float(CH3_N_GROW - 1) for j in range(CH3_N_GROW)]
    if CH3_N_SHRINK <= 1:
        us = [1.0]
    else:
        us = [float(j) / float(CH3_N_SHRINK - 1) for j in range(CH3_N_SHRINK)]
    def run_phase(which, n_prev_st, n_prev_el, n_prev_b):
        seq = seqs[which]
        trip = triples[which]
        ws0, we0, bb0 = trip[0]
        slot = {"st": 0, "el": 1, "b": 2}[which]
        for u in ug:
            sc = [1.0, 1.0, 1.0]
            sc[slot] = 1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
            emit(ws0, we0, bb0, n_prev_st, n_prev_el, n_prev_b, sc, None, phase_which=which)
        for i in range(len(seq)):
            ws, we, bb = trip[i]
            sc = [1.0, 1.0, 1.0]
            sc[slot] = CH3_KNOB_ACTIVE_SCALE
            emit(ws, we, bb, n_prev_st + (which == "st") * (i + 1), n_prev_el + (which == "el") * (i + 1), n_prev_b + (which == "b") * (i + 1), sc, None, phase_which=which)
        ws1, we1, bb1 = trip[-1]
        for u in us:
            sc = [1.0, 1.0, 1.0]
            sc[slot] = CH3_KNOB_ACTIVE_SCALE - (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
            emit(
                ws1,
                we1,
                bb1,
                n_prev_st + (which == "st") * len(seq),
                n_prev_el + (which == "el") * len(seq),
                n_prev_b + (which == "b") * len(seq),
                sc,
                None,
                phase_which=which,
            )
    run_phase("st", 0, 0, 0)
    run_phase("el", len(seqs["st"]), 0, 0)
    run_phase("b", len(seqs["st"]), len(seqs["el"]), 0)
    n_st = len(seqs["st"])
    n_el = len(seqs["el"])
    n_b = len(seqs["b"])
    up_st = _ch3_triptych_arrow_better_increase(loss_fn, study, exam, y, wr, we, br, "st", maximize=maximize)
    up_el = _ch3_triptych_arrow_better_increase(loss_fn, study, exam, y, wr, we, br, "el", maximize=maximize)
    up_b = _ch3_triptych_arrow_better_increase(loss_fn, study, exam, y, wr, we, br, "b", maximize=maximize)
    arr = (up_st, up_el, up_b)
    scales_done = [1.0, 1.0, 1.0]
    emit(wr, we, br, n_st, n_el, n_b, scales_done, arr)
    last = frames[-1]
    for _ in range(CH3_TRIP_HOLD_END):
        frames.append(last.copy())
    return frames
def ch3_export_mp4_triptych(loss_key, with_noise):
    tag = "noise" if with_noise else "clean"
    fn = ch3_export_filename(f"{loss_key}_triptych_{tag}", fallback=f"ch3_{loss_key}_triptych_{tag}.mp4")
    frames = ch3_build_frames_triptych(loss_key, with_noise)
    save_mp4(frames, fn, duration=CH3_TRIP_MS)
    del frames
    gc.collect()

CH3_PROB_DATA_LABEL_FS = NOTE_SIZE * 1.12
CH3_PROB_LABEL_DX = 0.20
CH3_PROB_LABEL_DY = 0.22
CH3_PROB_DATA_LABEL_COLOR = "#0d47a1"
CH3_PASS_LABEL_COLOR = "#1b5e20"
CH3_FAIL_LABEL_COLOR = "#b71c1c"
CH3_STRIP_STORY_MS = CH3_MP4_MS_PER_FRAME
CH3_STRIP_N_HOLD = max(12, _smooth_n(8))
CH3_STRIP_N_CMAP = max(22, _smooth_n(16))
CH3_STRIP_N_KNOB = max(28, _smooth_n(22))
CH3_STRIP_N_SLIDE = max(40, _smooth_n(30))
CH3_STRIP_N_LABEL_HOLD = max(14, _smooth_n(10))
CH3_STRIP_N_LABEL_REVEAL = max(2, _smooth_n(1))
CH3_STRIP_N_LABEL_PRE_FLIP = max(32, _smooth_n(24))
CH3_STRIP_N_PILE_PER = max(3, _smooth_n(2))
CH3_STRIP_N_PILE_FLY = max(18, _smooth_n(14))
CH3_STRIP_W_START = (1.4, -0.5, 0.0)
CH3_STRIP_W_MID = (1.0, -1.0, 0.0)
CH3_STRIP_W_AMP = (4.0, -4.0, 0.0)
CH3_STRIP_PT_99 = (5.0, 6.0)


def ch3_strip_data_rect(slide_u: float):
    d_strip, _ = ch3_ch2_strip_layout_rects()
    d_duo, _ = ch3_duo_left_layout_rects()
    return ch3_bridge_lerp_rect(float(slide_u), d_strip, d_duo)


def ch3_strip_knob_slot_lerp(slide_u: float, slot: int):
    _, k_strip = ch3_ch2_strip_layout_rects()
    _, k_duo = ch3_duo_left_layout_rects()
    return ch3_bridge_lerp_rect(float(slide_u), k_strip[slot], k_duo[slot])


def ch3_lerp_weights(w_a, w_b, t: float):
    t = float(np.clip(t, 0.0, 1.0))
    u = ch3_knob_smoothstep(t)
    return tuple(float(w_a[i] + u * (w_b[i] - w_a[i])) for i in range(3))


def ch3_student_pass_pct(study, exam, w_st, w_el, b):
    z = logits_plane(float(w_st), float(w_el), float(b), study, exam)
    return np.round(sigmoid(z) * 100.0).astype(int)


def ch3_student_label_indices(study, exam, y):
    pass_idx = [i for i in range(len(y)) if int(y[i]) == 1]
    fail_idx = [i for i in range(len(y)) if int(y[i]) == 0]
    return pass_idx, fail_idx


def ch3_pct_label_offset(sx, sy):
    """Default: tight offset; upper-left near legend: beside the point."""
    if float(sx) <= 1.55 and float(sy) >= 5.4:
        return 0.14, 0.0, "left", "center"
    return CH3_PROB_LABEL_DX, CH3_PROB_LABEL_DY, "left", "bottom"


def ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, *, fail_red=False):
    p = ch3_student_pass_pct(study, exam, w_st, w_el, b)
    sx, sy = float(study[i]), float(exam[i])
    if int(y[i]) == 1:
        return (f"{int(p[i])}%", sx, sy, CH3_PASS_LABEL_COLOR)
    if fail_red:
        return (f"{int(100 - p[i])}%", sx, sy, CH3_FAIL_LABEL_COLOR)
    return (f"{int(p[i])}%", sx, sy, CH3_PASS_LABEL_COLOR)


def ch3_row_to_data_label(row):
    txt, sx, sy, col = row
    dx, dy, ha, va = ch3_pct_label_offset(sx, sy)
    return (txt, sx, sy, CH3_PROB_DATA_LABEL_FS, col, dx, dy, ha, va)


def ch3_build_student_label_rows(study, exam, y, w_st, w_el, b, *, fail_red=False):
    return [
        ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=fail_red)
        for i in range(len(study))
    ]


def ch3_rows_to_data_labels(rows):
    return [ch3_row_to_data_label(r) for r in rows]


def ch3_draw_pct_label(ax, text, x, y, color, *, dx=0.20, dy=0.22, ha="left", va="bottom"):
    ax.annotate(
        str(text),
        (float(x), float(y)),
        xytext=(float(x) + float(dx), float(y) + float(dy)),
        textcoords="data",
        ha=str(ha),
        va=str(va),
        fontsize=float(CH3_PROB_DATA_LABEL_FS),
        fontweight="bold",
        color=str(color),
        zorder=25,
    )


def ch3_place_strip_knobs(fig, w_st, w_el, b, slide_u: float):
    ch3_ensure_knob_pngs()
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    rots = ch3_knob_rots_for_weights(float(w_st), float(w_el), float(b))
    for i in range(3):
        r = ch3_strip_knob_slot_lerp(slide_u, i)
        cx, cy, _, _ = ch3_bridge_rect_center_wh(r)
        side = ch3_bridge_knob_side_from_slot(r)
        arr = np.asarray(
            _ch3_knob_pil_rotated_square(knob_rgbs[i], rots[i], canvas_sides[i]), dtype=np.uint8
        )
        ch3_bridge_add_knob(fig, Image.fromarray(arr), cx, cy, side)


def ch3_pile_axis_rect_aligned(slide_u=1.0):
    """Right pile axis: same vertical extent as the dataset axes (duo-left data slot)."""
    data_r = ch3_strip_data_rect(float(slide_u))
    pile_r = ch3_duo_right_axis_rect()
    return (pile_r[0], data_r[1], pile_r[2], data_r[3])


def ch3_pile_bar_geom():
    return 0.19, 0.62


def ch3_data_to_fig_xy(fig, ax, x, y):
    return fig.transFigure.inverted().transform(ax.transData.transform((float(x), float(y))))


def ch3_data_to_fig_wh(fig, ax, w_data, h_data):
    p0 = ax.transData.transform((0.0, 0.0))
    p1 = ax.transData.transform((float(w_data), float(h_data)))
    f0 = fig.transFigure.inverted().transform(p0)
    f1 = fig.transFigure.inverted().transform(p1)
    return abs(float(f1[0] - f0[0])), abs(float(f1[1] - f0[1]))


def ch3_cmap_color_for_pass_prob(p_pass):
    """Face color from the σ colormap at P(pass) (same scale as the dataset heatmap)."""
    t = float(np.clip(float(p_pass), 0.0, 1.0))
    return mpl.colors.to_hex(CMAP_GD(t), keep_alpha=False)


def ch3_pile_block_specs(study, exam, y, w_st, w_el, b, label_rows):
    """Per student: (study, exam, prob height in [0,1], cmap color at P(pass))."""
    z = logits_plane(float(w_st), float(w_el), float(b), study, exam)
    p = sigmoid(z)
    specs = []
    for i, row in enumerate(label_rows):
        sx, sy = float(row[1]), float(row[2])
        p_pass = float(p[i])
        h = p_pass if int(y[i]) == 1 else float(1.0 - p_pass)
        col = ch3_cmap_color_for_pass_prob(p_pass)
        specs.append((sx, sy, h, col))
    return specs


def ch3_pile_landed_blocks(specs, n_landed):
    return [(float(specs[i][2]), specs[i][3]) for i in range(int(n_landed))]


def ch3_pile_blocks_for_labels(label_rows, y_arr):
    """Legacy: integer %% blocks (unused by flying pile)."""
    return [(int(str(txt).replace("%", "")), col) for txt, _sx, _sy, col in label_rows]


def ch3_draw_bar_pile(ax, blocks, *, y_span):
    """Stack blocks; each height is a probability mass in [0, 1] (max stack = y_span students)."""
    ax.clear()
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, float(y_span))
    ax.set_xticks([])
    n_st = int(round(float(y_span)))
    ax.set_ylabel("pile height (students)", fontsize=ANNOT_SIZE)
    ax.tick_params(axis="y", labelsize=ANNOT_SIZE)
    x0, bar_w = ch3_pile_bar_geom()
    cum = 0.0
    for h, col in blocks:
        hh = float(h)
        if hh > 1e-9:
            ax.add_patch(
                plt.Rectangle(
                    (x0, cum),
                    bar_w,
                    hh,
                    facecolor=col,
                    edgecolor="0.2",
                    linewidth=0.9,
                    zorder=2,
                )
            )
            cum += hh
    ax.text(0.5, float(y_span), str(n_st), ha="center", va="bottom", fontsize=ANNOT_SIZE, color="0.35")
    ax.text(0.5, 0.0, "0", ha="center", va="top", fontsize=ANNOT_SIZE, color="0.35")
    for sp in ax.spines.values():
        sp.set_visible(True)
    ax.grid(axis="y", alpha=0.2)


def ch3_draw_flying_pile_block(fig, axd, axp, spec, cum_h, y_span, t_fly):
    """Block grows from minuscule at the data point while flying to the pile slot."""
    sx, sy, h, col = spec
    u = ch3_knob_smoothstep(float(t_fly))
    x0, bar_w = ch3_pile_bar_geom()
    cx0, cy0 = ch3_data_to_fig_xy(fig, axd, sx, sy)
    h_vis = float(h) * u
    cx1, cy1 = ch3_data_to_fig_xy(fig, axp, x0 + bar_w * 0.5, float(cum_h) + h_vis * 0.5)
    fw1, fh1 = ch3_data_to_fig_wh(fig, axp, bar_w, h_vis)
    fw0 = fh0 = 0.004
    fw = fw0 + u * (fw1 - fw0)
    fh = fh0 + u * (fh1 - fh0)
    cx = cx0 + u * (cx1 - cx0)
    cy = cy0 + u * (cy1 - cy0)
    fig.add_artist(
        plt.Rectangle(
            (cx - fw * 0.5, cy - fh * 0.5),
            fw,
            fh,
            transform=fig.transFigure,
            facecolor=col,
            edgecolor="0.2",
            linewidth=0.9,
            zorder=30,
        )
    )


def ch3_frame_strip_dataset(
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    *,
    slide_u=0.0,
    cmap_alpha=None,
    highlight_mistakes=False,
    mistake_points=None,
    data_labels=None,
    pile_blocks=None,
    pile_n_show=None,
    show_pile_axis=False,
    pile_specs=None,
    pile_landed=0,
    pile_fly=None,
    likelihood_specs=None,
    likelihood_applied=0,
    likelihood_fly=None,
    likelihood_log_lo=None,
    likelihood_log_hi=None,
    show_likelihood_axis=False,
    pile_y_span=None,
):
    cmap_full = float(_CH3_SIGMA_CONTOUR_ALPHA)
    ca = cmap_full if cmap_alpha is None else float(cmap_alpha)
    stg, elg = ST_KNOB, EL_KNOB
    sigma_Z = sigmoid(logits_plane(float(w_st), float(w_el), float(b), stg, elg))
    data_r = ch3_strip_data_rect(slide_u)
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("white")
    axd = fig.add_axes(data_r)
    leg = legend_linear_equation_values_bold_param(float(w_st), float(w_el), float(b), None)
    ch3_draw_left_panel(
        axd,
        float(w_st),
        float(w_el),
        float(b),
        study,
        exam,
        y,
        leg,
        show_colormap=ca > 1e-5,
        highlight_mistakes_flag=bool(highlight_mistakes),
        sigma_Z=sigma_Z if ca > 1e-5 else None,
        colormap_alpha=ca,
        mistake_points=mistake_points,
    )
    axd.set_xlim(*xlim)
    axd.set_ylim(*ylim)
    finalize_style_legend_tex(axd)
    if data_labels:
        for item in data_labels:
            if item is None:
                continue
            txt, tx, ty = item[0], float(item[1]), float(item[2])
            col = item[4] if len(item) > 4 else CH3_PASS_LABEL_COLOR
            dx = float(item[5]) if len(item) > 5 else CH3_PROB_LABEL_DX
            dy = float(item[6]) if len(item) > 6 else CH3_PROB_LABEL_DY
            ha = item[7] if len(item) > 7 else "left"
            va = item[8] if len(item) > 8 else "bottom"
            ch3_draw_pct_label(axd, txt, tx, ty, col, dx=dx, dy=dy, ha=ha, va=va)
    ch3_place_strip_knobs(fig, w_st, w_el, b, slide_u)
    if show_pile_axis or pile_fly is not None:
        pile_r = ch3_pile_axis_rect_aligned(slide_u)
        ax_p = fig.add_axes(pile_r)
        y_span = float(len(study) if pile_y_span is None else pile_y_span)
        if pile_specs is not None:
            landed = ch3_pile_landed_blocks(pile_specs, pile_landed)
        else:
            n = len(pile_blocks) if pile_blocks is not None else 0
            k = n if pile_n_show is None else int(min(max(0, pile_n_show), n))
            landed = [(float(pct) / 100.0, col) for pct, col in (pile_blocks[:k] if pile_blocks else [])]
        ch3_draw_bar_pile(ax_p, landed, y_span=y_span)
        fig.canvas.draw()
        if pile_fly is not None:
            spec, cum_h, t_fly = pile_fly
            ch3_draw_flying_pile_block(fig, axd, ax_p, spec, float(cum_h), y_span, float(t_fly))
    if show_likelihood_axis or likelihood_fly is not None:
        lik_r = ch3_pile_axis_rect_aligned(slide_u)
        ax_l = fig.add_axes(lik_r)
        specs_l = likelihood_specs
        if specs_l is None:
            specs_l = []
        ch3_draw_likelihood_meter(
            ax_l,
            specs_l,
            int(likelihood_applied),
            log_lo=float(likelihood_log_lo if likelihood_log_lo is not None else 0.0),
            log_hi=float(likelihood_log_hi if likelihood_log_hi is not None else 1.0),
            fly=likelihood_fly,
        )
        fig.canvas.draw()
        if likelihood_fly is not None:
            spec_f, t_f = likelihood_fly
            ch3_draw_flying_likelihood_factor(fig, axd, ax_l, spec_f, float(t_f))
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def _ch3_strip_append_hold(frames, im, n=None):
    n = int(CH3_STRIP_N_HOLD if n is None else n)
    for _ in range(max(1, n)):
        frames.append(im.copy())


def ch3_build_frames_mistakes_strip_prob_story(with_noise=False):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    w = CH3_STRIP_W_START
    pt = CH3_STRIP_PT_99
    lbl99 = ch3_row_to_data_label(("99%", pt[0], pt[1], CH3_PASS_LABEL_COLOR))
    frames = []
    im = ch3_frame_strip_dataset(
        *w, study, exam, y, slide_u=0.0, cmap_alpha=0.0, highlight_mistakes=True,
    )
    _ch3_strip_append_hold(frames, im)
    for i in range(1, int(CH3_STRIP_N_CMAP) + 1):
        t = ch3_knob_smoothstep(float(i) / float(CH3_STRIP_N_CMAP))
        frames.append(
            ch3_frame_strip_dataset(
                *w, study, exam, y, slide_u=0.0,
                cmap_alpha=t * float(_CH3_SIGMA_CONTOUR_ALPHA), highlight_mistakes=True,
            )
        )
    im = ch3_frame_strip_dataset(
        *w, study, exam, y, slide_u=0.0,
        cmap_alpha=float(_CH3_SIGMA_CONTOUR_ALPHA), highlight_mistakes=False,
    )
    _ch3_strip_append_hold(frames, im)
    im = ch3_frame_strip_dataset(
        *w, study, exam, y, slide_u=0.0,
        cmap_alpha=float(_CH3_SIGMA_CONTOUR_ALPHA), highlight_mistakes=False,
        data_labels=[lbl99],
    )
    _ch3_strip_append_hold(frames, im)
    im = ch3_frame_strip_dataset(
        *w, study, exam, y, slide_u=0.0,
        cmap_alpha=float(_CH3_SIGMA_CONTOUR_ALPHA), highlight_mistakes=False,
        data_labels=None,
    )
    _ch3_strip_append_hold(frames, im)
    return frames



def ch3_pass_pct_at_point(w_st, w_el, b, sx, sy):
    z = logits_plane(float(w_st), float(w_el), float(b), np.array([float(sx)]), np.array([float(sy)]))
    return int(np.round(float(sigmoid(z)[0]) * 100.0))


def ch3_strip_pt_data_label(w_st, w_el, b, pt):
    pct = ch3_pass_pct_at_point(w_st, w_el, b, pt[0], pt[1])
    return ch3_row_to_data_label((f"{pct}%", float(pt[0]), float(pt[1]), CH3_PASS_LABEL_COLOR))



def ch3_pt99_student_index(study, exam, pt=CH3_STRIP_PT_99):
    for i in range(len(study)):
        if abs(float(study[i]) - float(pt[0])) < 1e-9 and abs(float(exam[i]) - float(pt[1])) < 1e-9:
            return i
    return None


def ch3_data_labels_with_pt99(w_st, w_el, b, label_rows=()):
    """Always include the tracked (5,6) P(pass)%% label plus any student rows."""
    out = [ch3_strip_pt_data_label(float(w_st), float(w_el), float(b), CH3_STRIP_PT_99)]
    if label_rows:
        out.extend(ch3_rows_to_data_labels(label_rows))
    return out


def ch3_build_frames_strip_knob_path(
    w_from,
    w_to,
    with_noise=False,
    *,
    slide_u=0.0,
    show_pile=False,
    pile_blocks=None,
    pile_n_show=None,
    label_rows=None,
    track_pt=None,
):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    frames = []
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_KNOB), endpoint=True):
        w = ch3_lerp_weights(w_from, w_to, float(tv))
        if label_rows is not None:
            labels = ch3_rows_to_data_labels(label_rows)
        elif track_pt is not None:
            labels = [ch3_strip_pt_data_label(*w, track_pt)]
        else:
            labels = None
        frames.append(
            ch3_frame_strip_dataset(
                *w, study, exam, y, slide_u=slide_u, cmap_alpha=cmap_a,
                highlight_mistakes=False, data_labels=labels,
                pile_blocks=pile_blocks, pile_n_show=pile_n_show, show_pile_axis=show_pile,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    return frames


def ch3_build_frames_strip_prob_labels(w_st, w_el, b, with_noise=False):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    w_st, w_el, b = float(w_st), float(w_el), float(b)
    frames = []
    pt_labels_only = ch3_data_labels_with_pt99(w_st, w_el, b)
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_SLIDE), endpoint=True):
        frames.append(
            ch3_frame_strip_dataset(
                w_st, w_el, b, study, exam, y,
                slide_u=float(tv), cmap_alpha=cmap_a, highlight_mistakes=False,
                data_labels=pt_labels_only,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    pass_idx, fail_idx = ch3_student_label_indices(study, exam, y)
    i99 = ch3_pt99_student_index(study, exam)
    if i99 is not None:
        pass_idx = [i for i in pass_idx if i != i99]
        fail_idx = [i for i in fail_idx if i != i99]
    n_rev = int(CH3_STRIP_N_LABEL_REVEAL)

    def _append_labeled(rows):
        labels = ch3_data_labels_with_pt99(w_st, w_el, b, rows)
        im = ch3_frame_strip_dataset(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False, data_labels=labels,
        )
        for _ in range(n_rev):
            frames.append(im)

    shown = []
    for i in pass_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
        _append_labeled(shown)
    for i in fail_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
        _append_labeled(shown)

    labels_pre = ch3_data_labels_with_pt99(w_st, w_el, b, shown)
    im_hold = ch3_frame_strip_dataset(
        w_st, w_el, b, study, exam, y,
        slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False, data_labels=labels_pre,
    )
    _ch3_strip_append_hold(frames, im_hold, CH3_STRIP_N_LABEL_PRE_FLIP)

    rows_red = ch3_build_student_label_rows(study, exam, y, w_st, w_el, b, fail_red=True)
    if i99 is not None:
        rows_red_no99 = [rows_red[i] for i in range(len(rows_red)) if i != i99]
        labels_fail = ch3_data_labels_with_pt99(w_st, w_el, b, rows_red_no99)
    else:
        labels_fail = ch3_data_labels_with_pt99(w_st, w_el, b, rows_red)
    im = ch3_frame_strip_dataset(
        w_st, w_el, b, study, exam, y,
        slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False, data_labels=labels_fail,
    )
    _ch3_strip_append_hold(frames, im, CH3_STRIP_N_LABEL_HOLD)
    return frames, rows_red



def ch3_strip_data_labels_for_pile(label_rows, pile_landed, pile_fly=None):
    """Hide labels once a block flies away (and for students already on the pile)."""
    hide = int(pile_landed) + (1 if pile_fly is not None else 0)
    if hide <= 0:
        return ch3_rows_to_data_labels(label_rows)
    return ch3_rows_to_data_labels(label_rows[hide:])


def ch3_build_frames_strip_bar_pile(w_st, w_el, b, label_rows, with_noise=False):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    specs = ch3_pile_block_specs(study, exam, y, w_st, w_el, b, label_rows)
    y_span = float(len(study))
    n_fly = int(CH3_STRIP_N_PILE_FLY)
    frames = []
    for k, spec in enumerate(specs):
        cum = sum(float(specs[i][2]) for i in range(k))
        for j in range(1, n_fly + 1):
            t = float(j) / float(n_fly)
            frames.append(
                ch3_frame_strip_dataset(
                    float(w_st), float(w_el), float(b), study, exam, y,
                    slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False,
                    data_labels=ch3_strip_data_labels_for_pile(
                        label_rows, k, pile_fly=(spec, cum, t),
                    ),
                    show_pile_axis=True,
                    pile_specs=specs, pile_landed=k, pile_fly=(spec, cum, t),
                    pile_y_span=y_span,
                )
            )
        frames.append(
            ch3_frame_strip_dataset(
                float(w_st), float(w_el), float(b), study, exam, y,
                slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False,
                data_labels=ch3_strip_data_labels_for_pile(label_rows, k + 1),
                show_pile_axis=True,
                pile_specs=specs, pile_landed=k + 1, pile_y_span=y_span,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    return frames


def ch3_export_strip_story(key, frames):
    fn = ch3_export_filename(key)
    save_mp4(frames, fn, duration=CH3_STRIP_STORY_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)


def ch3_export_script_mistakes_strip_prob_story(with_noise=False):
    frames = ch3_build_frames_mistakes_strip_prob_story(with_noise)
    ch3_export_strip_story("mistakes_strip_prob_story", frames)


def ch3_export_script_strip_knob_to_mid(with_noise=False):
    """Knobs (1.4,-0.5,0)→(1,-1,0) with live P(pass)%% label at CH3_STRIP_PT_99."""
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    pt = CH3_STRIP_PT_99
    frames = []
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_KNOB), endpoint=True):
        w = ch3_lerp_weights(CH3_STRIP_W_START, CH3_STRIP_W_MID, float(tv))
        labels = [ch3_strip_pt_data_label(*w, pt)]
        frames.append(
            ch3_frame_strip_dataset(
                *w, study, exam, y, slide_u=0.0, cmap_alpha=cmap_a,
                highlight_mistakes=False, data_labels=labels,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    ch3_export_strip_story("mistakes_strip_knob_to_mid", frames)


def ch3_export_script_strip_knob_to_amp(with_noise=False):
    """Knobs (1,-1,0)→(4,-4,0) with live P(pass)%% label at CH3_STRIP_PT_99."""
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    pt = CH3_STRIP_PT_99
    frames = []
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_KNOB), endpoint=True):
        w = ch3_lerp_weights(CH3_STRIP_W_MID, CH3_STRIP_W_AMP, float(tv))
        labels = [ch3_strip_pt_data_label(*w, pt)]
        frames.append(
            ch3_frame_strip_dataset(
                *w, study, exam, y, slide_u=0.0, cmap_alpha=cmap_a,
                highlight_mistakes=False, data_labels=labels,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    ch3_export_strip_story("mistakes_strip_knob_to_amp", frames)


def ch3_export_script_strip_prob_labels(with_noise=False):
    w = CH3_STRIP_W_AMP
    frames, _ = ch3_build_frames_strip_prob_labels(*w, with_noise)
    ch3_export_strip_story("mistakes_strip_prob_labels", frames)


def ch3_export_script_strip_pile_amp(with_noise=False):
    w = CH3_STRIP_W_AMP
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    rows = ch3_build_student_label_rows(study, exam, y, *w, fail_red=True)
    frames = ch3_build_frames_strip_bar_pile(*w, rows, with_noise)
    ch3_export_strip_story("mistakes_strip_pile_amp", frames)



CH3_STRIP_N_LIKELIHOOD_FLY = max(18, _smooth_n(14))
CH3_STRIP_N_CONTRAST = max(28, _smooth_n(22))
CH3_STRIP_N_CONTRAST_HOLD = max(16, _smooth_n(12))


def ch3_likelihood_factor_specs(study, exam, y, w_st, w_el, b, label_rows):
    """Per student: (sx, sy, p(y_i|x_i), cmap color) — same masses as the bar pile."""
    return ch3_pile_block_specs(study, exam, y, w_st, w_el, b, label_rows)


def ch3_likelihood_log_partial(specs, n_applied):
    if int(n_applied) <= 0:
        return 0.0
    s = 0.0
    for i in range(int(n_applied)):
        p = max(float(specs[i][2]), 1e-12)
        s += float(np.log(p))
    return s


def ch3_likelihood_value_from_log(logL):
    return float(np.exp(np.clip(float(logL), -700.0, 700.0)))


def ch3_likelihood_log_span(specs_a, specs_b=None, *, margin=0.08):
    logs = [ch3_likelihood_log_partial(specs_a, len(specs_a))]
    if specs_b is not None:
        logs.append(ch3_likelihood_log_partial(specs_b, len(specs_b)))
    lo, hi = float(min(logs)), float(max(logs))
    pad = margin * max(1e-6, hi - lo)
    return lo - pad, hi + pad


def ch3_likelihood_pile_sums(specs):
    """Additive pile height (sum of per-student probabilities)."""
    return float(sum(float(specs[i][2]) for i in range(len(specs))))


def ch3_draw_likelihood_meter(
    ax,
    specs,
    n_applied,
    *,
    log_lo,
    log_hi,
    fly=None,
    show_chain=True,
):
    """Right panel: log-scale meter for running product likelihood."""
    ax.clear()
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True)
    ax.set_title(
        r"Likelihood $\mathcal{L}=\prod_i p(y_i|x_i)$",
        fontsize=TITLE_SIZE,
        pad=6,
    )
    logL = ch3_likelihood_log_partial(specs, n_applied)
    fly_txt = None
    if fly is not None:
        spec, t = fly
        p = max(float(spec[2]), 1e-12)
        log_prev = ch3_likelihood_log_partial(specs, n_applied)
        log_next = log_prev + float(np.log(p))
        u = ch3_knob_smoothstep(float(t))
        logL = log_prev + u * (log_next - log_prev)
        fly_txt = f"×{p:.2f}"
    L = ch3_likelihood_value_from_log(logL)
    span = float(log_hi) - float(log_lo)
    frac = (float(logL) - float(log_lo)) / span if span > 1e-12 else 0.0
    frac = float(np.clip(frac, 0.0, 1.0))
    x0, bar_w = ch3_pile_bar_geom()
    ax.add_patch(
        plt.Rectangle(
            (x0, 0.0),
            bar_w,
            frac,
            facecolor="#5c6bc0",
            edgecolor="0.25",
            linewidth=1.0,
            zorder=2,
        )
    )
    ax.axhline(0.0, color="0.35", linewidth=0.9, zorder=1)
    ax.text(
        0.5,
        0.92,
        rf"$\mathcal{{L}}={L:.3g}$",
        ha="center",
        va="top",
        fontsize=ANNOT_SIZE * 1.15,
        color="0.12",
        transform=ax.transAxes,
        zorder=5,
    )
    ax.text(
        0.5,
        0.06,
        r"$\log_{10}\mathcal{L}$" + f"={logL / np.log(10):.2f}",
        ha="center",
        va="bottom",
        fontsize=ANNOT_SIZE * 0.95,
        color="0.4",
        transform=ax.transAxes,
        zorder=5,
    )
    if show_chain and int(n_applied) > 0:
        parts = [f"×{float(specs[i][2]):.2f}" for i in range(int(n_applied))]
        if fly_txt is not None:
            parts.append(fly_txt)
        chain = " ".join(parts[-8:])
        if len(parts) > 8:
            chain = "… " + chain
        ax.text(
            0.5,
            0.78,
            chain,
            ha="center",
            va="top",
            fontsize=ANNOT_SIZE * 0.82,
            color="0.35",
            transform=ax.transAxes,
            zorder=5,
        )
    ax.text(0.02, 0.02, "0", fontsize=ANNOT_SIZE, color="0.4", transform=ax.transAxes)
    ax.text(0.98, 0.98, "↑", fontsize=ANNOT_SIZE, color="0.4", ha="right", va="top", transform=ax.transAxes)
    ax.grid(axis="y", alpha=0.15)


def ch3_draw_flying_likelihood_factor(fig, axd, axm, spec, t_fly):
    """Factor badge flies from the student to the likelihood meter."""
    sx, sy, p, col = spec
    p = max(float(p), 1e-12)
    u = ch3_knob_smoothstep(float(t_fly))
    cx0, cy0 = ch3_data_to_fig_xy(fig, axd, sx, sy)
    x0, bar_w = ch3_pile_bar_geom()
    cx1, cy1 = ch3_data_to_fig_xy(fig, axm, x0 + bar_w * 0.5, 0.12)
    cx = cx0 + u * (cx1 - cx0)
    cy = cy0 + u * (cy1 - cy0)
    fs = ANNOT_SIZE * 0.9
    fig.text(
        cx,
        cy,
        f"×{p:.2f}",
        ha="center",
        va="center",
        fontsize=fs,
        color="0.1",
        transform=fig.transFigure,
        zorder=35,
        bbox=dict(boxstyle="round,pad=0.25", facecolor=col, edgecolor="0.25", alpha=0.95),
    )


def ch3_strip_data_labels_for_likelihood(label_rows, n_applied, likelihood_fly=None):
    hide = int(n_applied) + (1 if likelihood_fly is not None else 0)
    if hide <= 0:
        return ch3_rows_to_data_labels(label_rows)
    return ch3_rows_to_data_labels(label_rows[hide:])


def ch3_build_frames_strip_likelihood_compound(w_st, w_el, b, label_rows, with_noise=False):
    """Like ch3_42/43 bar pile, but each student multiplies into likelihood on the right."""
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    cmap_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    specs = ch3_likelihood_factor_specs(study, exam, y, w_st, w_el, b, label_rows)
    log_lo, log_hi = ch3_likelihood_log_span(specs)
    n_fly = int(CH3_STRIP_N_LIKELIHOOD_FLY)
    frames = []
    for k in range(len(specs)):
        for j in range(1, n_fly + 1):
            t = float(j) / float(n_fly)
            frames.append(
                ch3_frame_strip_dataset(
                    float(w_st), float(w_el), float(b), study, exam, y,
                    slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False,
                    data_labels=ch3_strip_data_labels_for_likelihood(
                        label_rows, k, likelihood_fly=(specs[k], t),
                    ),
                    show_likelihood_axis=True,
                    likelihood_specs=specs,
                    likelihood_applied=k,
                    likelihood_fly=(specs[k], t),
                    likelihood_log_lo=log_lo,
                    likelihood_log_hi=log_hi,
                )
            )
        frames.append(
            ch3_frame_strip_dataset(
                float(w_st), float(w_el), float(b), study, exam, y,
                slide_u=1.0, cmap_alpha=cmap_a, highlight_mistakes=False,
                data_labels=ch3_strip_data_labels_for_likelihood(label_rows, k + 1),
                show_likelihood_axis=True,
                likelihood_specs=specs,
                likelihood_applied=k + 1,
                likelihood_log_lo=log_lo,
                likelihood_log_hi=log_hi,
            )
        )
    _ch3_strip_append_hold(frames, frames[-1])
    return frames


def ch3_export_script_strip_likelihood_amp(with_noise=False):
    w = CH3_STRIP_W_AMP
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    rows = ch3_build_student_label_rows(study, exam, y, *w, fail_red=True)
    frames = ch3_build_frames_strip_likelihood_compound(*w, rows, with_noise)
    ch3_export_strip_story("mistakes_strip_likelihood_amp", frames)


def ch3_export_script_strip_likelihood_start(with_noise=False):
    w = CH3_STRIP_W_START
    rows = ch3_build_student_label_rows(
        study_sep, exam_sep, y_sep, *w, fail_red=True,
    )
    frames = ch3_build_frames_strip_likelihood_compound(*w, rows, with_noise)
    ch3_export_strip_story("mistakes_strip_likelihood_start", frames)


def ch3_frame_pile_vs_likelihood_contrast(*, phase="intro", bar_t=1.0):
    """
    Compare |Δ| between ch3_43→42 pile (additive sum) vs ch3_57→56 likelihood (log-scale).
    ``phase``: intro | pile | likelihood | both
    """
    study, exam, y = study_sep, exam_sep, y_sep
    rows_s = ch3_build_student_label_rows(study, exam, y, *CH3_STRIP_W_START, fail_red=True)
    rows_a = ch3_build_student_label_rows(study, exam, y, *CH3_STRIP_W_AMP, fail_red=True)
    spec_s = ch3_likelihood_factor_specs(study, exam, y, *CH3_STRIP_W_START, rows_s)
    spec_a = ch3_likelihood_factor_specs(study, exam, y, *CH3_STRIP_W_AMP, rows_a)
    sum_s = ch3_likelihood_pile_sums(spec_s)
    sum_a = ch3_likelihood_pile_sums(spec_a)
    log_s = ch3_likelihood_log_partial(spec_s, len(spec_s))
    log_a = ch3_likelihood_log_partial(spec_a, len(spec_a))
    L_s = ch3_likelihood_value_from_log(log_s)
    L_a = ch3_likelihood_value_from_log(log_a)
    d_sum = abs(sum_a - sum_s)
    d_log = abs(log_a - log_s)
    ratio = d_log / max(d_sum, 1e-12)

    data_r = ch3_strip_data_rect(1.0)
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("white")
    axd = fig.add_axes(data_r)
    w_show = CH3_STRIP_W_START
    leg = legend_linear_equation_values_bold_param(*w_show, None)
    ch3_draw_left_panel(
        axd, *w_show, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
        sigma_Z=None,
        colormap_alpha=float(_CH3_SIGMA_CONTOUR_ALPHA),
    )
    axd.set_xlim(*xlim)
    axd.set_ylim(*ylim)
    finalize_style_legend_tex(axd)
    ch3_place_strip_knobs(fig, *w_show, 1.0)

    pile_r = ch3_pile_axis_rect_aligned(1.0)
    lik_r = ch3_duo_right_axis_rect()
    # stack two panels in the right column
    pr = pile_r
    lr = lik_r
    mid_y = pr[1] + pr[3] * 0.52
    ax_pile = fig.add_axes((pr[0], pr[1], pr[2], pr[3] * 0.46))
    ax_lik = fig.add_axes((lr[0], mid_y, lr[2], pr[1] + pr[3] - mid_y))

    bt = float(np.clip(bar_t, 0.0, 1.0))
    u = ch3_knob_smoothstep(bt)
    max_bar = max(d_sum, d_log, 1e-12)

    def _draw_delta_bar(ax, title, d_val, color, show, label_extra=""):
        ax.clear()
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title, fontsize=TITLE_SIZE * 0.92, pad=4)
        w_show_bar = (d_val / max_bar) * u if show else 0.0
        ax.add_patch(
            plt.Rectangle(
                (0.12, 0.35),
                0.76 * w_show_bar,
                0.28,
                facecolor=color,
                edgecolor="0.2",
                linewidth=0.9,
                zorder=2,
            )
        )
        ax.text(
            0.5, 0.78, label_extra, ha="center", va="center",
            fontsize=ANNOT_SIZE * 0.95, color="0.2", transform=ax.transAxes,
        )
        ax.text(
            0.5, 0.12, f"Δ = {d_val:.3g}", ha="center", va="center",
            fontsize=ANNOT_SIZE, color="0.35", transform=ax.transAxes,
        )

    show_pile = phase in ("pile", "both")
    show_lik = phase in ("likelihood", "both")
    _draw_delta_bar(
        ax_pile,
        "Additive (pile ch3_43→42)",
        d_sum,
        "#ef6c00",
        show_pile,
        f"Σ p(y|x): {sum_s:.1f} → {sum_a:.1f}",
    )
    _draw_delta_bar(
        ax_lik,
        "Multiplicative (ℒ ch3_57→56)",
        d_log,
        "#5c6bc0",
        show_lik,
        rf"$\log\mathcal{{L}}$: {log_s:.2f} → {log_a:.2f}",
    )
    if phase == "both" and u > 0.9:
        fig.text(
            0.5, 0.02,
            rf"Likelihood change is {ratio:.0f}× larger on log scale than pile height change (additive vs multiply).",
            ha="center", va="bottom", fontsize=ANNOT_SIZE * 1.05, color="0.15",
        )
    if phase == "intro":
        fig.text(
            0.5, 0.02,
            "Same two weight settings: (1.4,−0.5,0) vs (4,−4,0). Compare how much each summary moves.",
            ha="center", va="bottom", fontsize=ANNOT_SIZE, color="0.35",
        )
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def ch3_build_frames_pile_vs_likelihood_contrast():
    frames = []
    for _ in range(CH3_STRIP_N_CONTRAST_HOLD):
        frames.append(ch3_frame_pile_vs_likelihood_contrast(phase="intro", bar_t=0.0))
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_CONTRAST), endpoint=True):
        frames.append(ch3_frame_pile_vs_likelihood_contrast(phase="pile", bar_t=float(tv)))
    _ch3_strip_append_hold(frames, frames[-1], CH3_STRIP_N_CONTRAST_HOLD // 2)
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_CONTRAST), endpoint=True):
        frames.append(ch3_frame_pile_vs_likelihood_contrast(phase="likelihood", bar_t=float(tv)))
    _ch3_strip_append_hold(frames, frames[-1], CH3_STRIP_N_CONTRAST_HOLD // 2)
    for tv in np.linspace(0.0, 1.0, int(CH3_STRIP_N_CONTRAST), endpoint=True):
        frames.append(ch3_frame_pile_vs_likelihood_contrast(phase="both", bar_t=float(tv)))
    _ch3_strip_append_hold(frames, frames[-1])
    return frames


def ch3_export_script_strip_pile_likelihood_contrast():
    frames = ch3_build_frames_pile_vs_likelihood_contrast()
    ch3_export_strip_story("mistakes_strip_pile_likelihood_contrast", frames)



def ch3_export_script_strip_pile_start(with_noise=False):
    w = CH3_STRIP_W_START
    rows = ch3_build_student_label_rows(
        study_sep, exam_sep, y_sep, *w, fail_red=True,
    )
    frames = ch3_build_frames_strip_bar_pile(*w, rows, with_noise)
    ch3_export_strip_story("mistakes_strip_pile_start", frames)


def ch3_build_frames_mistakes_triptych_prob_story(with_noise=False):
    return ch3_build_frames_mistakes_strip_prob_story(with_noise)


def ch3_export_script_mistakes_triptych_prob_story(with_noise=False):
    ch3_export_script_mistakes_strip_prob_story(with_noise)


# Sequential export filenames (``ch3_00`` … ``ch3_58``; no gaps).
CH3_EXPORT_FILES = {
    "bridge": "ch3_00_bridge_dataset_knobs_into_duo_slot.mp4",
    "dataset_clean_plain_still": "ch3_01_dataset_clean_plain_still.mp4",
    "dataset_clean_st_el_zero_still": "ch3_02_dataset_clean_st_el_zero_still.mp4",
    "dataset_noise_plain_still": "ch3_03_dataset_noise_plain_still.mp4",
    "keras_gd_dataset_knobs": "ch3_04_keras_gd_dataset_knobs_steps.mp4",
    "bridge_keras_init_knobs_intro": "ch3_05_bridge_keras_init_knobs_intro.mp4",
    "mistakes_triptych_noise": "ch3_06_mistakes_triptych_noise.mp4",
    "sumprob_far_flat": "ch3_07_sumprob_far_flat.mp4",
    "dataset_clean_still": "ch3_08_dataset_clean_still.mp4",
    "dataset_noise_still": "ch3_09_dataset_noise_still.mp4",
    "goals_clean_still": "ch3_10_goals_clean_still.mp4",
    "mistakes_st_clean": "ch3_11_mistakes_st_clean.mp4",
    "negloglik_keras_weight3d_clean": "ch3_12_negloglik_keras_dataset_weight3d_clean.mp4",
    "mistakes_k1_flat_small": "ch3_13_mistakes_k1_flat_small.mp4",
    "mistakes_k1_staircase_wide": "ch3_14_mistakes_k1_staircase_wide.mp4",
    "mistakes_k1_staircase_descent_w12": "ch3_15_mistakes_k1_staircase_descent_w12.mp4",
    "mistakes_k1_staircase_descent_w10": "ch3_16_mistakes_k1_staircase_descent_w10.mp4",
    "mistakes_k1_staircase_descent_w08": "ch3_17_mistakes_k1_staircase_descent_w08.mp4",
    "mistakes_k1_staircase_descent_w06": "ch3_18_mistakes_k1_staircase_descent_w06.mp4",
    "mistakes_k1_staircase_stop": "ch3_19_mistakes_k1_staircase_stop.mp4",
    "mistakes_triptych_reveal_el_b": "ch3_20_mistakes_triptych_reveal_el_b.mp4",
    "mistakes_k1_staircase_descent_triptych": "ch3_21_mistakes_k1_staircase_descent_triptych.mp4",
    "mistakes_k1_post_21_arrows": "ch3_22_mistakes_k1_post_21_arrows.mp4",
    "mistakes_k1_post_22_w2_nudge": "ch3_23_mistakes_k1_post_22_w2_nudge.mp4",
    "mistakes_k1_post_23_small_nudge": "ch3_24_mistakes_k1_post_23_small_nudge.mp4",
    "mistakes_triptych_clean": "ch3_25_mistakes_triptych_clean.mp4",
    "mistakes_k1_tiny_triptych_coupled": "ch3_26_mistakes_k1_tiny_triptych_coupled.mp4",
    "sumprob_triptych_clean": "ch3_27_sumprob_triptych_clean.mp4",
    "sumprob_st_clean": "ch3_28_sumprob_st_clean.mp4",
    "sumprob_k1_small": "ch3_29_sumprob_k1_small.mp4",
    "likelihood_triptych_clean": "ch3_30_likelihood_triptych_clean.mp4",
    "likelihood_st_clean": "ch3_31_likelihood_st_clean.mp4",
    "negloglik_triptych_clean": "ch3_32_negloglik_triptych_clean.mp4",
    "negloglik_keras_weight3d_noise": "ch3_33_negloglik_keras_dataset_weight3d_noise.mp4",
    "negloglik_keras_weight3d_ball_clean": "ch3_34_negloglik_keras_dataset_weight3d_ball_clean.mp4",
    "negloglik_keras_weight3d_ball_noise": "ch3_35_negloglik_keras_dataset_weight3d_ball_noise.mp4",
    "keras_gd_dataset_knobs_colormap": "ch3_36_keras_gd_dataset_knobs_steps_colormap.mp4",
    "keras_bruteforce_knob_tries": "ch3_37_keras_bruteforce_knob_tries.mp4",
    "mistakes_strip_prob_story": "ch3_38_mistakes_strip_prob_story.mp4",
    "mistakes_strip_knob_to_mid": "ch3_39_mistakes_strip_knob_to_mid.mp4",
    "mistakes_strip_knob_to_amp": "ch3_40_mistakes_strip_knob_to_amp.mp4",
    "mistakes_strip_prob_labels": "ch3_41_mistakes_strip_prob_labels.mp4",
    "mistakes_strip_pile_amp": "ch3_42_mistakes_strip_pile_amp.mp4",
    "mistakes_strip_pile_start": "ch3_43_mistakes_strip_pile_start.mp4",
    "sumprob_k1_flat_small": "ch3_44_sumprob_k1_flat_small.mp4",
    "sumprob_k1_staircase_wide": "ch3_45_sumprob_k1_staircase_wide.mp4",
    "sumprob_k1_staircase_descent_w12": "ch3_46_sumprob_k1_staircase_descent_w12.mp4",
    "sumprob_k1_staircase_descent_w10": "ch3_47_sumprob_k1_staircase_descent_w10.mp4",
    "sumprob_k1_staircase_descent_w08": "ch3_48_sumprob_k1_staircase_descent_w08.mp4",
    "sumprob_k1_staircase_descent_w06": "ch3_49_sumprob_k1_staircase_descent_w06.mp4",
    "sumprob_k1_tight_staircase_wide": "ch3_50_sumprob_k1_tight_staircase_wide.mp4",
    "sumprob_k1_tight_staircase_descent_w12": "ch3_51_sumprob_k1_tight_staircase_descent_w12.mp4",
    "sumprob_k1_tight_staircase_descent_w10": "ch3_52_sumprob_k1_tight_staircase_descent_w10.mp4",
    "sumprob_k1_tight_staircase_descent_w08": "ch3_53_sumprob_k1_tight_staircase_descent_w08.mp4",
    "sumprob_k1_tight_staircase_descent_w06": "ch3_54_sumprob_k1_tight_staircase_descent_w06.mp4",
    "sumprob_k1_slope_triangles": "ch3_55_sumprob_k1_slope_triangles.mp4",
    "mistakes_strip_likelihood_amp": "ch3_56_mistakes_strip_likelihood_amp.mp4",
    "mistakes_strip_likelihood_start": "ch3_57_mistakes_strip_likelihood_start.mp4",
    "mistakes_strip_pile_likelihood_contrast": "ch3_58_mistakes_strip_pile_likelihood_contrast.mp4",
}
def ch3_export_filename(key, *, fallback=None):
    if key in CH3_EXPORT_FILES:
        return CH3_EXPORT_FILES[key]
    if fallback is not None:
        return fallback
    raise KeyError(f"Unknown CH3 export key: {key!r}")
# =============================================================================
# SCRIPT CH.3 — tuned weights & export helpers (one MP4 per script beat)
# =============================================================================
# Fixed $w_2$, $b$ for the mistake-count staircase story (separable roster).
CH3_SCRIPT_W2 = -1.0
CH3_SCRIPT_B = 1.0
# K1 staircase clips (ch3_08–10): start / sweep center $(w_1,w_2,b)=(1.5,-0.5,0)$.
CH3_SCRIPT_K1_W_ST = 1.4
CH3_SCRIPT_K1_W_EL = -0.5
CH3_SCRIPT_K1_B = 0.0
CH3_SCRIPT_K1_FLAT_SWEEP_DELTA = 0.05
CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA = 0.05
CH3_SCRIPT_K1_Y_PAD = 0.2
CH3_SCRIPT_K1_SHOW_TICK_LABELS = True
CH3_SCRIPT_K1_DESCENT_STEP = 0.2
CH3_SCRIPT_K1_DESCENT_END = 0.6
CH3_SCRIPT_K1_STOP_SWEEP_DELTA = 0.5
CH3_SCRIPT_W1_WIDE = (0.35, 2.35)
CH3_SCRIPT_W1_REF_FRAC = 0.25
CH3_SCRIPT_K1_AX_HALF = CH3_LOSS_SYM_DELTA
CH3_SCRIPT_K1_SWEEP_FRAC = 0.25
CH3_SCRIPT_K1_CACHE_N = max(96, _smooth_n(72))
CH3_SCRIPT_W1_TINY_DELTA = 0.05
CH3_SCRIPT_SUMPROB_FAR_W1 = -0.6
# Legacy aliases (goals still / flat plateau near 1/4 mark)
CH3_SCRIPT_W1_FLAT = 0.82
CH3_SCRIPT_W1_FLAT_DELTA = 0.06
CH3_SCRIPT_W1_STOP = 0.96
CH3_SCRIPT_N_FLAT = max(24, _smooth_n(18))
CH3_SCRIPT_N_K1_BIDIR = max(28, _smooth_n(22))
CH3_SCRIPT_N_K1_RESET = max(8, _smooth_n(6))
def ch3_script_w1_at_frac(frac):
    lo, hi = CH3_SCRIPT_W1_WIDE
    return float(lo) + float(frac) * (float(hi) - float(lo))
CH3_SCRIPT_W1_REF = ch3_script_w1_at_frac(CH3_SCRIPT_W1_REF_FRAC)
def ch3_k1_sweep_delta():
    return float(CH3_SCRIPT_K1_AX_HALF) * float(CH3_SCRIPT_K1_SWEEP_FRAC)
def ch3_script_k1_descent_centers():
    """$w_1$ targets stepping down from ``CH3_SCRIPT_K1_W_ST`` by ``CH3_SCRIPT_K1_DESCENT_STEP``."""
    w = float(CH3_SCRIPT_K1_W_ST)
    step = float(CH3_SCRIPT_K1_DESCENT_STEP)
    end = float(CH3_SCRIPT_K1_DESCENT_END)
    centers = []
    while w - step >= end - 1e-9:
        w -= step
        centers.append(round(w, 10))
    return centers
def ch3_export_key_k1_descent(w_center):
    tag = int(round(float(w_center) * 10))
    return f"mistakes_k1_staircase_descent_w{tag:02d}"
CH3_SCRIPT_N_WIDE = max(72, _smooth_n(56))
CH3_SCRIPT_N_DESCENT = max(40, _smooth_n(32))
CH3_SCRIPT_N_DESCENT_MOVE = max(36, _smooth_n(28))
CH3_SCRIPT_N_DESCENT_FILL = max(56, _smooth_n(44))
CH3_SCRIPT_N_LOCAL = max(28, _smooth_n(22))
CH3_SCRIPT_N_TINY = max(20, _smooth_n(16))
CH3_SCRIPT_N_HOLD = max(18, _smooth_n(12))
CH3_SCRIPT_N_STILL = max(12, _smooth_n(8))
CH3_SCRIPT_COUPLED_CURVE_N = 48
CH3_SCRIPT_MS = CH3_MP4_MS_PER_FRAME
def ch3_script_seq_lin(lo, hi, n):
    return np.linspace(float(lo), float(hi), int(max(2, n)), endpoint=True)
def ch3_script_weights(which, val, w_st, w_el, b):
    if which == "st":
        return float(val), float(w_el), float(b)
    if which == "el":
        return float(w_st), float(val), float(b)
    if which == "b":
        return float(w_st), float(w_el), float(val)
    raise ValueError(which)
def ch3_script_loss_curve(loss_fn, which, w_st, w_el, b, study, exam, y, lo, hi, n):
    seq = ch3_script_seq_lin(lo, hi, n)
    ys = np.array(
        [loss_fn(*ch3_script_weights(which, float(v), w_st, w_el, b), study, exam, y) for v in seq],
        dtype=float,
    )
    return seq, ys
def ch3_script_xlim_ylim(seq, ys):
    pad_y = 0.06 * max(1e-6, float(np.nanmax(ys) - np.nanmin(ys)))
    y_lo = float(np.nanmin(ys) - pad_y)
    y_hi = float(np.nanmax(ys) + pad_y)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or abs(y_hi - y_lo) < 1e-9:
        y_lo, y_hi = y_lo - 1.0, y_hi + 1.0
    span_x = float(np.max(seq) - np.min(seq))
    pad_x = max(0.06 * span_x, 0.08)
    x_lo = float(np.min(seq) - pad_x)
    x_hi = float(np.max(seq) + pad_x)
    return (x_lo, x_hi), (y_lo, y_hi)
def _ch3_xlim_centered(center, half_width):
    hw = float(half_width)
    c = float(center)
    return (c - hw, c + hw)
class Ch3K1LossCache:
    """Precomputed $w_1$ grid → loss value for fast k1 triptych frames."""
    __slots__ = ("w2", "b", "w_center", "ax_half", "seq", "ys", "y_lim", "x_lim")
    def __init__(self, w2, b, w_center, ax_half, study, exam, y, loss_fn, *, n=None):
        self.w2 = float(w2)
        self.b = float(b)
        self.w_center = float(w_center)
        self.ax_half = float(ax_half)
        n_pts = int(n) if n is not None else int(CH3_SCRIPT_K1_CACHE_N)
        w_lo = self.w_center - self.ax_half
        w_hi = self.w_center + self.ax_half
        self.seq = np.linspace(w_lo, w_hi, max(16, n_pts), endpoint=True)
        self.ys = np.array(
            [float(loss_fn(float(w), self.w2, self.b, study, exam, y)) for w in self.seq],
            dtype=float,
        )
        self.y_lim = _ch3_ylim_for_ys(self.ys)
        self.x_lim = _ch3_xlim_centered(self.w_center, self.ax_half)
    def max_flat_halfwidth(self):
        seq, ys = self.seq, self.ys
        c = self.w_center
        i0 = int(np.argmin(np.abs(seq - c)))
        m0 = float(ys[i0])
        il = i0
        while il > 0 and float(ys[il - 1]) == m0:
            il -= 1
        ir = i0
        while ir < len(ys) - 1 and float(ys[ir + 1]) == m0:
            ir += 1
        return max(0.0, min(c - float(seq[il]), float(seq[ir]) - c))
    def slice_right(self, w_hi):
        w_hi = float(w_hi)
        m = (self.seq >= self.w_center - 1e-12) & (self.seq <= w_hi + 1e-12)
        xs, ys = self.seq[m], self.ys[m]
        if w_hi >= self.w_center - 1e-12 and (
            xs.size == 0 or float(xs[0]) > self.w_center + 1e-9
        ):
            cx, cy = self.point_at(self.w_center)
            xs = np.concatenate(([cx], xs))
            ys = np.concatenate(([cy], ys))
        return xs, ys
    def slice_left(self, w_lo):
        w_lo = float(w_lo)
        m = (self.seq >= w_lo - 1e-12) & (self.seq <= self.w_center + 1e-12)
        xs, ys = self.seq[m], self.ys[m]
        if w_lo <= self.w_center + 1e-12 and (
            xs.size == 0 or float(xs[-1]) < self.w_center - 1e-9
        ):
            cx, cy = self.point_at(self.w_center)
            xs = np.concatenate((xs, [cx]))
            ys = np.concatenate((ys, [cy]))
        return xs, ys
    def slice_through(self, w_lo, w_hi):
        w_lo, w_hi = float(w_lo), float(w_hi)
        m = (self.seq >= w_lo - 1e-12) & (self.seq <= w_hi + 1e-12)
        return self.seq[m], self.ys[m]
    def point_at(self, w):
        w = float(w)
        return w, float(np.interp(w, self.seq, self.ys))
def ch3_k1_loss_cache(loss_key, w2, b, w_center, with_noise=False, *, ax_half=None, n=None):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    if ax_half is None:
        ax_half = float(CH3_SCRIPT_K1_AX_HALF)
    loss_fn = CH3_LOSS_SPECS[loss_key]["fn"]
    return Ch3K1LossCache(w2, b, w_center, ax_half, study, exam, y, loss_fn, n=n)


def ch3_mistakes_k1_cache(w2, b, w_center, with_noise=False, *, ax_half=None, n=None):
    return ch3_k1_loss_cache("mistakes", w2, b, w_center, with_noise, ax_half=ax_half, n=n)
def ch3_k1_flat_sweep_delta(cache):
    """Sweep span on the mistake-count plateau; fallback if center is not flat."""
    fh = max(float(cache.max_flat_halfwidth()), 0.0)
    d = 0.92 * fh
    floor = float(CH3_SCRIPT_W1_FLAT_DELTA)
    if d < floor:
        d = floor
    return d
def ch3_k1_wide_xlim(w_center, d_wide=None):
    w_center = float(w_center)
    if d_wide is None:
        d_wide = ch3_k1_sweep_delta()
    return _ch3_xlim_centered(w_center, float(d_wide))
def ch3_k1_ylim_on_ys(ys, pad=None):
    """Y limits from mistake counts: min − ``pad``, max + ``pad``."""
    pad = float(CH3_SCRIPT_K1_Y_PAD if pad is None else pad)
    ys = np.asarray(ys, dtype=float).ravel()
    if ys.size == 0:
        return (-pad, pad)
    return (float(np.nanmin(ys)) - pad, float(np.nanmax(ys)) + pad)
def ch3_k1_ylim_for_sweep(cache, w_lo, w_hi, pad=None):
    """Y limits for the full $w_1$ interval shown across a k1 clip."""
    _, ys = cache.slice_through(float(w_lo), float(w_hi))
    return ch3_k1_ylim_on_ys(ys, pad=pad)


def ch3_k1_wide_plot_limits(loss_key, w2, b, w_center, with_noise=False, *, sweep_delta=None):
    """Shared wide x window for k1 flat/wide clips (y limits chosen per clip)."""
    w_center = float(w_center)
    d_wide = float(ch3_k1_sweep_delta() if sweep_delta is None else sweep_delta)
    cache = ch3_k1_loss_cache(loss_key, w2, b, w_center, with_noise, ax_half=d_wide)
    x_lim = ch3_k1_wide_xlim(w_center, d_wide)
    return x_lim, cache, d_wide
def ch3_build_frames_k1_wide_extend_from_flat(
    loss_key,
    with_noise,
    w_center,
    w2,
    b,
    xs_flat,
    ys_flat,
    d_flat,
    d_wide,
    *,
    cache=None,
    x_lim_st=None,
    y_lim=None,
    show_colormap=None,
    n_extend=None,
    n_retrace=None,
    hold_end=True,
    show_tick_labels=None,
    grow=True,
    shrink=True,
    arrows_left=False,
    y_lim_fixed=False,
    start_with_flat=True,
):
    """Start with ch3_13 flat curve; extend right then left to ``±d_wide`` (fixed axes)."""
    spec = CH3_LOSS_SPECS[loss_key]
    show_cmap = bool(spec["colormap"]) if show_colormap is None else bool(show_colormap)
    w_center = float(w_center)
    w2, b = float(w2), float(b)
    d_flat, d_wide = float(d_flat), float(d_wide)
    if cache is None:
        cache = ch3_k1_loss_cache(loss_key, w2, b, w_center, with_noise, ax_half=d_wide)
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    n_ext = int(CH3_SCRIPT_N_WIDE if n_extend is None else n_extend)
    w_right = w_center + d_wide
    w_left = w_center - d_wide
    w_flat_lo = w_center - d_flat
    if show_tick_labels is None:
        show_tick_labels = CH3_SCRIPT_K1_SHOW_TICK_LABELS
    n_ret = int(CH3_SCRIPT_N_K1_RESET if n_retrace is None else n_retrace)
    if x_lim_st is None:
        x_lim_st = ch3_k1_wide_xlim(w_center, d_wide)
    if y_lim_fixed:
        if y_lim is None:
            y_lim_lo, y_lim_hi = ch3_k1_ylim_for_sweep(
                cache, w_center - d_wide, w_center + d_wide,
            )
        else:
            y_lim_lo, y_lim_hi = float(y_lim[0]), float(y_lim[1])
    else:
        y_lim_lo, y_lim_hi = (
            ch3_k1_ylim_on_ys(ys_flat) if y_lim is None else (float(y_lim[0]), float(y_lim[1]))
        )
    x_lim_el = _ch3_xlim_centered(w2, d_wide)
    x_lim_b = _ch3_xlim_centered(b, d_wide)
    xs_flat = np.asarray(xs_flat, dtype=float)
    ys_flat = np.asarray(ys_flat, dtype=float)
    frames = []

    def emit(w1, xs, ys, scale_st, *, marker_w=None, pin_rots_w1=None, arrows_left=False):
        nonlocal y_lim_lo, y_lim_hi
        w_rot = float(w1 if pin_rots_w1 is None else pin_rots_w1)
        _hl, _sz = ch3_k1_data_panel_extras(loss_key, w_rot, w2, b, show_cmap)
        mk = None
        marker_y = None
        if marker_w is not None:
            pw, py = cache.point_at(marker_w)
            mk = (float(pw), float(py))
            marker_y = float(py)
        if y_lim_fixed:
            y_lim_use = (y_lim_lo, y_lim_hi)
        else:
            y_lo, y_hi = _ch3_ylim_expand_if_needed(
                (y_lim_lo, y_lim_hi), ys, marker_y=marker_y, pad=CH3_SCRIPT_K1_Y_PAD,
            )
            y_lim_lo = min(y_lim_lo, y_lo)
            y_lim_hi = max(y_lim_hi, y_hi)
            y_lim_use = (y_lim_lo, y_lim_hi)
        frames.append(
            ch3_triptych_frame_panels(
                float(w1),
                w2,
                b,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                highlight_mistakes_flag=_hl,
                sigma_Z=_sz,
                xs_st=np.asarray(xs, dtype=float).tolist(),
                ys_st=np.asarray(ys, dtype=float).tolist(),
                xs_el=[],
                ys_el=[],
                xs_b=[],
                ys_b=[],
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lim=y_lim_use,
                knob_rots=ch3_k1_knob_rots_at(w_rot, w2, b),
                knob_scales=ch3_knob_scales_emphasize("st", scale_st),
                panel_visible=(True, False, False),
                panel_alpha=(1.0, 1.0, 1.0),
                emphasize_knob="st",
                marker_st=mk,
                show_tick_labels=show_tick_labels,
                arrows_left=arrows_left,
            )
        )

    w_flat_hi = w_center + d_flat
    w_sweep_lo, w_sweep_hi = w_center, w_right
    if grow:
        for sc in ch3_k1_active_scale_grow_sequence():
            if start_with_flat:
                emit(
                    w_center, xs_flat, ys_flat, sc,
                    marker_w=w_center, pin_rots_w1=w_center,
                )
            else:
                emit(
                    w_center, [], [], sc,
                    marker_w=w_center, pin_rots_w1=w_center,
                )
    if n_ext <= 1:
        wr_seq = [w_right]
    else:
        wr_seq = np.linspace(w_sweep_lo, w_sweep_hi, n_ext, endpoint=True)
    for wv in wr_seq:
        wv = float(wv)
        if start_with_flat and wv <= w_flat_hi + 1e-9:
            xs, ys = xs_flat, ys_flat
        elif wv <= w_center + 1e-9:
            xs, ys = [], []
        elif wv <= w_flat_hi + 1e-9:
            xs, ys = cache.slice_right(wv)
        else:
            xs, ys = cache.slice_through(w_flat_lo, wv)
        emit(wv, xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=wv)
    # Retrace right → center (curve shrinks with marker; avoids snap into left sweep).
    n_back = max(2, int(n_ret))
    if n_back <= 1:
        wr_back = [w_center]
    else:
        wr_back = np.linspace(w_right, w_center, n_back, endpoint=True)[1:]
    for wv in wr_back:
        xs, ys = cache.slice_right(float(wv))
        emit(float(wv), xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv))
    if n_ext <= 1:
        wl_seq = [w_left]
    else:
        wl_seq = np.linspace(w_center, w_left, n_ext, endpoint=True)[1:]
    for wv in wl_seq:
        xs, ys = cache.slice_through(float(wv), w_right)
        emit(float(wv), xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv), arrows_left=arrows_left)
    w_end = w_center
    if n_back <= 1:
        wc_back = [w_center]
    else:
        wc_back = np.linspace(w_left, w_center, n_back, endpoint=True)[1:]
    for wv in wc_back:
        xs, ys = cache.slice_through(w_left, float(wv))
        emit(float(wv), xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv), arrows_left=arrows_left)
    if shrink:
        for sc in ch3_k1_active_scale_shrink_sequence():
            emit(
                w_end, xs_flat, ys_flat, sc,
                marker_w=w_end, pin_rots_w1=w_end, arrows_left=arrows_left,
            )
    if hold_end and frames:
        last = frames[-1]
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(last.copy())
    return frames
def ch3_build_frames_k1_descent_shift_fill(
    loss_key,
    with_noise,
    w_from,
    w_to,
    w2,
    b,
    *,
    sweep_delta=None,
    show_colormap=None,
    n_shift=None,
    n_fill_left=None,
    grow_start=True,
    shrink_end=True,
    hold_end=True,
    show_tick_labels=None,
    n_retrace=None,
    arrows_left=False,
    y_lim=None,
    y_lim_fixed=False,
):
    """Full curve at ``w_from``; slide window to ``w_to`` (right clips); sweep left to fill."""
    spec = CH3_LOSS_SPECS[loss_key]
    show_cmap = bool(spec["colormap"]) if show_colormap is None else bool(show_colormap)
    if show_tick_labels is None:
        show_tick_labels = CH3_SCRIPT_K1_SHOW_TICK_LABELS
    n_ret = int(CH3_SCRIPT_N_K1_RESET if n_retrace is None else n_retrace)
    w_from, w_to = float(w_from), float(w_to)
    w2, b = float(w2), float(b)
    d = float(ch3_k1_sweep_delta() if sweep_delta is None else sweep_delta)
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    n_shift = int(CH3_SCRIPT_N_DESCENT_MOVE if n_shift is None else n_shift)
    n_fill = int(CH3_SCRIPT_N_DESCENT_FILL if n_fill_left is None else n_fill_left)
    cache_from = ch3_k1_loss_cache(loss_key, w2, b, w_from, with_noise, ax_half=d)
    cache_to = ch3_k1_loss_cache(loss_key, w2, b, w_to, with_noise, ax_half=d)
    w_band_hi = w_from + d
    w_band_lo = w_from - d
    xs_seed, ys_seed = cache_from.slice_through(w_band_lo, w_band_hi)
    x_lim_st_fixed = ch3_k1_wide_xlim(float(CH3_SCRIPT_K1_W_ST), d)
    x_lim_el = _ch3_xlim_centered(w2, d)
    x_lim_b = _ch3_xlim_centered(b, d)
    frames = []
    w_fill_hi = w_to + d
    if y_lim_fixed:
        if y_lim is None:
            _, ys_from = cache_from.slice_through(w_band_lo, w_band_hi)
            _, ys_to = cache_to.slice_through(w_to - d, w_fill_hi)
            y_lim_lo, y_lim_hi = ch3_k1_ylim_on_ys(np.concatenate([ys_from, ys_to]))
        else:
            y_lim_lo, y_lim_hi = float(y_lim[0]), float(y_lim[1])
    else:
        y_lim_lo, y_lim_hi = (
            ch3_k1_ylim_on_ys(ys_seed) if y_lim is None else (float(y_lim[0]), float(y_lim[1]))
        )

    def emit(w_knob, xs, ys, scale_st, *, marker_w=None, x_lim_st=None, pin_rots_w1=None, arrows_left=False):
        nonlocal y_lim_lo, y_lim_hi
        w_rot = float(w_knob if pin_rots_w1 is None else pin_rots_w1)
        _hl, _sz = ch3_k1_data_panel_extras(loss_key, w_rot, w2, b, show_cmap)
        mk = None
        marker_y = None
        if marker_w is not None:
            pw, py = cache_to.point_at(marker_w)
            mk = (float(pw), float(py))
            marker_y = float(py)
        if y_lim_fixed:
            y_lim_use = (y_lim_lo, y_lim_hi)
        else:
            y_lo, y_hi = _ch3_ylim_expand_if_needed(
                (y_lim_lo, y_lim_hi), ys, marker_y=marker_y, pad=CH3_SCRIPT_K1_Y_PAD,
            )
            y_lim_lo = min(y_lim_lo, y_lo)
            y_lim_hi = max(y_lim_hi, y_hi)
            y_lim_use = (y_lim_lo, y_lim_hi)
        if x_lim_st is None:
            x_lim_st = x_lim_st_fixed
        frames.append(
            ch3_triptych_frame_panels(
                float(w_knob),
                w2,
                b,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                highlight_mistakes_flag=_hl,
                sigma_Z=_sz,
                xs_st=np.asarray(xs, dtype=float).tolist(),
                ys_st=np.asarray(ys, dtype=float).tolist(),
                xs_el=[],
                ys_el=[],
                xs_b=[],
                ys_b=[],
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lim=y_lim_use,
                knob_rots=ch3_k1_knob_rots_at(w_rot, w2, b),
                knob_scales=ch3_knob_scales_emphasize("st", scale_st),
                panel_visible=(True, False, False),
                panel_alpha=(1.0, 1.0, 1.0),
                emphasize_knob="st",
                marker_st=mk,
                show_tick_labels=show_tick_labels,
                arrows_left=arrows_left,
            )
        )

    if grow_start:
        for sc in ch3_k1_active_scale_grow_sequence():
            emit(
                w_from, xs_seed, ys_seed, sc,
                marker_w=w_from, pin_rots_w1=w_from,
            )

    if n_shift <= 1:
        w_shift = [w_to]
    else:
        w_shift = np.linspace(w_from, w_to, n_shift, endpoint=True)
    for w_knob in w_shift:
        emit(
            float(w_knob), xs_seed, ys_seed, CH3_K1_ACTIVE_SCALE_FULL,
            marker_w=float(w_knob),
        )

    # Shift done: lock x-axis at w_to, then sweep knob left and grow the curve into view.
    emit(
        w_to, xs_seed, ys_seed, CH3_K1_ACTIVE_SCALE_FULL,
        marker_w=w_to, x_lim_st=x_lim_st_fixed,
    )

    if n_fill <= 1:
        wl_seq = [w_to - d]
    else:
        wl_seq = np.linspace(w_to, w_to - d, n_fill, endpoint=True)[1:]
    for w_knob in wl_seq:
        xs_vis, ys_vis = cache_to.slice_through(float(w_knob), w_fill_hi)
        emit(
            float(w_knob), xs_vis, ys_vis, CH3_K1_ACTIVE_SCALE_FULL,
            marker_w=float(w_knob), x_lim_st=x_lim_st_fixed, arrows_left=arrows_left,
        )
    xs_final, ys_final = cache_to.slice_through(w_to - d, w_fill_hi)
    w_knob_end = float(wl_seq[-1]) if len(wl_seq) else w_to - d
    if n_ret <= 1:
        w_back = []
    else:
        w_back = np.linspace(w_knob_end, w_to, n_ret, endpoint=True)[1:]
    for w_knob in w_back:
        emit(
            float(w_knob), xs_final, ys_final, CH3_K1_ACTIVE_SCALE_FULL,
            marker_w=float(w_knob), x_lim_st=x_lim_st_fixed, arrows_left=arrows_left,
        )
    w_shrink = w_to
    emit(
        w_shrink, xs_final, ys_final, CH3_K1_ACTIVE_SCALE_FULL,
        marker_w=w_shrink, x_lim_st=x_lim_st_fixed, arrows_left=arrows_left,
        )
    if shrink_end:
        for sc in ch3_k1_active_scale_shrink_sequence():
            emit(
                w_shrink, xs_final, ys_final, sc,
                marker_w=w_shrink, x_lim_st=x_lim_st_fixed, pin_rots_w1=w_shrink, arrows_left=arrows_left,
            )
    if hold_end and frames:
        last = frames[-1]
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(last.copy())
    return frames
def ch3_build_frames_k1_center_bidirectional(
    loss_key,
    with_noise,
    w_center,
    w2,
    b,
    *,
    sweep_delta,
    show_colormap=None,
    n_bidir=None,
    n_reset=None,
    ax_half=None,
    cache=None,
    hold_end=True,
    grow=True,
    shrink=True,
    x_lim_st_override=None,
    y_lim_override=None,
    show_tick_labels=None,
    y_lim_fixed=False,
):
    """Knob 1: extend right, retrace to center, extend left (full curve), retrace to center."""
    spec = CH3_LOSS_SPECS[loss_key]
    show_cmap = bool(spec["colormap"]) if show_colormap is None else bool(show_colormap)
    w_center = float(w_center)
    w2, b = float(w2), float(b)
    sweep_delta = float(sweep_delta)
    sweep_half = float(sweep_delta)
    if ax_half is None:
        ax_half = sweep_half
    if cache is None:
        cache = ch3_k1_loss_cache(loss_key, w2, b, w_center, with_noise, ax_half=sweep_half)
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    n_bidir = int(CH3_SCRIPT_N_K1_BIDIR if n_bidir is None else n_bidir)
    n_reset = int(CH3_SCRIPT_N_K1_RESET if n_reset is None else n_reset)
    w_right = w_center + sweep_delta
    w_left = w_center - sweep_delta
    x_lim_st = (
        x_lim_st_override
        if x_lim_st_override is not None
        else _ch3_xlim_centered(w_center, sweep_half)
    )
    x_lim_el = _ch3_xlim_centered(w2, ax_half if x_lim_st_override is None else float(x_lim_st[1] - x_lim_st[0]) * 0.5)
    x_lim_b = _ch3_xlim_centered(b, ax_half if x_lim_st_override is None else float(x_lim_st[1] - x_lim_st[0]) * 0.5)
    if show_tick_labels is None:
        show_tick_labels = CH3_SCRIPT_K1_SHOW_TICK_LABELS
    xs_full, ys_full = cache.slice_through(w_left, w_right)
    if y_lim_fixed:
        if y_lim_override is not None:
            y_lim_lo, y_lim_hi = float(y_lim_override[0]), float(y_lim_override[1])
        else:
            y_lim_lo, y_lim_hi = ch3_k1_ylim_on_ys(ys_full)
    elif y_lim_override is not None:
        y_lim_lo, y_lim_hi = float(y_lim_override[0]), float(y_lim_override[1])
    else:
        y_lim_lo, y_lim_hi = ch3_k1_ylim_on_ys(ys_full)
    frames = []
    def emit(w1, xs, ys, scale_st, *, marker_w=None, pin_rots_w1=None):
        nonlocal y_lim_lo, y_lim_hi
        w_rot = float(w1 if pin_rots_w1 is None else pin_rots_w1)
        _hl, _sz = ch3_k1_data_panel_extras(loss_key, w_rot, w2, b, show_cmap)
        mk = None
        marker_y = None
        if marker_w is not None:
            pw, py = cache.point_at(marker_w)
            mk = (float(pw), float(py))
            marker_y = float(py)
        if y_lim_fixed:
            y_lim = (y_lim_lo, y_lim_hi)
        else:
            y_lo, y_hi = _ch3_ylim_expand_if_needed(
                (y_lim_lo, y_lim_hi), ys, marker_y=marker_y, pad=CH3_SCRIPT_K1_Y_PAD,
            )
            y_lim_lo = min(y_lim_lo, y_lo)
            y_lim_hi = max(y_lim_hi, y_hi)
            y_lim = (y_lim_lo, y_lim_hi)
        frames.append(
            ch3_triptych_frame_panels(
                float(w1),
                w2,
                b,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                highlight_mistakes_flag=_hl,
                sigma_Z=_sz,
                xs_st=np.asarray(xs, dtype=float).tolist(),
                ys_st=np.asarray(ys, dtype=float).tolist(),
                xs_el=[],
                ys_el=[],
                xs_b=[],
                ys_b=[],
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lim=y_lim,
                knob_rots=ch3_k1_knob_rots_at(w_rot, w2, b),
                knob_scales=ch3_knob_scales_emphasize("st", scale_st),
                panel_visible=(True, False, False),
                panel_alpha=(1.0, 1.0, 1.0),
                emphasize_knob="st",
                marker_st=mk,
                show_tick_labels=show_tick_labels,
            )
        )
    if grow:
        for sc in ch3_k1_active_scale_grow_sequence():
            emit(
                w_center, [], [], sc,
                marker_w=w_center, pin_rots_w1=w_center,
            )
    if n_bidir <= 1:
        wr_seq = [w_right]
    else:
        wr_seq = np.linspace(w_center, w_right, n_bidir, endpoint=True)
    for wv in wr_seq:
        xs, ys = cache.slice_right(wv)
        emit(float(wv), xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=wv)
    if n_reset <= 1:
        wr_back = [w_center]
    else:
        wr_back = np.linspace(w_right, w_center, n_reset, endpoint=True)
    xs_right, ys_right = cache.slice_right(w_right)
    for wv in wr_back:
        emit(float(wv), xs_right, ys_right, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv))
    if n_bidir <= 1:
        wl_seq = [w_left]
    else:
        wl_seq = np.linspace(w_center, w_left, n_bidir, endpoint=True)
    for wv in wl_seq:
        xs, ys = cache.slice_through(float(wv), w_right)
        emit(float(wv), xs, ys, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv))
    xs_full, ys_full = cache.slice_through(w_left, w_right)
    if n_reset <= 1:
        wl_back = [w_center]
    else:
        wl_back = np.linspace(w_left, w_center, n_reset, endpoint=True)
    for wv in wl_back:
        emit(float(wv), xs_full, ys_full, CH3_K1_ACTIVE_SCALE_FULL, marker_w=float(wv))
    if shrink:
        for sc in ch3_k1_active_scale_shrink_sequence():
            emit(
                w_center, xs_full, ys_full, sc,
                marker_w=w_center, pin_rots_w1=w_center,
            )
    if hold_end and frames:
        last = frames[-1]
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(last.copy())
    return frames
def ch3_build_frames_duo_path(
    loss_key,
    which,
    with_noise,
    w_st,
    w_el,
    b,
    path_vals,
    *,
    show_colormap=None,
    grow=True,
    shrink=True,
):
    """Duo layout: dataset + knobs + one loss panel; sweep ``which`` along ``path_vals``."""
    spec = CH3_LOSS_SPECS[loss_key]
    loss_fn = spec["fn"]
    show_cmap = bool(spec["colormap"]) if show_colormap is None else bool(show_colormap)
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    path = np.asarray(path_vals, dtype=float).ravel()
    if path.size < 2:
        path = np.array([float(path.flat[0]), float(path.flat[0]) + 1e-6], dtype=float)
    seq = path
    triples = [ch3_script_weights(which, float(v), w_st, w_el, b) for v in seq]
    losses = np.array([loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples], dtype=float)
    x_lim, y_lim = ch3_script_xlim_ylim(seq, losses)
    lo_rot, hi_rot = float(np.min(seq)), float(np.max(seq))
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    frames = []
    def emit(ws, we, bb, n_hist, rot_strip_deg, strip_scale):
        xs = seq[:n_hist].tolist()
        ys = losses[:n_hist].tolist()
        frames.append(
            ch3_frame(
                ws,
                we,
                bb,
                which,
                study,
                exam,
                y,
                spec,
                xs,
                ys,
                knob_rgbs,
                canvas_sides,
                show_colormap=show_cmap,
                rot_strip_deg=rot_strip_deg,
                strip_scale=strip_scale,
                x_lim_loss=x_lim,
                y_lim_loss=y_lim,
            )
        )
    ws0, we0, bb0 = triples[0]
    if grow:
        ug = [1.0] if CH3_N_GROW <= 1 else [float(j) / float(CH3_N_GROW - 1) for j in range(CH3_N_GROW)]
        for u in ug:
            sc = 1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
            emit(ws0, we0, bb0, 1, 0.0, sc)
    for i, (ws, we, bb) in enumerate(triples):
        val = ch3_active_value(which, ws, we, bb)
        rot = ch3_knob_rots_for_weights(*ch3_script_weights(which, val, w_st, w_el, b))[{"st": 0, "el": 1, "b": 2}[which]]
        emit(ws, we, bb, i + 1, rot, CH3_KNOB_ACTIVE_SCALE)
    if shrink:
        ws1, we1, bb1 = triples[-1]
        val_end = ch3_active_value(which, ws1, we1, bb1)
        rot_end = ch3_knob_rots_for_weights(*triples[-1])[{"st": 0, "el": 1, "b": 2}[which]]
        us = [1.0] if CH3_N_SHRINK <= 1 else [float(j) / float(CH3_N_SHRINK - 1) for j in range(CH3_N_SHRINK)]
        for u in us:
            sc = CH3_KNOB_ACTIVE_SCALE - (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
            emit(ws1, we1, bb1, len(seq), rot_end, sc)
    return frames
def ch3_build_frames_duo_segments(loss_key, which, with_noise, w_st, w_el, b, segments, **kwargs):
    """Concatenate several linear sweeps (staircase story) into one clip."""
    frames = []
    for lo, hi, n in segments:
        seg_frames = ch3_build_frames_triptych_k1_path(
            loss_key,
            with_noise,
            w_st,
            w_el,
            b,
            ch3_script_seq_lin(lo, hi, n),
            grow=(len(frames) == 0),
            shrink=False,
            show_colormap=kwargs.get("show_colormap", None),
        )
        if frames and seg_frames:
            seg_frames = seg_frames[1:]
        frames.extend(seg_frames)
    if frames:
        last = frames[-1]
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(last.copy())
    return frames
def ch3_triptych_frame_coupled(
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    spec,
    *,
    show_colormap,
    active,
    seq_active,
    n_active,
    curve_delta,
    curve_n,
    knob_rots,
    knob_scales,
    sigma_Z=None,
):
    """Three loss panels: grow sweep on ``active``; other two show full 1D slices at current weights."""
    fig, ax_data, axes_loss, axes_k = ch3_figure_duo_triptych()
    loss_fn = spec["fn"]
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, active)
    ch3_draw_left_panel(
        ax_data,
        w_st,
        w_el,
        b,
        study,
        exam,
        y,
        leg,
        show_colormap=show_colormap,
        highlight_mistakes_flag=True,
        sigma_Z=sigma_Z,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig,
        axes_k,
        w_st,
        w_el,
        b,
        active,
        knob_rgbs,
        canvas_sides,
        rot_strip_deg=0.0,
        strip_scale=1.0,
        knob_rots=knob_rots,
        knob_scales=knob_scales,
        ax_data=ax_data,
    )
    d = float(curve_delta)
    seq_st, ys_st = ch3_script_loss_curve(
        loss_fn, "st", w_st, w_el, b, study, exam, y,
        ch3_active_value("st", w_st, w_el, b) - d, ch3_active_value("st", w_st, w_el, b) + d, curve_n,
    )
    seq_el, ys_el = ch3_script_loss_curve(
        loss_fn, "el", w_st, w_el, b, study, exam, y,
        ch3_active_value("el", w_st, w_el, b) - d, ch3_active_value("el", w_st, w_el, b) + d, curve_n,
    )
    seq_b, ys_b = ch3_script_loss_curve(
        loss_fn, "b", w_st, w_el, b, study, exam, y,
        ch3_active_value("b", w_st, w_el, b) - d, ch3_active_value("b", w_st, w_el, b) + d, curve_n,
    )
    n_a = max(1, int(n_active))
    if active == "st":
        seq_st = np.asarray(seq_active, dtype=float)
        ys_st = np.array(
            [loss_fn(float(v), w_el, b, study, exam, y) for v in seq_st[:n_a]],
            dtype=float,
        )
    elif active == "el":
        seq_el = np.asarray(seq_active, dtype=float)
        ys_el = np.array(
            [loss_fn(w_st, float(v), b, study, exam, y) for v in seq_el[:n_a]],
            dtype=float,
        )
    else:
        seq_b = np.asarray(seq_active, dtype=float)
        ys_b = np.array(
            [loss_fn(w_st, w_el, float(v), study, exam, y) for v in seq_b[:n_a]],
            dtype=float,
        )
    all_ys = np.concatenate([ys_st, ys_el, ys_b])
    pad_y = 0.06 * max(1e-6, float(np.nanmax(all_ys) - np.nanmin(all_ys)))
    y_lo = float(np.nanmin(all_ys) - pad_y)
    y_hi = float(np.nanmax(all_ys) + pad_y)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or abs(y_hi - y_lo) < 1e-9:
        y_lo, y_hi = y_lo - 1.0, y_hi + 1.0
    d_pad = float(curve_delta)
    x_lim_st = _ch3_xlim_centered(ch3_active_value("st", w_st, w_el, b), d_pad)
    x_lim_el = _ch3_xlim_centered(ch3_active_value("el", w_st, w_el, b), d_pad)
    x_lim_b = _ch3_xlim_centered(ch3_active_value("b", w_st, w_el, b), d_pad)
    ax_st_l, ax_el_l, ax_b_l = axes_loss
    y_lim = (y_lo, y_hi)
    vis = (active == "st", active == "el", active == "b")
    panels = (
        (ax_st_l, seq_st[:n_active].tolist(), ys_st[:n_active].tolist(), x_lim_st, vis[0]),
        (ax_el_l, seq_el[:n_active].tolist(), ys_el[:n_active].tolist(), x_lim_el, vis[1]),
        (ax_b_l, seq_b[:n_active].tolist(), ys_b[:n_active].tolist(), x_lim_b, vis[2]),
    )
    for ax_p, xs, ys, xlim_p, show in panels:
        if show and len(xs) > 0:
            ch3_draw_loss_panel(
                ax_p, xs, ys, spec, xlim_p, y_lim, title_override="", show_axis_labels=False
            )
        else:
            ch3_clear_loss_panel(ax_p, xlim_p, y_lim)
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_build_frames_triptych_coupled(
    loss_key,
    with_noise,
    active,
    w_st,
    w_el,
    b,
    path_vals,
    *,
    curve_delta=None,
    curve_n=None,
):
    spec = CH3_LOSS_SPECS[loss_key]
    show_cmap = bool(spec["colormap"])
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    if curve_delta is None:
        curve_delta = float(CH3_LOSS_SYM_DELTA)
    if curve_n is None:
        curve_n = int(CH3_SCRIPT_COUPLED_CURVE_N)
    path = np.asarray(path_vals, dtype=float).ravel()
    seq_active = path
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    lo_r = float(np.min(seq_active))
    hi_r = float(np.max(seq_active))
    frames = []
    slot = {"st": 0, "el": 1, "b": 2}[active]
    def rots_scales_for(ws, we, bb, scale_active):
        rots = list(ch3_knob_rots_for_weights(ws, we, bb))
        sc = [1.0, 1.0, 1.0]
        sc[slot] = float(scale_active)
        return rots, sc
    ws0, we0, bb0 = ch3_script_weights(active, float(seq_active[0]), w_st, w_el, b)
    ug = [1.0] if CH3_N_GROW <= 1 else [float(j) / float(CH3_N_GROW - 1) for j in range(CH3_N_GROW)]
    for u in ug:
        sc_a = 1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u)
        kr, ks = rots_scales_for(ws0, we0, bb0, sc_a)
        frames.append(
            ch3_triptych_frame_coupled(
                ws0, we0, bb0, study, exam, y, spec,
                show_colormap=show_cmap, active=active, seq_active=seq_active, n_active=1,
                curve_delta=curve_delta, curve_n=curve_n, knob_rots=kr, knob_scales=ks,
            )
        )
    for i in range(len(seq_active)):
        ws, we, bb = ch3_script_weights(active, float(seq_active[i]), w_st, w_el, b)
        kr, ks = rots_scales_for(ws, we, bb, CH3_KNOB_ACTIVE_SCALE)
        frames.append(
            ch3_triptych_frame_coupled(
                ws, we, bb, study, exam, y, spec,
                show_colormap=show_cmap, active=active, seq_active=seq_active, n_active=i + 1,
                curve_delta=curve_delta, curve_n=curve_n, knob_rots=kr, knob_scales=ks,
            )
        )
    last = frames[-1]
    for _ in range(CH3_SCRIPT_N_HOLD):
        frames.append(last.copy())
    return frames
def ch3_build_frames_still(loss_key, with_noise, w_st, w_el, b, which="st", *, show_colormap=None):
    spec = CH3_LOSS_SPECS[loss_key]
    ws, we, bb = float(w_st), float(w_el), float(b)
    c = ch3_active_value(which, ws, we, bb)
    seq = np.array([c, c + 1e-6], dtype=float)
    return ch3_build_frames_duo_path(
        loss_key, which, with_noise, ws, we, bb, seq,
        show_colormap=show_colormap, grow=False, shrink=False,
    )[:1]
def ch3_export_script_mp4(fn, frames, *, ms=None):
    save_mp4(frames, fn, duration=ms if ms is not None else CH3_SCRIPT_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
CH3_SCRIPT_N_REVEAL = max(24, _smooth_n(18))
def _ch3_xlim_for_seq(seq):
    span = float(np.max(seq) - np.min(seq))
    pad_x = max(0.06 * span, 0.08)
    return float(np.min(seq) - pad_x), float(np.max(seq) + pad_x)
def _ch3_ylim_for_ys(*ys_arrays):
    ys = np.concatenate([np.asarray(a, dtype=float).ravel() for a in ys_arrays if len(a)])
    if ys.size == 0:
        return (0.0, 1.0)
    pad_y = 0.06 * max(1e-6, float(np.nanmax(ys) - np.nanmin(ys)))
    y_lo = float(np.nanmin(ys) - pad_y)
    y_hi = float(np.nanmax(ys) + pad_y)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or abs(y_hi - y_lo) < 1e-9:
        y_lo, y_hi = y_lo - 1.0, y_hi + 1.0
    return (y_lo, y_hi)
def _ch3_ylim_expand_if_needed(y_lim, *ys_arrays, marker_y=None, pad=None):
    """Extend ``y_lim`` outward only — never shrink (K1: fixed ``pad`` below/above extrema)."""
    y_lo, y_hi = float(y_lim[0]), float(y_lim[1])
    if pad is None:
        pad_use = 0.06 * max(1e-6, y_hi - y_lo)
    else:
        pad_use = float(pad)
    chunks = []
    for arr in ys_arrays:
        a = np.asarray(arr, dtype=float).ravel()
        if a.size:
            chunks.append(a)
    if marker_y is not None and np.isfinite(marker_y):
        chunks.append(np.asarray([float(marker_y)], dtype=float))
    if not chunks:
        return (y_lo, y_hi)
    data = np.concatenate(chunks)
    data_lo = float(np.nanmin(data))
    data_hi = float(np.nanmax(data))
    new_lo, new_hi = y_lo, y_hi
    if data_lo < y_lo:
        new_lo = data_lo - pad_use
    if data_hi > y_hi:
        new_hi = data_hi + pad_use
    return (new_lo, new_hi)
def ch3_k1_data_panel_extras(loss_key, w_st, w_el, b, show_colormap):
    # Colormap Z grid; disable mistake rings for sumprob clips.
    hl = str(loss_key) != "sumprob"
    sz = None
    if show_colormap:
        stg, elg = ST_KNOB, EL_KNOB
        sz = sigmoid(logits_plane(float(w_st), float(w_el), float(b), stg, elg))
    return hl, sz


def ch3_triptych_frame_panels(
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    spec,
    *,
    show_colormap,
    xs_st,
    ys_st,
    xs_el,
    ys_el,
    xs_b,
    ys_b,
    x_lim_st,
    x_lim_el,
    x_lim_b,
    y_lim,
    knob_rots,
    knob_scales,
    panel_visible=(True, True, True),
    panel_alpha=(1.0, 1.0, 1.0),
    marker_st=None,
    arrows=None,
    sigma_Z=None,
    emphasize_knob="st",
    show_tick_labels=False,
    highlight_mistakes_flag=True,
    arrows_left=False,
    slope_triangle=None,
):
    """Triptych layout; each right panel can be hidden (invisible) or faded via ``panel_alpha``."""
    fig, ax_data, axes_loss, axes_k = ch3_figure_duo_triptych()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, emphasize_knob)
    ch3_draw_left_panel(
        ax_data,
        w_st,
        w_el,
        b,
        study,
        exam,
        y,
        leg,
        show_colormap=show_colormap,
        highlight_mistakes_flag=highlight_mistakes_flag,
        sigma_Z=sigma_Z,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig,
        axes_k,
        w_st,
        w_el,
        b,
        emphasize_knob,
        knob_rgbs,
        canvas_sides,
        rot_strip_deg=0.0,
        strip_scale=1.0,
        knob_rots=knob_rots,
        knob_scales=knob_scales,
        ax_data=ax_data,
    )
    ax_st_l, ax_el_l, ax_b_l = axes_loss
    panels = (
        (ax_st_l, xs_st, ys_st, x_lim_st, panel_visible[0], panel_alpha[0]),
        (ax_el_l, xs_el, ys_el, x_lim_el, panel_visible[1], panel_alpha[1]),
        (ax_b_l, xs_b, ys_b, x_lim_b, panel_visible[2], panel_alpha[2]),
    )
    for j, (ax_p, xs, ys, xlim_p, vis, pa) in enumerate(panels):
        mk = marker_st if j == 0 else None
        if vis and pa > 1e-6 and (len(xs) > 0 or mk is not None):
            ch3_draw_loss_panel(
                ax_p,
                xs,
                ys,
                spec,
                xlim_p,
                y_lim,
                title_override="",
                show_axis_labels=False,
                show_tick_labels=show_tick_labels and j == 0,
                curve_alpha=pa,
                marker_xy=mk,
            )
        else:
            ch3_clear_loss_panel(ax_p, xlim_p, y_lim)
    if arrows is not None:
        _ch3_triptych_draw_arrows((ax_st_l, ax_el_l, ax_b_l), arrows)
    if slope_triangle is not None:
        ch3_draw_slope_triangle(ax_st_l, zorder=8, **slope_triangle)
    if arrows_left:
        _ch3_triptych_draw_left_arrows((ax_st_l, ax_el_l, ax_b_l))
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_build_frames_triptych_k1_path(
    loss_key,
    with_noise,
    w_st,
    w_el,
    b,
    path_vals,
    *,
    show_colormap=None,
    grow=True,
    shrink=True,
):
    """Knob 1 sweep with three-panel layout; w2 and b panels stay invisible."""
    spec = CH3_LOSS_SPECS[loss_key]
    loss_fn = spec["fn"]
    show_cmap = bool(spec["colormap"]) if show_colormap is None else bool(show_colormap)
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    path = np.asarray(path_vals, dtype=float).ravel()
    if path.size < 2:
        path = np.array([float(path.flat[0]), float(path.flat[0]) + 1e-6], dtype=float)
    seq = path
    triples = [ch3_script_weights("st", float(v), w_st, w_el, b) for v in seq]
    losses = np.array([loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples], dtype=float)
    x_lim_st = _ch3_xlim_for_seq(seq)
    ws1, we1, bb1 = triples[-1]
    x_lim_el = _ch3_xlim_for_seq(
        np.linspace(ch3_active_value("el", ws1, we1, bb1) - CH3_LOSS_SYM_DELTA, ch3_active_value("el", ws1, we1, bb1) + CH3_LOSS_SYM_DELTA, 8)
    )
    x_lim_b = _ch3_xlim_for_seq(
        np.linspace(ch3_active_value("b", ws1, we1, bb1) - CH3_LOSS_SYM_DELTA, ch3_active_value("b", w_st, w_el, b) + CH3_LOSS_SYM_DELTA, 8)
    )
    y_lim = _ch3_ylim_for_ys(losses)
    lo_rot, hi_rot = float(np.min(seq)), float(np.max(seq))
    frames = []
    def emit(ws, we, bb, n_hist, rot_deg, scales):
        rots = [
            ch3_knob_strip_rotation_deg(ch3_active_value("st", ws, we, bb), lo_rot, hi_rot),
            0.0,
            0.0,
        ]
        frames.append(
            ch3_triptych_frame_panels(
                ws,
                we,
                bb,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                xs_st=seq[:n_hist].tolist(),
                ys_st=losses[:n_hist].tolist(),
                xs_el=[],
                ys_el=[],
                xs_b=[],
                ys_b=[],
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lim=y_lim,
                knob_rots=rots,
                knob_scales=list(scales),
                panel_visible=(True, False, False),
                panel_alpha=(1.0, 0.0, 0.0),
                emphasize_knob="st",
            )
        )
    ws0, we0, bb0 = triples[0]
    if grow:
        ug = [1.0] if CH3_N_GROW <= 1 else [float(j) / float(CH3_N_GROW - 1) for j in range(CH3_N_GROW)]
        for u in ug:
            sc = [1.0 + (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u), 1.0, 1.0]
            emit(ws0, we0, bb0, 1, 0.0, sc)
    for i, (ws, we, bb) in enumerate(triples):
        val = ch3_active_value("st", ws, we, bb)
        rot = ch3_knob_rots_for_weights(*ch3_script_weights(which, val, w_st, w_el, b))[{"st": 0, "el": 1, "b": 2}[which]]
        sc = [CH3_KNOB_ACTIVE_SCALE, 1.0, 1.0]
        emit(ws, we, bb, i + 1, rot, sc)
    if shrink:
        us = [1.0] if CH3_N_SHRINK <= 1 else [float(j) / float(CH3_N_SHRINK - 1) for j in range(CH3_N_SHRINK)]
        val_end = ch3_active_value("st", ws1, we1, bb1)
        rot_end = ch3_knob_rots_for_weights(*triples[-1])[{"st": 0, "el": 1, "b": 2}[which]]
        for u in us:
            sc = [
                CH3_KNOB_ACTIVE_SCALE - (CH3_KNOB_ACTIVE_SCALE - 1.0) * ch3_knob_smoothstep(u),
                1.0,
                1.0,
            ]
            emit(ws1, we1, bb1, len(seq), rot_end, sc)
    return frames
def ch3_build_frames_triptych_reveal_el_b(loss_key, with_noise, w_st, w_el, b):
    """Fade in the w2 and b loss panels at fixed weights (before sweeping those knobs)."""
    spec = CH3_LOSS_SPECS[loss_key]
    loss_fn = spec["fn"]
    show_cmap = bool(spec["colormap"])
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    d = float(CH3_LOSS_SYM_DELTA)
    cn = int(CH3_SCRIPT_COUPLED_CURVE_N)
    seq_st, ys_st = ch3_script_loss_curve(loss_fn, "st", w_st, w_el, b, study, exam, y, w_st - d, w_st + d, cn)
    seq_el, ys_el = ch3_script_loss_curve(loss_fn, "el", w_st, w_el, b, study, exam, y, w_el - d, w_el + d, cn)
    seq_b, ys_b = ch3_script_loss_curve(loss_fn, "b", w_st, w_el, b, study, exam, y, b - d, b + d, cn)
    x_lim_st = _ch3_xlim_centered(w_st, d)
    x_lim_el = _ch3_xlim_centered(w_el, d)
    x_lim_b = _ch3_xlim_centered(b, d)
    y_lim = _ch3_ylim_for_ys(ys_st, ys_el, ys_b)
    lo1, hi1 = float(np.min(seq_st)), float(np.max(seq_st))
    rots = list(ch3_knob_rot_deg_from_refs(w_st, w_el, b, (w_st, w_el, b)))
    scales = [1.0, 1.0, 1.0]
    frames = []
    if CH3_SCRIPT_N_REVEAL <= 1:
        alphas = [1.0]
    else:
        alphas = [float(j) / float(CH3_SCRIPT_N_REVEAL - 1) for j in range(CH3_SCRIPT_N_REVEAL)]
    for u in alphas:
        au = ch3_knob_smoothstep(u)
        frames.append(
            ch3_triptych_frame_panels(
                w_st,
                w_el,
                b,
                study,
                exam,
                y,
                spec,
                show_colormap=show_cmap,
                xs_st=seq_st.tolist(),
                ys_st=ys_st.tolist(),
                xs_el=seq_el.tolist(),
                ys_el=ys_el.tolist(),
                xs_b=seq_b.tolist(),
                ys_b=ys_b.tolist(),
                x_lim_st=x_lim_st,
                x_lim_el=x_lim_el,
                x_lim_b=x_lim_b,
                y_lim=y_lim,
                knob_rots=rots,
                knob_scales=scales,
                panel_visible=(True, True, True),
                panel_alpha=(1.0, au, au),
                emphasize_knob="st",
            )
        )
    last = frames[-1]
    for _ in range(CH3_SCRIPT_N_HOLD):
        frames.append(last.copy())
    return frames
def ch3_export_script_mistakes_triptych_reveal_el_b():
    w2, b = CH3_SCRIPT_W2, CH3_SCRIPT_B
    ws = float(CH3_SCRIPT_W1_REF)
    frames = ch3_build_frames_triptych_reveal_el_b("mistakes", False, ws, w2, b)
    ch3_export_script_mp4(ch3_export_filename("mistakes_triptych_reveal_el_b"), frames)
def ch3_export_script_mistakes_k1_flat_small():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_W_ST)
    d_flat = float(CH3_SCRIPT_K1_FLAT_SWEEP_DELTA)
    x_lim, cache_wide, _d_wide = ch3_k1_wide_plot_limits("mistakes", w2, b, c, False)
    xs_flat, ys_flat = cache_wide.slice_through(c - d_flat, c + d_flat)
    y_lim = ch3_k1_ylim_on_ys(ys_flat)
    frames = ch3_build_frames_k1_center_bidirectional(
        "mistakes", False, c, w2, b,
        sweep_delta=d_flat, cache=cache_wide, show_colormap=False,
        n_bidir=CH3_SCRIPT_N_FLAT,
        x_lim_st_override=x_lim, y_lim_override=y_lim,
    )
    ch3_export_script_mp4(ch3_export_filename("mistakes_k1_flat_small"), frames)
def ch3_export_script_mistakes_k1_staircase_wide():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_W_ST)
    d_flat = float(CH3_SCRIPT_K1_FLAT_SWEEP_DELTA)
    x_lim, cache_wide, d_wide = ch3_k1_wide_plot_limits(w2, b, c, False)
    xs_flat, ys_flat = cache_wide.slice_through(c - d_flat, c + d_flat)
    y_lim = ch3_k1_ylim_on_ys(ys_flat)
    frames = ch3_build_frames_k1_wide_extend_from_flat(
        "mistakes", False, c, w2, b, xs_flat, ys_flat, d_flat, d_wide,
        cache=cache_wide, x_lim_st=x_lim, y_lim=y_lim, show_colormap=False,
    )
    ch3_export_script_mp4(ch3_export_filename("mistakes_k1_staircase_wide"), frames)
def ch3_build_frames_mistakes_k1_descent_step(w_from, w_to, *, grow_start=True, shrink_end=True):
    """Slide full curve to ``w_to`` (clips right), then sweep left (staircase descent)."""
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    return ch3_build_frames_k1_descent_shift_fill(
        "mistakes", False, float(w_from), float(w_to), w2, b,
        show_colormap=False, grow_start=grow_start, shrink_end=shrink_end,
    )
def ch3_export_script_mistakes_k1_staircase_descent_at(w_to, *, w_from, grow_start=True, shrink_end=True):
    frames = ch3_build_frames_mistakes_k1_descent_step(
        w_from, w_to, grow_start=grow_start, shrink_end=shrink_end,
    )
    ch3_export_script_mp4(ch3_export_filename(ch3_export_key_k1_descent(w_to)), frames)
def ch3_export_script_mistakes_k1_staircase_descent_all():
    centers = ch3_script_k1_descent_centers()
    w_prev = float(CH3_SCRIPT_K1_W_ST)
    for i, w_to in enumerate(centers):
        ch3_export_script_mistakes_k1_staircase_descent_at(
            w_to, w_from=w_prev, grow_start=True, shrink_end=True,
        )
        w_prev = w_to
def ch3_export_script_mistakes_k1_staircase_descent():
    centers = ch3_script_k1_descent_centers()
    ch3_export_script_mistakes_k1_staircase_descent_at(
        centers[0], w_from=float(CH3_SCRIPT_K1_W_ST), grow_start=True,
    )
def ch3_export_script_mistakes_k1_staircase_stop():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_DESCENT_END)
    d = float(CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
    cache = ch3_k1_loss_cache("mistakes", w2, b, c, False, ax_half=d)
    frames = ch3_build_frames_k1_center_bidirectional(
        "mistakes", False, c, w2, b,
        sweep_delta=d, cache=cache, show_colormap=False, n_bidir=CH3_SCRIPT_N_LOCAL,
    )
    ch3_export_script_mp4(ch3_export_filename("mistakes_k1_staircase_stop"), frames)
def ch3_export_script_mistakes_k1_tiny_triptych_coupled():
    w2, b = CH3_SCRIPT_W2, CH3_SCRIPT_B
    c = float(CH3_SCRIPT_W1_REF)
    d = CH3_SCRIPT_W1_TINY_DELTA
    path = ch3_script_seq_lin(c - d, c + d, CH3_SCRIPT_N_TINY)
    frames = ch3_build_frames_triptych_coupled(
        "mistakes", False, "st", c, w2, b, path, curve_delta=CH3_LOSS_SYM_DELTA,
    )
    ch3_export_script_mp4(ch3_export_filename("mistakes_k1_tiny_triptych_coupled"), frames)
def ch3_export_script_sumprob_k1_small():
    w2, b = CH3_SCRIPT_W2, CH3_SCRIPT_B
    c = float(CH3_SCRIPT_W1_REF)
    d = CH3_SCRIPT_W1_TINY_DELTA
    path = ch3_script_seq_lin(c - d, c + d, CH3_SCRIPT_N_TINY)
    frames = ch3_build_frames_duo_path("sumprob", "st", False, c, w2, b, path, show_colormap=True)
    ch3_export_script_mp4(ch3_export_filename("sumprob_k1_small"), frames)

def ch3_export_key_sumprob_k1_descent(w_center):
    tag = int(round(float(w_center) * 10))
    return f"sumprob_k1_staircase_descent_w{tag:02d}"

def ch3_build_frames_sumprob_k1_descent_step(w_from, w_to, *, grow_start=True, shrink_end=True):
    return ch3_build_frames_k1_descent_shift_fill(
        "sumprob", False, float(w_from), float(w_to), CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B,
        show_colormap=True, grow_start=grow_start, shrink_end=shrink_end,
        y_lim_fixed=True,
    )

def ch3_export_script_sumprob_k1_flat_small():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_W_ST)
    d_flat = float(CH3_SCRIPT_K1_FLAT_SWEEP_DELTA)
    x_lim, cache_wide, _d_wide = ch3_k1_wide_plot_limits("sumprob", w2, b, c, False)
    xs_flat, ys_flat = cache_wide.slice_through(c - d_flat, c + d_flat)
    y_lim = ch3_k1_ylim_for_sweep(cache_wide, c - d_flat, c + d_flat)
    frames = ch3_build_frames_k1_center_bidirectional(
        "sumprob", False, c, w2, b,
        sweep_delta=d_flat, cache=cache_wide, show_colormap=True,
        n_bidir=CH3_SCRIPT_N_FLAT,
        x_lim_st_override=x_lim, y_lim_override=y_lim, y_lim_fixed=True,
    )
    ch3_export_script_mp4(ch3_export_filename("sumprob_k1_flat_small"), frames)

def ch3_export_script_sumprob_k1_staircase_wide():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_W_ST)
    d_flat = float(CH3_SCRIPT_K1_FLAT_SWEEP_DELTA)
    x_lim, cache_wide, d_wide = ch3_k1_wide_plot_limits("sumprob", w2, b, c, False)
    xs_flat, ys_flat = cache_wide.slice_through(c - d_flat, c + d_flat)
    y_lim = ch3_k1_ylim_for_sweep(cache_wide, c - d_wide, c + d_wide)
    frames = ch3_build_frames_k1_wide_extend_from_flat(
        "sumprob", False, c, w2, b, xs_flat, ys_flat, d_flat, d_wide,
        cache=cache_wide, x_lim_st=x_lim, y_lim=y_lim, show_colormap=True,
        y_lim_fixed=True, start_with_flat=False,
    )
    ch3_export_script_mp4(ch3_export_filename("sumprob_k1_staircase_wide"), frames)

def ch3_export_script_sumprob_k1_staircase_descent_at(w_to, *, w_from, grow_start=True, shrink_end=True):
    frames = ch3_build_frames_sumprob_k1_descent_step(
        w_from, w_to, grow_start=grow_start, shrink_end=shrink_end,
    )
    ch3_export_script_mp4(ch3_export_filename(ch3_export_key_sumprob_k1_descent(w_to)), frames)


def ch3_export_key_sumprob_k1_tight_descent(w_center):
    tag = int(round(float(w_center) * 10))
    return f"sumprob_k1_tight_staircase_descent_w{tag:02d}"

def ch3_build_frames_sumprob_k1_tight_descent_step(w_from, w_to, *, grow_start=True, shrink_end=True):
    return ch3_build_frames_k1_descent_shift_fill(
        "sumprob", False, float(w_from), float(w_to), CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B,
        sweep_delta=CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA,
        show_colormap=True,
        grow_start=grow_start,
        shrink_end=shrink_end,
        arrows_left=True,
        y_lim_fixed=True,
    )

def ch3_export_script_sumprob_k1_tight_staircase_wide():
    w2, b = CH3_SCRIPT_K1_W_EL, CH3_SCRIPT_K1_B
    c = float(CH3_SCRIPT_K1_W_ST)
    d_tight = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA)
    x_lim, cache_wide, _ = ch3_k1_wide_plot_limits("sumprob", w2, b, c, False, sweep_delta=d_tight)
    xs_flat, ys_flat = cache_wide.slice_through(c - d_tight, c + d_tight)
    y_lim = ch3_k1_ylim_for_sweep(cache_wide, c - d_tight, c + d_tight)
    frames = ch3_build_frames_k1_wide_extend_from_flat(
        "sumprob", False, c, w2, b, xs_flat, ys_flat, d_tight, d_tight,
        cache=cache_wide, x_lim_st=x_lim, y_lim=y_lim, show_colormap=True,
        arrows_left=True, y_lim_fixed=True,
    )
    ch3_export_script_mp4(ch3_export_filename("sumprob_k1_tight_staircase_wide"), frames)

def ch3_export_script_sumprob_k1_tight_staircase_descent_at(w_to, *, w_from, grow_start=True, shrink_end=True):
    frames = ch3_build_frames_sumprob_k1_tight_descent_step(
        w_from, w_to, grow_start=grow_start, shrink_end=shrink_end,
    )
    ch3_export_script_mp4(ch3_export_filename(ch3_export_key_sumprob_k1_tight_descent(w_to)), frames)

# End state of ch3_52 (tight descent 1.15 → 0.95).
CH3_SCRIPT_K1_TIGHT_DESCENT_W52 = 0.95
CH3_SCRIPT_K1_SLOPE_DX = 0.04
CH3_SCRIPT_N_SLOPE_PHASE = max(14, _smooth_n(10))
CH3_SCRIPT_N_SLOPE_MOVE = max(10, _smooth_n(6))
CH3_SLOPE_DX_COLOR = "#1565c0"
CH3_SLOPE_DY_COLOR = "#e65100"


def ch3_k1_slope_at(cache, w, *, h=None):
    """d(loss)/dw1 at ``w`` via central difference on the cached 1D slice."""
    w = float(w)
    if h is None:
        h = max(1e-5, 0.08 * float(cache.ax_half))
    _, yp = cache.point_at(w + h)
    _, ym = cache.point_at(w - h)
    return float((yp - ym) / (2.0 * h))


def ch3_draw_slope_triangle(ax, x0, y0, dx, slope, *, phase="both", zorder=6):
    """
    Right triangle for Δy = slope·Δx; base centered on ``(x0, y0)`` on the curve.
    ``phase``: ``both`` | ``dx`` | ``dy`` — which leg is emphasized.
    """
    dx = float(dx)
    x0, y0 = float(x0), float(y0)
    dy = float(slope) * dx
    x_lo = x0 - 0.5 * dx
    x_hi = x0 + 0.5 * dx
    y_top = y0 + dy
    if phase == "dx":
        lw_dx, al_dx, lw_dy, al_dy = 4.2, 1.0, 1.4, 0.22
        lw_hyp, al_hyp = 1.2, 0.18
    elif phase == "dy":
        lw_dx, al_dx, lw_dy, al_dy = 1.4, 0.22, 4.2, 1.0
        lw_hyp, al_hyp = 1.2, 0.18
    else:
        lw_dx = lw_dy = lw_hyp = 2.6
        al_dx = al_dy = al_hyp = 0.88

    tri = plt.Polygon(
        [(x_lo, y0), (x_hi, y0), (x_hi, y_top)],
        closed=True,
        facecolor=CH3_SLOPE_DY_COLOR,
        edgecolor="none",
        alpha=0.10 * min(al_dx, al_dy, 1.0),
        zorder=zorder - 1,
    )
    ax.add_patch(tri)
    ax.plot(
        [x_lo, x_hi],
        [y0, y0],
        color=CH3_SLOPE_DX_COLOR,
        linewidth=lw_dx,
        solid_capstyle="round",
        zorder=zorder,
        alpha=al_dx,
    )
    ax.plot(
        [x_hi, x_hi],
        [y0, y_top],
        color=CH3_SLOPE_DY_COLOR,
        linewidth=lw_dy,
        solid_capstyle="round",
        zorder=zorder + 1,
        alpha=al_dy,
    )
    ax.plot(
        [x_lo, x_hi],
        [y0, y_top],
        color="0.35",
        linewidth=lw_hyp,
        linestyle="--",
        zorder=zorder,
        alpha=al_hyp,
    )
    ax.scatter(
        [x0],
        [y0],
        s=140,
        color=CH3_SLOPE_DX_COLOR,
        zorder=zorder + 2,
        edgecolors="white",
        linewidths=0.9,
        alpha=max(al_dx, al_dy),
    )


def ch3_sumprob_k1_tight_panel_state(w_knob, *, cache=None):
    """Triptych panel data matching the end of a tight descent clip at ``w_knob``."""
    w_knob = float(w_knob)
    w2, b = float(CH3_SCRIPT_K1_W_EL), float(CH3_SCRIPT_K1_B)
    d = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA)
    if cache is None:
        cache = ch3_k1_loss_cache("sumprob", w2, b, w_knob, False, ax_half=d)
    xs, ys = cache.slice_through(w_knob - d, w_knob + d)
    x_lim_st = ch3_k1_wide_xlim(float(CH3_SCRIPT_K1_W_ST), d)
    y_lim = ch3_k1_ylim_on_ys(ys)
    mx, my = cache.point_at(w_knob)
    return {
        "cache": cache,
        "w_knob": w_knob,
        "w2": w2,
        "b": b,
        "xs": xs,
        "ys": ys,
        "x_lim_st": x_lim_st,
        "y_lim": y_lim,
        "marker": (float(mx), float(my)),
        "slope": ch3_k1_slope_at(cache, w_knob),
    }


def ch3_build_frames_sumprob_k1_slope_triangles(
    *,
    w_end=None,
    w_samples=None,
    n_phase=None,
    n_move=None,
    hold_intro=True,
):
    """
    After ch3_52 (tight descent to w≈0.95): slope triangles along the visible curve.
    At each knob location: show triangle → emphasize Δx → emphasize Δy.
    """
    w_end = float(CH3_SCRIPT_K1_TIGHT_DESCENT_W52 if w_end is None else w_end)
    d = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA)
    n_phase = int(CH3_SCRIPT_N_SLOPE_PHASE if n_phase is None else n_phase)
    n_move = int(CH3_SCRIPT_N_SLOPE_MOVE if n_move is None else n_move)
    if w_samples is None:
        w_lo, w_hi = w_end - d, w_end + d
        w_samples = np.linspace(w_lo, w_hi, 5, endpoint=True)
    else:
        w_samples = np.asarray(w_samples, dtype=float).ravel()

    st0 = ch3_sumprob_k1_tight_panel_state(w_end)
    cache = st0["cache"]
    spec = CH3_LOSS_SPECS["sumprob"]
    study, exam, y = study_sep, exam_sep, y_sep
    frames = []

    def frame_at(w_knob, *, phase=None, scale_st=CH3_K1_ACTIVE_SCALE_FULL):
        st = ch3_sumprob_k1_tight_panel_state(w_knob, cache=cache)
        _hl, _sz = ch3_k1_data_panel_extras("sumprob", w_knob, st["w2"], st["b"], True)
        slope_tri = None
        if phase is not None:
            slope_tri = {
                "x0": st["marker"][0],
                "y0": st["marker"][1],
                "dx": float(CH3_SCRIPT_K1_SLOPE_DX),
                "slope": st["slope"],
                "phase": phase,
            }
        return ch3_triptych_frame_panels(
            w_knob,
            st["w2"],
            st["b"],
            study,
            exam,
            y,
            spec,
            show_colormap=True,
            highlight_mistakes_flag=_hl,
            sigma_Z=_sz,
            xs_st=np.asarray(st["xs"], dtype=float).tolist(),
            ys_st=np.asarray(st["ys"], dtype=float).tolist(),
            xs_el=[],
            ys_el=[],
            xs_b=[],
            ys_b=[],
            x_lim_st=st["x_lim_st"],
            x_lim_el=_ch3_xlim_centered(st["w2"], d),
            x_lim_b=_ch3_xlim_centered(st["b"], d),
            y_lim=st["y_lim"],
            knob_rots=ch3_k1_knob_rots_at(w_knob, st["w2"], st["b"]),
            knob_scales=ch3_knob_scales_emphasize("st", scale_st),
            panel_visible=(True, False, False),
            marker_st=st["marker"],
            show_tick_labels=CH3_SCRIPT_K1_SHOW_TICK_LABELS,
            slope_triangle=slope_tri,
        )

    if hold_intro:
        intro = frame_at(w_end, phase=None)
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(intro.copy())

    w_prev = float(w_samples[0])
    for i, w_knob in enumerate(w_samples):
        w_knob = float(w_knob)
        if i > 0 and n_move > 1:
            for t in np.linspace(0.0, 1.0, n_move, endpoint=True)[1:]:
                w_mid = w_prev + t * (w_knob - w_prev)
                frames.append(frame_at(w_mid, phase=None))
        w_prev = w_knob
        for phase in ("both", "dx", "dy"):
            img = frame_at(w_knob, phase=phase)
            for _ in range(n_phase):
                frames.append(img.copy())

    if frames:
        last = frames[-1]
        for _ in range(CH3_SCRIPT_N_HOLD):
            frames.append(last.copy())
    return frames


def ch3_export_script_sumprob_k1_slope_triangles():
    frames = ch3_build_frames_sumprob_k1_slope_triangles()
    ch3_export_script_mp4(ch3_export_filename("sumprob_k1_slope_triangles"), frames)


def ch3_export_script_sumprob_k1_staircase_descent_all():
    centers = ch3_script_k1_descent_centers()
    w_prev = float(CH3_SCRIPT_K1_W_ST)
    for w_to in centers:
        ch3_export_script_sumprob_k1_staircase_descent_at(
            w_to, w_from=w_prev, grow_start=True, shrink_end=True,
        )
        w_prev = w_to

def ch3_export_script_sumprob_far_flat():
    w2, b = CH3_SCRIPT_W2, CH3_SCRIPT_B
    c = CH3_SCRIPT_SUMPROB_FAR_W1
    d = 0.35
    path = ch3_script_seq_lin(c - d, c + d, CH3_SCRIPT_N_WIDE)
    frames = ch3_build_frames_duo_path("sumprob", "st", False, c, w2, b, path, show_colormap=True)
    ch3_export_script_mp4(ch3_export_filename("sumprob_far_flat"), frames)
def ch3_export_script_still_dataset_clean():
    frames = ch3_build_frames_still("mistakes", False, CH3_W_ST_REF, CH3_W_EL_REF, CH3_B_REF, show_colormap=True)
    for _ in range(CH3_SCRIPT_N_STILL - 1):
        frames.append(frames[0].copy())
    ch3_export_script_mp4(ch3_export_filename("dataset_clean_still"), frames)
def ch3_export_script_still_dataset_noise():
    frames = ch3_build_frames_still("mistakes", True, CH3_W_ST_REF, CH3_W_EL_REF, CH3_B_REF, show_colormap=True)
    for _ in range(CH3_SCRIPT_N_STILL - 1):
        frames.append(frames[0].copy())
    ch3_export_script_mp4(ch3_export_filename("dataset_noise_still"), frames)
def ch3_export_script_still_dataset_clean_plain():
    frames = ch3_build_frames_dataset_still(study_sep, exam_sep, y_sep)
    ch3_export_script_mp4(ch3_export_filename("dataset_clean_plain_still"), frames)
def ch3_export_script_still_dataset_clean_st_el_zero():
    ws, we, bb = CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B
    frames = ch3_build_frames_dataset_still(study_sep, exam_sep, y_sep, w_st=ws, w_el=we, b=bb, show_colormap=False)
    ch3_export_script_mp4(ch3_export_filename("dataset_clean_st_el_zero_still"), frames)
def ch3_export_script_still_dataset_noise_plain():
    frames = ch3_build_frames_dataset_still(study_real, exam_real, y_real)
    ch3_export_script_mp4(ch3_export_filename("dataset_noise_plain_still"), frames)
def ch3_export_script_still_goals_clean():
    frames = ch3_build_frames_still("mistakes", False, CH3_SCRIPT_W1_FLAT, CH3_SCRIPT_W2, CH3_SCRIPT_B, show_colormap=True)
    for _ in range(CH3_SCRIPT_N_STILL - 1):
        frames.append(frames[0].copy())
    ch3_export_script_mp4(ch3_export_filename("goals_clean_still"), frames)
# --- Keras NLL weight-space (script L55–63); TensorFlow imported inside exporters ---
CH3_KERAS_NLL3D_LR = 1e-2
CH3_KERAS_NLL3D_EPOCHS = 200
CH3_KERAS_NLL3D_BATCH = 100
CH3_KERAS_NLL3D_HOLD_LAST = 20
CH3_KERAS_NLL3D_MS = CH3_MP4_MS_PER_FRAME
CH3_KERAS_NLL3D_GRID_PER_AXIS = 24
CH3_KERAS_NLL3D_SCATTER_ALPHA = 0.4
CH3_KERAS_NLL3D_SCATTER_SIZE = 11.0
CH3_KERAS_NLL3D_FIGSIZE = (16.0, 9.5)
CH3_KERAS_NLL3D_WIDTH_RATIOS = (1.14, 1.24)
CH3_KERAS_NLL3D_WSPACE = 0.11
CH3_KERAS_NLL3D_PAD_FRAC = 0.12
CH3_KERAS_NLL3D_BALL_NSAMPLE = 40
CH3_KERAS_NLL3D_BALL_R_SCALE = 0.07
CH3_KERAS_NLL3D_BALL_POINT_ALPHA = 0.42
CH3_KERAS_NLL3D_BALL_POINT_SIZE = 13.0
CH3_KERAS_NLL3D_CAM_ELEV = 22.0
CH3_KERAS_NLL3D_CAM_AZIM_START = -135.0
CH3_KERAS_NLL3D_CAM_AZIM_DELTA = 45.0
CH3_KERAS_NLL3D_VIEW_PAD_FRAC = 0.034
CH3_KERAS_NLL3D_ZOOM_MIN_SPAN_FRAC = 0.22
CH3_KERAS_NLL3D_SCATTER_ALPHA_ZOOM_HI = 0.52
CH3_KERAS_NLL3D_SCATTER_ALPHA_ZOOM_LO = 0.22
CH3_KERAS_NLL3D_BALL_ALPHA_ZOOM_HI = 0.55
CH3_KERAS_NLL3D_BALL_ALPHA_ZOOM_LO = 0.18
CH3_KERAS_NLL3D_POINT_QUIVER_LEN = 0.1
CH3_KERAS_NLL3D_POINT_QUIVER_COLOR = "0.2"
CH3_KERAS_NLL3D_POINT_QUIVER_LW = 1.15
CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_LEN = 0.12
CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_COLOR = "#d62728"
CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_LW = 1.35
# --- Bridge MP4: dataset → three knobs (reverse of chap2 ``ch2_99``) → slide into chap3 duo left slot ---
# Bridge plane: $1.36\,\mathrm{ST}+0.48\,\mathrm{EL}+0=0$ (keras-adjacent init display).
CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B = 1.36, 0.48, 0.0
CH3_BRIDGE_HOLD_PLAIN = max(8, _smooth_n(5))
CH3_BRIDGE_PHASE1_N = max(36, _smooth_n(28))
CH3_BRIDGE_PHASE2_N = max(40, _smooth_n(30))
CH3_BRIDGE_HOLD_END = max(10, _smooth_n(6))
CH3_BRIDGE_MP4_MS = CH3_MP4_MS_PER_FRAME
_CH3_DEFAULT_AX_RECT = None
_CH3_CH2_STRIP_LAYOUT = None
_CH3_DUO_LEFT_LAYOUT = None
_CH3_DUO_RIGHT_RECT = None
def ch3_default_export_axes_rect():
    """Same axes box as ``plt.subplots(figsize=EXPORT_FIGSIZE)`` (chap1 / chap2 dataset panels)."""
    global _CH3_DEFAULT_AX_RECT
    if _CH3_DEFAULT_AX_RECT is not None:
        return _CH3_DEFAULT_AX_RECT
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    fig.canvas.draw()
    pr = ax.get_position()
    _CH3_DEFAULT_AX_RECT = (float(pr.x0), float(pr.y0), float(pr.width), float(pr.height))
    plt.close(fig)
    return _CH3_DEFAULT_AX_RECT
def ch3_ch2_strip_layout_rects():
    """Chap2 knob-strip geometry: full-width data row + three bottom slots."""
    global _CH3_CH2_STRIP_LAYOUT
    if _CH3_CH2_STRIP_LAYOUT is not None:
        return _CH3_CH2_STRIP_LAYOUT
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.34], hspace=0.14, wspace=0.08)
    axd = fig.add_subplot(gs[0, :])
    axk = [fig.add_subplot(gs[1, j]) for j in range(3)]
    fig.subplots_adjust(left=0.06, right=0.98, top=0.94, bottom=0.06)
    fig.canvas.draw()
    dr = axd.get_position()
    data_r = (float(dr.x0), float(dr.y0), float(dr.width), float(dr.height))
    kr = []
    for a in axk:
        pr = a.get_position()
        kr.append((float(pr.x0), float(pr.y0), float(pr.width), float(pr.height)))
    plt.close(fig)
    _CH3_CH2_STRIP_LAYOUT = (data_r, tuple(kr))
    return _CH3_CH2_STRIP_LAYOUT
def ch3_duo_left_layout_rects():
    """Left column rects inside the same duo figure used by ``ch3_figure_duo``."""
    global _CH3_DUO_LEFT_LAYOUT
    if _CH3_DUO_LEFT_LAYOUT is not None:
        return _CH3_DUO_LEFT_LAYOUT
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    axd = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axk = [fig.add_subplot(g_k[0, j]) for j in range(3)]
    fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, axd, tuple(axk))
    fig.canvas.draw()
    dr = axd.get_position()
    data_r = (float(dr.x0), float(dr.y0), float(dr.width), float(dr.height))
    kr = []
    for a in axk:
        pr = a.get_position()
        kr.append((float(pr.x0), float(pr.y0), float(pr.width), float(pr.height)))
    plt.close(fig)
    _CH3_DUO_LEFT_LAYOUT = (data_r, tuple(kr))
    return _CH3_DUO_LEFT_LAYOUT
def ch3_duo_right_axis_rect():
    """Right subplot slot (empty loss panel) in the duo layout."""
    global _CH3_DUO_RIGHT_RECT
    if _CH3_DUO_RIGHT_RECT is not None:
        return _CH3_DUO_RIGHT_RECT
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    axd = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axk = [fig.add_subplot(g_k[0, j]) for j in range(3)]
    axr = fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, axd, tuple(axk))
    fig.canvas.draw()
    pr = axr.get_position()
    _CH3_DUO_RIGHT_RECT = (float(pr.x0), float(pr.y0), float(pr.width), float(pr.height))
    plt.close(fig)
    return _CH3_DUO_RIGHT_RECT
def ch3_bridge_knob_side_from_slot(slot_rect):
    _, _, w, h = slot_rect
    return float(min(w, h))
def ch3_bridge_lerp_center(u, rect_a, rect_b):
    ax, ay, _, _ = ch3_bridge_rect_center_wh(rect_a)
    bx, by, _, _ = ch3_bridge_rect_center_wh(rect_b)
    u = float(np.clip(u, 0.0, 1.0))
    return ax + u * (bx - ax), ay + u * (by - ay)
def ch3_bridge_add_knob(fig, img, cx, cy, side):
    side = float(side)
    axk = fig.add_axes([cx - 0.5 * side, cy - 0.5 * side, side, side])
    axk.imshow(np.asarray(img))
    axk.axis("off")
def ch3_bridge_smoothstep(u):
    u = float(np.clip(u, 0.0, 1.0))
    return u * u * (3.0 - 2.0 * u)
def ch3_bridge_lerp_rect(t, a, b):
    u = ch3_bridge_smoothstep(float(np.clip(t, 0.0, 1.0)))
    return tuple(float(a[i] + u * (b[i] - a[i])) for i in range(4))
def ch3_bridge_rect_center_wh(r):
    x0, y0, w, h = r
    return x0 + 0.5 * w, y0 + 0.5 * h, w, h
def ch3_bridge_legend_three_starts(fig, axd, renderer, data_r):
    leg = axd.get_legend()
    if leg is None or not leg.get_texts():
        ddx, ddy, ddw, ddh = data_r
        s1 = (ddx + 0.08 * ddw, ddy + 0.88 * ddh)
        s2 = (ddx + 0.30 * ddw, ddy + 0.88 * ddh)
        s3 = (2.0 * s2[0] - s1[0], 2.0 * s2[1] - s1[1])
        return s1, s2, s3
    bb = leg.get_window_extent(renderer=renderer)
    inv = fig.transFigure.inverted()
    y_disp = float(bb.y0 + 0.42 * bb.height)
    x_lo = float(bb.x0 + 0.10 * bb.width)
    x_hi = float(bb.x1 - 0.10 * bb.width)
    out = []
    for alpha in (0.18, 0.52, 0.86):
        xf, yf = inv.transform((x_lo + alpha * (x_hi - x_lo), y_disp))
        out.append((float(xf), float(yf)))
    return tuple(out)
def ch3_bridge_frame_hold_plain_dataset():
    fig, axd = plt.subplots(figsize=EXPORT_FIGSIZE)
    ws, we, bb = CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        axd,
        ws,
        we,
        bb,
        study_sep,
        exam_sep,
        y_sep,
        leg,
        show_colormap=True,
        highlight_mistakes_flag=True,
    )
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 100))
def ch3_bridge_frame_phase1_intro_knobs(t, k1, k2, k3):
    t = float(np.clip(t, 0.0, 1.0))
    u = ch3_bridge_smoothstep(t)
    data_def = ch3_default_export_axes_rect()
    data_strip, knobs_strip = ch3_ch2_strip_layout_rects()
    data_r = ch3_bridge_lerp_rect(t, data_def, data_strip)
    fw, fh = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    fig = plt.figure(figsize=(fw, fh))
    axd = fig.add_axes(data_r)
    ws, we, bb = CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        axd,
        ws,
        we,
        bb,
        study_sep,
        exam_sep,
        y_sep,
        leg,
        show_colormap=True,
        highlight_mistakes_flag=True,
    )
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    s1c, s2c, s3c = ch3_bridge_legend_three_starts(fig, axd, renderer, data_r)
    tiny = max(0.006, 0.02 * (1.0 - u) + 0.001)
    ends = knobs_strip[0], knobs_strip[1], knobs_strip[2]
    e1x, e1y, e1w, e1h = ch3_bridge_rect_center_wh(ends[0])
    e2x, e2y, e2w, e2h = ch3_bridge_rect_center_wh(ends[1])
    _, _, e3w, e3h = ch3_bridge_rect_center_wh(ends[2])
    e3x = e2x + (e2x - e1x)
    e3y = e2y + (e2y - e1y)
    for ki, (img, sc, end_r) in enumerate(((k1, s1c, ends[0]), (k2, s2c, ends[1]), (k3, s3c, ends[2]))):
        if ki == 2:
            end_slot = ends[2]
            ex, ey = e3x, e3y
        else:
            end_slot = end_r
            ex, ey, _, _ = ch3_bridge_rect_center_wh(end_r)
        sx, sy = sc
        cx = sx + u * (ex - sx)
        cy = sy + u * (ey - sy)
        side_tgt = ch3_bridge_knob_side_from_slot(end_slot)
        side = float(tiny + u * (side_tgt - tiny))
        ch3_bridge_add_knob(fig, img, cx, cy, side)
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 100))
def ch3_bridge_frame_phase2_slide_into_duo(t, k1, k2, k3):
    t = float(np.clip(t, 0.0, 1.0))
    u = ch3_bridge_smoothstep(t)
    d_strip, knobs_strip = ch3_ch2_strip_layout_rects()
    d_duo, knobs_duo = ch3_duo_left_layout_rects()
    data_r = ch3_bridge_lerp_rect(u, d_strip, d_duo)
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("white")
    axd = fig.add_axes(data_r)
    ws, we, bb = CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        axd,
        ws,
        we,
        bb,
        study_sep,
        exam_sep,
        y_sep,
        leg,
        show_colormap=True,
        highlight_mistakes_flag=True,
    )
    imgs = (k1, k2, k3)
    for i in range(3):
        cx, cy = ch3_bridge_lerp_center(u, knobs_strip[i], knobs_duo[i])
        side = ch3_bridge_knob_side_from_slot(knobs_duo[i])
        ch3_bridge_add_knob(fig, imgs[i], cx, cy, side)
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 100))
CH3_BRIDGE_KERAS_INIT_HOLD = max(8, _smooth_n(5))
def ch3_bridge_draw_keras_dataset_panel(ax, w_st, w_el, b):
    """Separable roster + dashed boundary at Keras init weights (no sigma colormap)."""
    leg = legend_linear_equation_values_bold_param(float(w_st), float(w_el), float(b), None)
    bxy = boundary_line_xy(
        float(w_st), float(w_el), float(b),
        float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]),
    )
    if bxy is not None:
        bx, by = bxy
        ax.plot(bx, by, c="grey", linestyle="--", linewidth=1.8, label=leg, zorder=3)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.2)
    ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
    finalize_style_legend_tex(ax)
def ch3_bridge_frame_hold_keras_init(w_st, w_el, b):
    """Keras Step-0 weights on the full-page dataset axes (no knobs yet)."""
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    axd = fig.add_axes(ch3_default_export_axes_rect())
    ch3_bridge_draw_keras_dataset_panel(axd, w_st, w_el, b)
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 100))
def ch3_bridge_frame_phase1_intro_knobs_keras(t, k1, k2, k3, w_st, w_el, b):
    """Bridge phase 1 only: legend to strip knobs; panel shows Keras init weights."""
    t = float(np.clip(t, 0.0, 1.0))
    u = ch3_bridge_smoothstep(t)
    data_def = ch3_default_export_axes_rect()
    data_strip, knobs_strip = ch3_ch2_strip_layout_rects()
    data_r = ch3_bridge_lerp_rect(t, data_def, data_strip)
    fw, fh = float(EXPORT_FIGSIZE[0]), float(EXPORT_FIGSIZE[1])
    fig = plt.figure(figsize=(fw, fh))
    axd = fig.add_axes(data_r)
    ch3_bridge_draw_keras_dataset_panel(axd, w_st, w_el, b)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    s1c, s2c, s3c = ch3_bridge_legend_three_starts(fig, axd, renderer, data_r)
    tiny = max(0.006, 0.02 * (1.0 - u) + 0.001)
    ends = knobs_strip[0], knobs_strip[1], knobs_strip[2]
    e1x, e1y, e1w, e1h = ch3_bridge_rect_center_wh(ends[0])
    e2x, e2y, e2w, e2h = ch3_bridge_rect_center_wh(ends[1])
    _, _, e3w, e3h = ch3_bridge_rect_center_wh(ends[2])
    e3x = e2x + (e2x - e1x)
    e3y = e2y + (e2y - e1y)
    for ki, (img, sc, end_r) in enumerate(((k1, s1c, ends[0]), (k2, s2c, ends[1]), (k3, s3c, ends[2]))):
        if ki == 2:
            end_slot = ends[2]
            ex, ey = e3x, e3y
        else:
            end_slot = end_r
            ex, ey, _, _ = ch3_bridge_rect_center_wh(end_r)
        sx, sy = sc
        cx = sx + u * (ex - sx)
        cy = sy + u * (ey - sy)
        side_tgt = ch3_bridge_knob_side_from_slot(end_slot)
        side = float(tiny + u * (side_tgt - tiny))
        ch3_bridge_add_knob(fig, img, cx, cy, side)
    return fig_to_image(fig, dpi=min(int(EXPORT_DPI), 100))
def ch3_export_bridge_dataset_knobs_into_duo_slot_mp4():
    """Reverse-``ch2_99`` knob intro on the separable dataset, then slide into the chap3 duo left column."""
    ch3_ensure_knob_pngs()
    k1 = Image.open(OUTPUT_DIR / "knob_1_cropped.png").convert("RGBA")
    k2 = Image.open(OUTPUT_DIR / "knob_2_cropped.png").convert("RGBA")
    k3 = Image.open(OUTPUT_DIR / "knob_3_cropped.png").convert("RGBA")
    frames = []
    for _ in range(CH3_BRIDGE_HOLD_PLAIN):
        frames.append(ch3_bridge_frame_hold_plain_dataset())
    for tv in np.linspace(0.0, 1.0, CH3_BRIDGE_PHASE1_N, endpoint=True):
        frames.append(ch3_bridge_frame_phase1_intro_knobs(float(tv), k1, k2, k3))
    for tv in np.linspace(0.0, 1.0, CH3_BRIDGE_PHASE2_N, endpoint=True):
        frames.append(ch3_bridge_frame_phase2_slide_into_duo(float(tv), k1, k2, k3))
    for _ in range(CH3_BRIDGE_HOLD_END):
        frames.append(ch3_bridge_frame_phase2_slide_into_duo(1.0, k1, k2, k3))
    save_mp4(frames, ch3_export_filename("bridge"), duration=CH3_BRIDGE_MP4_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / ch3_export_filename("bridge"))
print("Chapter 3 setup OK —", len(study_sep), "clean,", len(study_real), "with noise.")
# Run after the Chapter 3 setup cell. Requires TensorFlow.
import gc
import tensorflow as tf
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from tensorflow import keras
from matplotlib.gridspec import GridSpecFromSubplotSpec
# Adam LR = 1/10 of the Chapter 2 Keras default (1e-1).
CH3_KERAS_GD2D_EPOCHS = 20
CH3_KERAS_GD2D_LR = 1e-1
CH3_KERAS_GD2D_BATCH = 100
CH3_KERAS_GD2D_HOLD_PER_STEP = max(22, _smooth_n(16))
CH3_KERAS_GD2D_MS = max(55, CH3_MP4_MS_PER_FRAME)
# Match Chapter 2 ``_GIF123_DPI`` / ``_snap_keras_plane`` raster size.
CH3_KERAS_GD2D_DPI = min(int(EXPORT_DPI), 100)
CH3_KERAS_GD2D_STEP_XY = (3.5, 3.5)
CH3_KERAS_GD2D_STEP_FS = NOTE_SIZE * 1.35
# Fixed Step-0 plane: 1.36·ST + 0.48·EL + 0 = 0 (same as bridge / weight3d).
CH3_KERAS_GD_INIT_W_ST, CH3_KERAS_GD_INIT_W_EL, CH3_KERAS_GD_INIT_B = (
    CH3_BRIDGE_W_ST,
    CH3_BRIDGE_W_EL,
    CH3_BRIDGE_B,
)
_CH3_KNOB_SLOT_RECTS_CH2 = None
def ch3_knob_rot_deg_from_refs(ws, we, bb, ref=None):
    """PIL dial rotation from coefficient values (integer → upright). ``ref`` ignored."""
    _ = ref
    return ch3_knob_rots_for_weights(ws, we, bb)
def ch3_knob_slot_rects_ch2():
    """Cached knob slots from Chapter 2 strip layout (``_ch2_figure_main_and_knob_axes``)."""
    global _CH3_KNOB_SLOT_RECTS_CH2
    if _CH3_KNOB_SLOT_RECTS_CH2 is not None:
        return _CH3_KNOB_SLOT_RECTS_CH2
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.34], hspace=0.14, wspace=0.08)
    fig.add_subplot(gs[0, :])
    axk = [fig.add_subplot(gs[1, j]) for j in range(3)]
    fig.subplots_adjust(left=0.06, right=0.98, top=0.94, bottom=0.06)
    fig.canvas.draw()
    out = []
    for a in axk:
        pr = a.get_position()
        out.append((float(pr.x0), float(pr.y0), float(pr.width), float(pr.height)))
    plt.close(fig)
    _CH3_KNOB_SLOT_RECTS_CH2 = tuple(out)
    return _CH3_KNOB_SLOT_RECTS_CH2
def ch3_figure_ch2_dataset_knobs():
    """Two-row layout on ``EXPORT_FIGSIZE`` — same geometry as Chapter 2 keras knob clips."""
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.34], hspace=0.14, wspace=0.08)
    ax_data = fig.add_subplot(gs[0, :])
    axes_k = tuple(fig.add_subplot(gs[1, j]) for j in range(3))
    fig.subplots_adjust(left=0.06, right=0.98, top=0.94, bottom=0.06)
    return fig, ax_data, axes_k
def ch3_place_pil_knob_row_ch2(fig, axes_k, knob_rgbs, canvas_sides, rot_degs, slot_scales):
    slot_rects = ch3_knob_slot_rects_ch2()
    for i, axk in enumerate(axes_k):
        x0, y0, w, h = slot_rects[i]
        cx = x0 + 0.5 * w
        cy = y0 + 0.5 * h
        sc = float(np.clip(slot_scales[i], 1.0, float(CH3_KNOB_ACTIVE_SCALE)))
        sw = sc * w
        sh = sc * h
        axk.set_position((cx - 0.5 * sw, cy - 0.5 * sh, sw, sh))
        rd = float(rot_degs[i])
        arr = np.asarray(_ch3_knob_pil_rotated_square(knob_rgbs[i], rd, canvas_sides[i]))
        axk.clear()
        axk.imshow(arr, interpolation="nearest")
        axk.axis("off")
    fig.canvas.draw()
def ch3_draw_knob_row_ch2(fig, axes_k, ws, we, bb, knob_angle_refs):
    """Chapter 2 style: all three dials ×1.25, rotation vs ``knob_angle_refs``."""
    rots = ch3_knob_rot_deg_from_refs(ws, we, bb, knob_angle_refs)
    scales = (CH3_KNOB_ACTIVE_SCALE, CH3_KNOB_ACTIVE_SCALE, CH3_KNOB_ACTIVE_SCALE)
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_place_pil_knob_row_ch2(fig, axes_k, knob_rgbs, canvas_sides, rots, scales)
def ch3_frame_keras_gd_step(study, exam, y, ws, we, bb, step_n, knob_angle_refs, *, show_colormap=False, show_step_label=True):
    fig, ax_data, axes_k = ch3_figure_ch2_dataset_knobs()
    leg = legend_linear_equation_values_bold_param(float(ws), float(we), float(bb), None)
    if show_colormap:
        ch3_draw_left_panel(
            ax_data,
            ws,
            we,
            bb,
            study,
            exam,
            y,
            leg,
            show_colormap=True,
            highlight_mistakes_flag=False,
        )
    else:
        bxy = boundary_line_xy(
            float(ws), float(we), float(bb), float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1])
        )
        if bxy is not None:
            bx, by = bxy
            ax_data.plot(bx, by, c="grey", linestyle="--", linewidth=1.8, label=leg, zorder=3)
        draw_dataset(ax_data, study, exam, y)
        ax_data.legend(loc="upper left", prop={"size": LEGEND_SIZE})
        finalize_style_legend_tex(ax_data)
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    ax_data.grid(alpha=0.2)
    if show_step_label:
        sx, sy = CH3_KERAS_GD2D_STEP_XY
        ax_data.text(
            sx,
            sy,
            f"Step {int(step_n)}",
            ha="center",
            va="center",
            fontsize=CH3_KERAS_GD2D_STEP_FS,
            color="0.12",
            zorder=25,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="0.75", alpha=0.92),
        )
    ch3_draw_knob_row_ch2(fig, axes_k, ws, we, bb, knob_angle_refs)
    dpi_k = min(int(CH3_KERAS_GD2D_DPI), 110)
    return fig_to_image(fig, dpi=dpi_k, tight_layout=False)
def ch3_keras_gd_set_model_weights(model, w_st, w_el, b):
    """Set Dense(1) kernel/bias to match script plane coefficients."""
    W = np.array([[float(w_st)], [float(w_el)]], dtype=np.float32)
    bias = np.array([float(b)], dtype=np.float32)
    model.layers[0].set_weights([W, bias])
def ch3_keras_gd_step0_weights_clean():
    """Step-0 (ws, we, bb) for clean keras GD + bridge intro."""
    return float(CH3_KERAS_GD_INIT_W_ST), float(CH3_KERAS_GD_INIT_W_EL), float(CH3_KERAS_GD_INIT_B)
def ch3_keras_gd_trajectory(with_noise=False):
    """Train Keras logistic; return roster arrays, weight trajectory, knob angle ref."""
    study = np.asarray(study_real if with_noise else study_sep, dtype=np.float32)
    exam = np.asarray(exam_real if with_noise else exam_sep, dtype=np.float32)
    yv = np.asarray(y_real if with_noise else y_sep, dtype=np.float32).reshape(-1, 1)
    X_k = np.column_stack([study, exam])
    model = keras.models.Sequential()
    model.add(keras.layers.Dense(1, input_dim=2, activation="sigmoid"))
    model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=CH3_KERAS_GD2D_LR))
    ch3_keras_gd_set_model_weights(
        model, CH3_KERAS_GD_INIT_W_ST, CH3_KERAS_GD_INIT_W_EL, CH3_KERAS_GD_INIT_B
    )
    traj = [_ch3_keras_wb(model)]
    for _ in range(int(CH3_KERAS_GD2D_EPOCHS)):
        model.fit(X_k, yv, epochs=1, verbose=0, batch_size=int(CH3_KERAS_GD2D_BATCH))
        traj.append(_ch3_keras_wb(model))
    kref = traj[0]
    study_f = study_real if with_noise else study_sep
    exam_f = exam_real if with_noise else exam_sep
    y_f = y_real if with_noise else y_sep
    return study_f, exam_f, y_f, traj, kref
def ch3_build_frames_keras_gd_dataset_knobs(with_noise=False, *, show_colormap=False):
    study_f, exam_f, y_f, traj, kref = ch3_keras_gd_trajectory(with_noise)
    frames = []
    for step, (ws, we, bb) in enumerate(traj):
        im = ch3_frame_keras_gd_step(
            study_f, exam_f, y_f, ws, we, bb, step, kref, show_colormap=show_colormap
        )
        for _ in range(int(CH3_KERAS_GD2D_HOLD_PER_STEP)):
            frames.append(im.copy())
    return frames
def ch3_export_mp4_keras_gd_dataset_knobs(with_noise=False, *, show_colormap=False):
    frames = ch3_build_frames_keras_gd_dataset_knobs(with_noise, show_colormap=show_colormap)
    tag = "noise" if with_noise else "clean"
    cmap_tag = "colormap" if show_colormap else "nocmap"
    key = "keras_gd_dataset_knobs_colormap" if show_colormap else "keras_gd_dataset_knobs"
    fn = ch3_export_filename(
        key,
        fallback=f"ch3_keras_gd_dataset_knobs_{tag}_{cmap_tag}.mp4",
    )
    save_mp4(frames, fn, duration=CH3_KERAS_GD2D_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
# Short “try random knob combos” beat (script: too many combinations).
CH3_BRUTEFORCE_SEED = 19
CH3_BRUTEFORCE_N_SAMPLES = 16
CH3_BRUTEFORCE_HOLD_INIT = max(14, _smooth_n(10))
CH3_BRUTEFORCE_HOLD_SAMPLE = max(3, _smooth_n(2))
CH3_BRUTEFORCE_MS = max(42, CH3_MP4_MS_PER_FRAME)
CH3_BRUTEFORCE_W_ST_RANGE = (0.15, 2.85)
CH3_BRUTEFORCE_W_EL_RANGE = (-2.1, 1.6)
CH3_BRUTEFORCE_B_RANGE = (-1.6, 1.6)
def ch3_bruteforce_knob_weight_samples():
    """Haphazard (ws, we, b) triples from init — not an exhaustive grid."""
    w0 = ch3_keras_gd_step0_weights_clean()
    rng = np.random.default_rng(int(CH3_BRUTEFORCE_SEED))
    lo_st, hi_st = CH3_BRUTEFORCE_W_ST_RANGE
    lo_el, hi_el = CH3_BRUTEFORCE_W_EL_RANGE
    lo_b, hi_b = CH3_BRUTEFORCE_B_RANGE
    seen = {tuple(round(x, 4) for x in w0)}
    out = [w0]
    guard = 0
    while len(out) < int(CH3_BRUTEFORCE_N_SAMPLES) and guard < 400:
        guard += 1
        ws = float(rng.uniform(lo_st, hi_st))
        we = float(rng.uniform(lo_el, hi_el))
        bb = float(rng.uniform(lo_b, hi_b))
        key = (round(ws, 4), round(we, 4), round(bb, 4))
        if key in seen:
            continue
        seen.add(key)
        out.append((ws, we, bb))
    return out
def ch3_build_frames_bruteforce_knob_tries(*, show_colormap=False):
    study_f, exam_f, y_f = study_sep, exam_sep, y_sep
    kref = ch3_keras_gd_step0_weights_clean()
    frames = []
    for i, (ws, we, bb) in enumerate(ch3_bruteforce_knob_weight_samples()):
        hold = int(CH3_BRUTEFORCE_HOLD_INIT if i == 0 else CH3_BRUTEFORCE_HOLD_SAMPLE)
        im = ch3_frame_keras_gd_step(
            study_f,
            exam_f,
            y_f,
            ws,
            we,
            bb,
            i,
            kref,
            show_colormap=show_colormap,
            show_step_label=False,
        )
        for _ in range(hold):
            frames.append(im.copy())
    return frames
def ch3_export_mp4_bruteforce_knob_tries(*, show_colormap=False):
    frames = ch3_build_frames_bruteforce_knob_tries(show_colormap=show_colormap)
    fn = ch3_export_filename("keras_bruteforce_knob_tries")
    save_mp4(frames, fn, duration=CH3_BRUTEFORCE_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
def ch3_export_script_bruteforce_knob_tries_clean():
    ch3_export_mp4_bruteforce_knob_tries(show_colormap=False)
# Ball-trail variant: small NLL clouds around each trajectory step (no full grid).
# Single quiver at current (w1,w2,b): direction of grad(sum NLL), length 0.1 in weight space.
def _ch3_keras_wb(model):
    W, b = model.layers[0].get_weights()
    return float(W[0, 0]), float(W[1, 0]), float(b[0])
def _ch3_nll_sum_on_flat_grid(study, exam, y, w1_f, w2_f, b_f):
    """Sum negative log-likelihood at each column; study, exam shape (n,), w* shape (G,)."""
    study = np.asarray(study, dtype=np.float64)
    exam = np.asarray(exam, dtype=np.float64)
    yy = np.asarray(y, dtype=np.float64).reshape(-1, 1)
    eps = 1e-12
    z = w1_f[np.newaxis, :] * study[:, np.newaxis] + w2_f[np.newaxis, :] * exam[:, np.newaxis] + b_f[np.newaxis, :]
    z = np.clip(z, -80.0, 80.0)
    p = 1.0 / (1.0 + np.exp(-z))
    nll_pt = -(yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps))
    return np.asarray(nll_pt.sum(axis=0), dtype=np.float64)
def _ch3_nll_sum_grad_at_point(study, exam, y, w1, w2, b):
    # Gradient of sum NLL at one (w1, w2, b); same model as _ch3_nll_sum_on_flat_grid.
    study = np.asarray(study, dtype=np.float64)
    exam = np.asarray(exam, dtype=np.float64)
    yy = np.asarray(y, dtype=np.float64).reshape(-1)
    w1 = float(w1)
    w2 = float(w2)
    b = float(b)
    z = w1 * study + w2 * exam + b
    z = np.clip(z, -80.0, 80.0)
    p = 1.0 / (1.0 + np.exp(-z))
    r = p - yy
    g1 = float(np.dot(r, study))
    g2 = float(np.dot(r, exam))
    gb = float(np.sum(r))
    return g1, g2, gb
def _ch3_keras_draw_point_grad_quiver(ax3d, study, exam, y, ws, we, bb):
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    gn = float(np.sqrt(g1 * g1 + g2 * g2 + gb * gb))
    if gn < 1e-14:
        return
    ax3d.quiver(
        [float(ws)],
        [float(we)],
        [float(bb)],
        [g1],
        [g2],
        [gb],
        length=float(CH3_KERAS_NLL3D_POINT_QUIVER_LEN),
        normalize=True,
        color=CH3_KERAS_NLL3D_POINT_QUIVER_COLOR,
        linewidth=float(CH3_KERAS_NLL3D_POINT_QUIVER_LW),
        arrow_length_ratio=0.38,
        zorder=15,
    )
def _ch3_keras_draw_neg_grad_quiver(ax3d, study, exam, y, ws, we, bb):
    """Red arrow: negative gradient of sum NLL at (ws, we, bb)."""
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    gn = float(np.sqrt(g1 * g1 + g2 * g2 + gb * gb))
    if gn < 1e-14:
        return
    ax3d.quiver(
        [float(ws)],
        [float(we)],
        [float(bb)],
        [-g1],
        [-g2],
        [-gb],
        length=float(CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_LEN),
        normalize=True,
        color=CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_COLOR,
        linewidth=float(CH3_KERAS_NLL3D_NEG_GRAD_QUIVER_LW),
        arrow_length_ratio=0.38,
        zorder=16,
    )
def _ch3_weight_box_from_traj(traj, pad_frac=CH3_KERAS_NLL3D_PAD_FRAC):
    t = np.asarray(traj, dtype=float)
    lo = t.min(axis=0)
    hi = t.max(axis=0)
    span = np.maximum(hi - lo, 1e-6)
    pad = pad_frac * span
    return (
        float(lo[0] - pad[0]),
        float(hi[0] + pad[0]),
        float(lo[1] - pad[1]),
        float(hi[1] + pad[1]),
        float(lo[2] - pad[2]),
        float(hi[2] + pad[2]),
    )
def _ch3_keras_nll_train_trajectory(with_noise):
    study = np.asarray(study_real if with_noise else study_sep, dtype=np.float32)
    exam = np.asarray(exam_real if with_noise else exam_sep, dtype=np.float32)
    yv = np.asarray(y_real if with_noise else y_sep, dtype=np.float32)
    X_k = np.column_stack([study, exam])
    Y_k = yv.reshape(-1, 1)
    model = keras.models.Sequential()
    model.add(keras.layers.Dense(1, input_dim=2, activation="sigmoid"))
    model.compile(
        loss="binary_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=CH3_KERAS_NLL3D_LR),
    )
    ch3_keras_gd_set_model_weights(model, CH3_BRIDGE_W_ST, CH3_BRIDGE_W_EL, CH3_BRIDGE_B)
    traj = [_ch3_keras_wb(model)]
    for _ in range(CH3_KERAS_NLL3D_EPOCHS):
        model.fit(X_k, Y_k, epochs=1, verbose=0, batch_size=CH3_KERAS_NLL3D_BATCH)
        traj.append(_ch3_keras_wb(model))
    return study, exam, yv, np.asarray(traj, dtype=float)
def _ch3_ball_unit_offsets(n, seed=12345):
    g = np.random.default_rng(int(seed))
    v = g.normal(size=(int(n), 3))
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v.astype(np.float64)
def ch3_figure_nllkeras_dataset_weight3d():
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH3_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    ax_data = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    ax3d = fig.add_subplot(gs[0, 1], projection="3d")
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, ax3d, axes_k
def _ch3_keras_ax3d_zoom_bounds(ref_lo1, hi1, lo2, hi2, lob, hib, pts_xyz):
    ref_lo = np.array([float(ref_lo1), float(lo2), float(lob)], dtype=float)
    ref_hi = np.array([float(hi1), float(hi2), float(hib)], dtype=float)
    ref_span = np.maximum(ref_hi - ref_lo, 1e-9)
    P = np.asarray(pts_xyz, dtype=float).reshape(-1, 3)
    P = P[np.isfinite(P).all(axis=1)]
    if P.shape[0] == 0:
        return float(ref_lo1), float(hi1), float(lo2), float(hi2), float(lob), float(hib)
    lo = P.min(axis=0)
    hi = P.max(axis=0)
    span = np.maximum(hi - lo, CH3_KERAS_NLL3D_ZOOM_MIN_SPAN_FRAC * ref_span)
    center = 0.5 * (lo + hi)
    lo = center - 0.5 * span
    hi = center + 0.5 * span
    pad = CH3_KERAS_NLL3D_VIEW_PAD_FRAC * (hi - lo)
    return (
        float(lo[0] - pad[0]),
        float(hi[0] + pad[0]),
        float(lo[1] - pad[1]),
        float(hi[1] + pad[1]),
        float(lo[2] - pad[2]),
        float(hi[2] + pad[2]),
    )
def _ch3_keras_mean_zoom_frac(dlo1, dhi1, dlo2, dhi2, dlob, dhib, k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib):
    ref = np.array(
        [float(k_hi1 - k_lo1), float(k_hi2 - k_lo2), float(k_hib - k_lob)], dtype=np.float64
    )
    zsp = np.array(
        [float(dhi1 - dlo1), float(dhi2 - dlo2), float(dhib - dlob)], dtype=np.float64
    )
    return float(np.mean(zsp / np.maximum(ref, 1e-9)))
def _ch3_keras_alpha_from_zoom_frac(frac, alpha_hi, alpha_lo):
    t = (float(frac) - 0.10) / 0.78
    t = float(np.clip(t, 0.0, 1.0))
    return float(alpha_hi + (alpha_lo - alpha_hi) * t)
def _ch3_frame_nllkeras_weight3d(
    study,
    exam,
    y,
    ws,
    we,
    bb,
    *,
    W1m,
    W2m,
    Bm,
    Lf,
    vmin,
    vmax,
    traj_so_far,
):
    fig, ax_data, ax3d, axes_k = ch3_figure_nllkeras_dataset_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data,
        ws,
        we,
        bb,
        study,
        exam,
        y,
        leg,
        show_colormap=True,
        highlight_mistakes_flag=True,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    kr = [
        ch3_knob_strip_rotation_deg(ws, k_lo1, k_hi1),
        ch3_knob_strip_rotation_deg(we, k_lo2, k_hi2),
        ch3_knob_strip_rotation_deg(bb, k_lob, k_hib),
    ]
    ks = [1.0, 1.0, 1.0]
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig,
        axes_k,
        ws,
        we,
        bb,
        "st",
        knob_rgbs,
        canvas_sides,
        rot_strip_deg=0.0,
        strip_scale=1.0,
        knob_rots=kr,
        knob_scales=ks,
        ax_data=ax_data,
    )
    if len(traj_so_far) >= 1:
        trows = np.asarray(traj_so_far, dtype=float)
    else:
        trows = np.empty((0, 3), dtype=float)
    pts_zoom = np.vstack([trows, np.array([[ws, we, bb]], dtype=float)])
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_keras_ax3d_zoom_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, pts_zoom
    )
    ax3d.set_xlim(dlo1, dhi1)
    ax3d.set_ylim(dlo2, dhi2)
    ax3d.set_zlim(dlob, dhib)
    zfrac = _ch3_keras_mean_zoom_frac(dlo1, dhi1, dlo2, dhi2, dlob, dhib, k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib)
    a_sc = _ch3_keras_alpha_from_zoom_frac(
        zfrac, CH3_KERAS_NLL3D_SCATTER_ALPHA_ZOOM_HI, CH3_KERAS_NLL3D_SCATTER_ALPHA_ZOOM_LO
    )
    ax3d.scatter(
        W1m.ravel(),
        W2m.ravel(),
        Bm.ravel(),
        c=Lf.ravel(),
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=CH3_KERAS_NLL3D_SCATTER_SIZE,
        alpha=a_sc,
        linewidths=0,
        depthshade=False,
        zorder=1,
    )
    ax3d.set_xlabel(r"$w_1$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_ylabel(r"$w_2$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_zlabel(r"$b$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)
    ax3d.view_init(elev=22.0, azim=-135.0)
    if len(traj_so_far) >= 2:
        tx, ty, tz = zip(*traj_so_far)
        ax3d.plot(tx, ty, tz, color="0.2", lw=1.15, alpha=0.65, zorder=6)
    ax3d.scatter(
        [ws],
        [we],
        [bb],
        color="white",
        edgecolors="black",
        linewidths=1.0,
        s=120,
        zorder=20,
        depthshade=False,
    )
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_keras_gd_init_weights():
    """Fixed Step-0 plane coefficients (same as ``CH3_KERAS_GD_INIT_*``)."""
    return ch3_keras_gd_step0_weights_clean()
def ch3_export_bridge_keras_init_knobs_intro_mp4():
    """Hold Keras init panel, then bridge knob intro (stops before duo slide)."""
    ch3_ensure_knob_pngs()
    ws, we, bb = ch3_keras_gd_step0_weights_clean()
    k1 = Image.open(OUTPUT_DIR / "knob_1_cropped.png").convert("RGBA")
    k2 = Image.open(OUTPUT_DIR / "knob_2_cropped.png").convert("RGBA")
    k3 = Image.open(OUTPUT_DIR / "knob_3_cropped.png").convert("RGBA")
    frames = []
    for _ in range(CH3_BRIDGE_KERAS_INIT_HOLD):
        frames.append(ch3_bridge_frame_hold_keras_init(ws, we, bb))
    for tv in np.linspace(0.0, 1.0, CH3_BRIDGE_PHASE1_N, endpoint=True):
        frames.append(ch3_bridge_frame_phase1_intro_knobs_keras(float(tv), k1, k2, k3, ws, we, bb))
    last = frames[-1]
    for _ in range(CH3_BRIDGE_HOLD_END):
        frames.append(last.copy())
    save_mp4(frames, ch3_export_filename("bridge_keras_init_knobs_intro"), duration=CH3_BRIDGE_MP4_MS)
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / ch3_export_filename("bridge_keras_init_knobs_intro"))
def ch3_export_script_keras_gd_dataset_knobs_clean():
    ch3_export_mp4_keras_gd_dataset_knobs(False, show_colormap=False)
def ch3_export_script_keras_gd_dataset_knobs_clean_colormap():
    ch3_export_mp4_keras_gd_dataset_knobs(False, show_colormap=True)
def ch3_export_mp4_keras_nll_weight3d(with_noise: bool):
    study, exam, yv, traj = _ch3_keras_nll_train_trajectory(with_noise)
    lo1, hi1, lo2, hi2, lob, hib = _ch3_weight_box_from_traj(traj)
    gn = int(CH3_KERAS_NLL3D_GRID_PER_AXIS)
    g1 = np.linspace(lo1, hi1, gn, dtype=np.float64)
    g2 = np.linspace(lo2, hi2, gn, dtype=np.float64)
    gb = np.linspace(lob, hib, gn, dtype=np.float64)
    W1m, W2m, Bm = np.meshgrid(g1, g2, gb, indexing="ij")
    w1f = W1m.ravel()
    w2f = W2m.ravel()
    bf = Bm.ravel()
    Lf = _ch3_nll_sum_on_flat_grid(study, exam, yv, w1f, w2f, bf)
    vmin = float(np.percentile(Lf, 2.0))
    vmax = float(np.percentile(Lf, 98.0))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = float(Lf.min()), float(Lf.max() + 1e-6)
    frames = []
    for ti in range(len(traj)):
        ws, we, bb = (float(traj[ti, 0]), float(traj[ti, 1]), float(traj[ti, 2]))
        prefix = [tuple(map(float, row)) for row in traj[: ti + 1]]
        frames.append(
            _ch3_frame_nllkeras_weight3d(
                study,
                exam,
                yv,
                ws,
                we,
                bb,
                W1m=W1m,
                W2m=W2m,
                Bm=Bm,
                Lf=Lf,
                vmin=vmin,
                vmax=vmax,
                traj_so_far=prefix,
            )
        )
    last = frames[-1]
    for _ in range(CH3_KERAS_NLL3D_HOLD_LAST):
        frames.append(last.copy())
    tag = "noise" if with_noise else "clean"
    fn = ch3_export_filename(f"negloglik_keras_weight3d_{tag}", fallback=f"ch3_negloglik_keras_dataset_weight3d_{tag}.mp4")
    save_mp4(frames, fn, duration=CH3_KERAS_NLL3D_MS)
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
def _ch3_frame_nllkeras_weight3d_balltrail(
    study,
    exam,
    y,
    ws,
    we,
    bb,
    *,
    lo1,
    hi1,
    lo2,
    hi2,
    lob,
    hib,
    P_acc,
    L_acc,
    vmin,
    vmax,
    elev,
    azim,
    traj_so_far,
    ax3d_xlim,
    ax3d_ylim,
    ax3d_zlim,
):
    fig, ax_data, ax3d, axes_k = ch3_figure_nllkeras_dataset_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data,
        ws,
        we,
        bb,
        study,
        exam,
        y,
        leg,
        show_colormap=True,
        highlight_mistakes_flag=True,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    kr = [
        ch3_knob_strip_rotation_deg(ws, lo1, hi1),
        ch3_knob_strip_rotation_deg(we, lo2, hi2),
        ch3_knob_strip_rotation_deg(bb, lob, hib),
    ]
    ks = [1.0, 1.0, 1.0]
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig,
        axes_k,
        ws,
        we,
        bb,
        "st",
        knob_rgbs,
        canvas_sides,
        rot_strip_deg=0.0,
        strip_scale=1.0,
        knob_rots=kr,
        knob_scales=ks,
        ax_data=ax_data,
    )
    dlo1, dhi1 = ax3d_xlim
    dlo2, dhi2 = ax3d_ylim
    dlob, dhib = ax3d_zlim
    ax3d.set_xlim(dlo1, dhi1)
    ax3d.set_ylim(dlo2, dhi2)
    ax3d.set_zlim(dlob, dhib)
    zfrac = _ch3_keras_mean_zoom_frac(dlo1, dhi1, dlo2, dhi2, dlob, dhib, lo1, hi1, lo2, hi2, lob, hib)
    a_ball = _ch3_keras_alpha_from_zoom_frac(
        zfrac, CH3_KERAS_NLL3D_BALL_ALPHA_ZOOM_HI, CH3_KERAS_NLL3D_BALL_ALPHA_ZOOM_LO
    )
    if len(P_acc) > 0:
        ax3d.scatter(
            P_acc[:, 0],
            P_acc[:, 1],
            P_acc[:, 2],
            c=L_acc,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            s=CH3_KERAS_NLL3D_BALL_POINT_SIZE,
            alpha=a_ball,
            linewidths=0,
            depthshade=False,
            zorder=1,
        )
    ax3d.set_xlabel(r"$w_1$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_ylabel(r"$w_2$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_zlabel(r"$b$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)
    ax3d.view_init(elev=float(elev), azim=float(azim))
    if len(traj_so_far) >= 2:
        tx, ty, tz = zip(*traj_so_far)
        ax3d.plot(tx, ty, tz, color="0.2", lw=1.15, alpha=0.65, zorder=6)
    ax3d.scatter(
        [ws],
        [we],
        [bb],
        color="white",
        edgecolors="black",
        linewidths=1.0,
        s=120,
        zorder=20,
        depthshade=False,
    )
    _ch3_keras_draw_neg_grad_quiver(ax3d, study, exam, y, ws, we, bb)
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)
def ch3_export_mp4_keras_nll_weight3d_balltrail(with_noise: bool):
    study, exam, yv, traj = _ch3_keras_nll_train_trajectory(with_noise)
    lo1, hi1, lo2, hi2, lob, hib = _ch3_weight_box_from_traj(traj)
    span = np.asarray(traj.max(axis=0) - traj.min(axis=0), dtype=float)
    R = float(CH3_KERAS_NLL3D_BALL_R_SCALE * max(float(span.max()), 1e-6))
    offs = _ch3_ball_unit_offsets(int(CH3_KERAS_NLL3D_BALL_NSAMPLE))
    T = int(traj.shape[0])
    ball_pts = np.empty((T, offs.shape[0], 3), dtype=np.float64)
    ball_L = np.empty((T, offs.shape[0]), dtype=np.float64)
    for j in range(T):
        c = traj[j, :]
        Pj = c + R * offs
        ball_pts[j] = Pj
        ball_L[j] = _ch3_nll_sum_on_flat_grid(study, exam, yv, Pj[:, 0], Pj[:, 1], Pj[:, 2])
    Lflat = ball_L.ravel()
    vmin = float(np.percentile(Lflat, 2.0))
    vmax = float(np.percentile(Lflat, 98.0))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = float(Lflat.min()), float(Lflat.max() + 1e-6)
    pts_final = np.vstack([traj, ball_pts.reshape(-1, 3)])
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_keras_ax3d_zoom_bounds(
        lo1, hi1, lo2, hi2, lob, hib, pts_final
    )
    ax3d_xlim = (dlo1, dhi1)
    ax3d_ylim = (dlo2, dhi2)
    ax3d_zlim = (dlob, dhib)
    nF = T
    frames = []
    for ti in range(nF):
        ws, we, bb = (float(traj[ti, 0]), float(traj[ti, 1]), float(traj[ti, 2]))
        Pacc = ball_pts[: ti + 1].reshape(-1, 3)
        Lacc = ball_L[: ti + 1].ravel()
        az = float(CH3_KERAS_NLL3D_CAM_AZIM_START + CH3_KERAS_NLL3D_CAM_AZIM_DELTA * (float(ti) / float(max(nF - 1, 1))))
        prefix = [tuple(map(float, row)) for row in traj[: ti + 1]]
        frames.append(
            _ch3_frame_nllkeras_weight3d_balltrail(
                study,
                exam,
                yv,
                ws,
                we,
                bb,
                lo1=lo1,
                hi1=hi1,
                lo2=lo2,
                hi2=hi2,
                lob=lob,
                hib=hib,
                P_acc=Pacc,
                L_acc=Lacc,
                vmin=vmin,
                vmax=vmax,
                elev=CH3_KERAS_NLL3D_CAM_ELEV,
                azim=az,
                traj_so_far=prefix,
                ax3d_xlim=ax3d_xlim,
                ax3d_ylim=ax3d_ylim,
                ax3d_zlim=ax3d_zlim,
            )
        )
    last = frames[-1]
    for _ in range(CH3_KERAS_NLL3D_HOLD_LAST):
        frames.append(last.copy())
    tag = "noise" if with_noise else "clean"
    fn = ch3_export_filename(f"negloglik_keras_weight3d_ball_{tag}", fallback=f"ch3_negloglik_keras_dataset_weight3d_ball_{tag}.mp4")
    save_mp4(frames, fn, duration=CH3_KERAS_NLL3D_MS)
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)


Chapter 3 setup OK — 20 clean, 26 with noise.


## Exports (ch3_00–58)

Re-run **cell 1** (setup) before any export cell.


### Bridge & dataset stills (ch3_00–03)


In [ ]:
# ch3_00 — ch3_00_bridge_dataset_knobs_into_duo_slot.mp4
ch3_export_bridge_dataset_knobs_into_duo_slot_mp4()


In [ ]:
# ch3_01 — ch3_01_dataset_clean_plain_still.mp4
ch3_export_script_still_dataset_clean_plain()
ch3_export_script_still_dataset_clean_st_el_zero()
ch3_export_script_still_dataset_noise_plain()


In [ ]:
# ch3_02 — ch3_02_dataset_clean_st_el_zero_still.mp4
ch3_export_script_still_dataset_clean_st_el_zero()


In [ ]:
# ch3_03 — ch3_03_dataset_noise_plain_still.mp4
ch3_export_script_still_dataset_noise_plain()


### Keras GD & bridges (ch3_04–07)


In [ ]:
# ch3_04 — ch3_04_keras_gd_dataset_knobs_steps.mp4
ch3_export_script_keras_gd_dataset_knobs_clean_colormap()


In [ ]:
# ch3_05 — ch3_05_bridge_keras_init_knobs_intro.mp4
ch3_export_bridge_keras_init_knobs_intro_mp4()


In [ ]:
# ch3_06 — ch3_06_mistakes_triptych_noise.mp4
ch3_export_mp4_triptych("mistakes", True)


In [ ]:
# ch3_07 — ch3_07_sumprob_far_flat.mp4
ch3_export_script_sumprob_far_flat()


### Dataset & goals stills (ch3_08–12)


In [ ]:
# ch3_08 — ch3_08_dataset_clean_still.mp4
ch3_export_script_still_dataset_clean()


In [ ]:
# ch3_09 — ch3_09_dataset_noise_still.mp4
ch3_export_script_still_dataset_noise()


In [ ]:
# ch3_10 — ch3_10_goals_clean_still.mp4
ch3_export_script_still_goals_clean()


In [ ]:
# ch3_11 — ch3_11_mistakes_st_clean.mp4
ch3_export_mp4("mistakes", "st", False)


In [ ]:
# ch3_12 — ch3_12_negloglik_keras_dataset_weight3d_clean.mp4
ch3_export_mp4_keras_nll_weight3d(False)


### Mistakes k1 staircases (ch3_13–19)


In [ ]:
# ch3_13 — ch3_13_mistakes_k1_flat_small.mp4
ch3_export_script_mistakes_k1_flat_small()


In [ ]:
# ch3_14 — ch3_14_mistakes_k1_staircase_wide.mp4
ch3_export_script_mistakes_k1_staircase_wide()


In [ ]:
# ch3_15 — ch3_15_mistakes_k1_staircase_descent_w12.mp4
xs14, ys14 = ch3_k1_curve_carry_from_wide(float(CH3_SCRIPT_K1_W_ST))
y_lim_w12, (xs_w12, ys_w12) = ch3_export_script_mistakes_k1_staircase_descent_at(
    1.15,
    w_from=float(CH3_SCRIPT_K1_W_ST),
    grow_start=True,
    xs_shift=xs14,
    ys_shift=ys14,
)


In [ ]:
# ch3_16 — ch3_16_mistakes_k1_staircase_descent_w10.mp4
y_lim_w10, (xs_w10, ys_w10) = ch3_export_script_mistakes_k1_staircase_descent_at(
    0.95,
    w_from=1.15,
    grow_start=False,
    y_lim_start=y_lim_w12,
    xs_shift=xs_w12,
    ys_shift=ys_w12,
)


In [ ]:
# ch3_17 — ch3_17_mistakes_k1_staircase_descent_w08.mp4
y_lim_w08, (xs_w08, ys_w08) = ch3_export_script_mistakes_k1_staircase_descent_at(
    0.75,
    w_from=0.95,
    grow_start=False,
    y_lim_start=y_lim_w10,
    xs_shift=xs_w10,
    ys_shift=ys_w10,
)


In [ ]:
# ch3_18 — ch3_18_mistakes_k1_staircase_descent_w06.mp4
y_lim_w06, (xs_w06, ys_w06) = ch3_export_script_mistakes_k1_staircase_descent_at(
    0.55,
    w_from=0.75,
    grow_start=False,
    y_lim_start=y_lim_w08,
    xs_shift=xs_w08,
    ys_shift=ys_w08,
)


In [ ]:
# ch3_19 — ch3_19_mistakes_k1_staircase_stop.mp4
ch3_export_script_mistakes_k1_staircase_stop()


### Triptych reveal & post-staircase (ch3_20–26)


In [ ]:
# ch3_20 — ch3_20_mistakes_triptych_reveal_el_b.mp4
ch3_export_script_mistakes_triptych_reveal_el_b()
# After ch3_18 (with y_lim_w06, xs_w06, ys_w06 assigned), for pixel-perfect match:
# ch3_export_script_mistakes_triptych_reveal_el_b(xs_st=xs_w06, ys_st=ys_w06, y_lim_st=y_lim_w06)


In [ ]:
# ch3_21 — ch3_21_mistakes_k1_staircase_descent_triptych.mp4
# ch3_21: ch3_18 $w_1$ hold + reveal; $w_1$ pan on fixed curve; $w_2$/$b$ ghost + live; knobs 2–3 fixed.
# After ch3_20, chain: ch3_export_script_mistakes_k1_staircase_descent_triptych(xs_st=xs_w06, ys_st=ys_w06, y_lim_st=y_lim_w06, skip_intro=True)
ch3_export_script_mistakes_k1_staircase_descent_triptych()


In [ ]:
# ch3_22 — ch3_22_mistakes_k1_post_21_arrows.mp4
ch3_export_script_mistakes_k1_post_21_arrows()


In [ ]:
# ch3_23 — ch3_23_mistakes_k1_post_22_w2_nudge.mp4
ch3_export_script_mistakes_k1_post_22_w2_nudge()


In [ ]:
# ch3_24 — ch3_24_mistakes_k1_post_23_small_nudge.mp4
ch3_export_script_mistakes_k1_post_23_small_nudge()


In [ ]:
# ch3_25 — ch3_25_mistakes_triptych_clean.mp4
ch3_export_mp4_triptych("mistakes", False)


In [ ]:
# ch3_26 — ch3_26_mistakes_k1_tiny_triptych_coupled.mp4
ch3_export_script_mistakes_k1_tiny_triptych_coupled()


### Sumprob / likelihood / NLL triptychs (ch3_27–37)


In [ ]:
# ch3_27 — ch3_27_sumprob_triptych_clean.mp4
ch3_export_mp4_triptych("sumprob", False)


In [ ]:
# ch3_28 — ch3_28_sumprob_st_clean.mp4
ch3_export_mp4("sumprob", "st", False)


In [ ]:
# ch3_29 — ch3_29_sumprob_k1_small.mp4
ch3_export_mp4_keras_nll_weight3d(True)


In [ ]:
# ch3_30 — ch3_30_likelihood_triptych_clean.mp4
ch3_export_mp4_keras_nll_weight3d_balltrail(False)


In [ ]:
# ch3_31 — ch3_31_likelihood_st_clean.mp4
ch3_export_mp4_keras_nll_weight3d_balltrail(True)


In [ ]:
# ch3_32 — ch3_32_negloglik_triptych_clean.mp4
ch3_export_mp4_triptych("negloglik", False)


In [ ]:
# ch3_33 — ch3_33_negloglik_keras_dataset_weight3d_noise.mp4
ch3_export_mp4_keras_nll_weight3d(True)


In [ ]:
# ch3_34 — ch3_34_negloglik_keras_dataset_weight3d_ball_clean.mp4
ch3_export_mp4_keras_nll_weight3d_balltrail(False)


In [ ]:
# ch3_35 — ch3_35_negloglik_keras_dataset_weight3d_ball_noise.mp4
ch3_export_mp4_keras_nll_weight3d_balltrail(True)


In [ ]:
# ch3_36 — ch3_36_keras_gd_dataset_knobs_steps_colormap.mp4
ch3_export_script_keras_gd_dataset_knobs_clean_colormap()


In [ ]:
# ch3_37 — ch3_37_keras_bruteforce_knob_tries.mp4
ch3_export_script_bruteforce_knob_tries_clean()


### Strip σ / knobs / labels / pile (ch3_38–43)


In [ ]:
# ch3_38 — ch3_38_mistakes_strip_prob_story.mp4
ch3_export_script_mistakes_strip_prob_story(False)


In [ ]:
# ch3_39 — ch3_39_mistakes_strip_knob_to_mid.mp4
ch3_export_script_strip_knob_to_mid(False)


In [ ]:
# ch3_40 — ch3_40_mistakes_strip_knob_to_amp.mp4
ch3_export_script_strip_knob_to_amp(False)


In [ ]:
# ch3_41 — ch3_41_mistakes_strip_prob_labels.mp4
ch3_export_script_strip_prob_labels(False)


In [ ]:
# ch3_42 — ch3_42_mistakes_strip_pile_amp.mp4
ch3_export_script_strip_pile_amp(False)


In [ ]:
# ch3_43 — ch3_43_mistakes_strip_pile_start.mp4
ch3_export_script_strip_pile_start(False)


### Sumprob k1 sequence (ch3_44–49)


In [ ]:
# ch3_44 — ch3_44_sumprob_k1_flat_small.mp4
ch3_export_script_sumprob_k1_flat_small()


In [ ]:
# ch3_45 — ch3_45_sumprob_k1_staircase_wide.mp4
ch3_export_script_sumprob_k1_staircase_wide()


In [ ]:
# ch3_46 — ch3_46_sumprob_k1_staircase_descent_w12.mp4
ch3_export_script_sumprob_k1_staircase_descent_at(1.15, w_from=float(CH3_SCRIPT_K1_W_ST), grow_start=True)


In [ ]:
# ch3_47 — ch3_47_sumprob_k1_staircase_descent_w10.mp4
ch3_export_script_sumprob_k1_staircase_descent_at(0.95, w_from=1.15, grow_start=False)


In [ ]:
# ch3_48 — ch3_48_sumprob_k1_staircase_descent_w08.mp4
ch3_export_script_sumprob_k1_staircase_descent_at(0.75, w_from=0.95, grow_start=False)


In [ ]:
# ch3_49 — ch3_49_sumprob_k1_staircase_descent_w06.mp4
ch3_export_script_sumprob_k1_staircase_descent_at(0.55, w_from=0.75, grow_start=False)


### Sumprob k1 tight sweep (ch3_50–54)


In [ ]:
# ch3_50 — ch3_50_sumprob_k1_tight_staircase_wide.mp4
ch3_export_script_sumprob_k1_tight_staircase_wide()


In [ ]:
# ch3_51 — ch3_51_sumprob_k1_tight_staircase_descent_w12.mp4
ch3_export_script_sumprob_k1_tight_staircase_descent_at(1.15, w_from=float(CH3_SCRIPT_K1_W_ST), grow_start=True)


In [ ]:
# ch3_52 — ch3_52_sumprob_k1_tight_staircase_descent_w10.mp4
ch3_export_script_sumprob_k1_tight_staircase_descent_at(0.95, w_from=1.15, grow_start=False)


In [ ]:
# ch3_53 — ch3_53_sumprob_k1_tight_staircase_descent_w08.mp4
ch3_export_script_sumprob_k1_tight_staircase_descent_at(0.75, w_from=0.95, grow_start=False)


In [ ]:
# ch3_54 — ch3_54_sumprob_k1_tight_staircase_descent_w06.mp4
ch3_export_script_sumprob_k1_tight_staircase_descent_at(0.55, w_from=0.75, grow_start=False)


### Slope triangles (ch3_55)


In [ ]:
# ch3_55 — ch3_55_sumprob_k1_slope_triangles.mp4
ch3_export_script_sumprob_k1_slope_triangles()


### Likelihood compound & contrast (ch3_56–58)


In [ ]:
# ch3_56 — ch3_56_mistakes_strip_likelihood_amp.mp4
ch3_export_script_strip_likelihood_amp(False)


In [ ]:
# ch3_57 — ch3_57_mistakes_strip_likelihood_start.mp4
ch3_export_script_strip_likelihood_start(False)


In [ ]:
# ch3_58 — ch3_58_mistakes_strip_pile_likelihood_contrast.mp4
ch3_export_script_strip_pile_likelihood_contrast()
